In [3]:
!conda uninstall -c conda-forge cupy cudatoolkit=12.0


PackagesNotFoundError: The following packages are missing from the target environment:

  - cupy
  - cudatoolkit=12.0




In [1]:
!pip install numpy scipy scikit-learn tqdm geopandas shapely requests mgrs openeo rasterio openpyxl supabase

  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached shapely-2.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.8 kB)
  Using cached mgrs-1.5.4-cp312-cp312-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl.metadata (4.7 kB)
  Using cached openeo-0.48.0-py3-none-any.whl.metadata (8.4 kB)
  Using cached rasterio-1.5.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (8.6 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached supabase-2.28.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached pyogrio-0.12.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.9 kB)
  Using cached pyproj-3.7.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached xarray-2025.1.1-py3-none-any.whl.metadata (11 kB)
  Using cached pystac-1.14.3-py3-none-any.whl.metadata (4.7 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0

In [4]:
pip uninstall cupy cudatoolkit==12.0

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

# =============================================================================
# ENV-DRIVEN CONFIGURATION
# Set these environment variables before running (or use docker-compose / ECS):
#
#   MILL          - one of: EMSA | IPSA | MONTE_ROSA | PANTALEON | AMAJAC
#   FECHA_INICIO  - image search window start date  (YYYY-MM-DD)
#   FECHA_FIN     - image search window end date    (YYYY-MM-DD)
#   FECHAS        - comma-separated processing dates (e.g. 2026-03-26)
#
# For interactive notebook use you can also just set them here directly:
#   os.environ['MILL'] = 'EMSA'
# =============================================================================

MILL_CONFIGS = {
    "EMSA": {
        "aoi_geojson":      "MX07_EMSA_NewBBox.geojson",
        "out_dir":          "emsa-auto",
        "inference_folder": "emsa-auto",
        "output_folder":    "inputs-emsa-auto",
        "prefix":           "MX07_Grupo Pantaleon",
        "location":         "EMSA",
        "ingenio_name":     "EMSA",
        "input_dir":        "inputs-emsa-auto",
        "output_dir":       "Output-emsa",
    },
    "IPSA": {
        "aoi_geojson":      "IPSA.geojson",
        "out_dir":          "IPSA-auto",
        "inference_folder": "IPSA-auto",
        "output_folder":    "inputs-ipsa-auto",
        "prefix":           "MX02_Grupo Pantaleon",
        "location":         "IPSA",
        "ingenio_name":     "IPSA",
        "input_dir":        "inputs-ipsa-auto",
        "output_dir":       "Output-ipsa",
    },
    "MONTE_ROSA": {
        "aoi_geojson":      "MR01_BoundingBox_margin001.geojson",
        "out_dir":          "monterosa-auto",
        "inference_folder": "monterosa-auto",
        "output_folder":    "inputs-monterosa-auto",
        "prefix":           "NI_Grupo Pantaleon",
        "location":         "Monte_Rosa",
        "ingenio_name":     "Monte Rosa",
        "input_dir":        "inputs-monterosa-auto",
        "output_dir":       "Output-monterosa",
    },
    "PANTALEON": {
        "aoi_geojson":      "GT01_BoundingBox_margin001.geojson",
        "out_dir":          "pantaleon-auto",
        "inference_folder": "pantaleon-auto",
        "output_folder":    "inputs-pantaleon-auto",
        "prefix":           "GT_Grupo Pantaleon",
        "location":         "Pantaleon",
        "ingenio_name":     "Pantaleon",
        "input_dir":        "inputs-pantaleon-auto",
        "output_dir":       "Output-gt",
    },
    "AMAJAC": {
        "aoi_geojson":      "AM01_BoundingBox_margin001.geojson",
        "out_dir":          "amajac-auto",
        "inference_folder": "amajac-auto",
        "output_folder":    "inputs-amajac-auto",
        "prefix":           "MX06_Grupo Pantaleon",
        "location":         "Amajac",
        "ingenio_name":     "Amajac",
        "input_dir":        "inputs-amajac-auto",
        "output_dir":       "Output-amajac",
    },
}

# ── Read env vars ─────────────────────────────────────────────────────────────
MILL = os.environ.get("MILL", "").upper().replace(" ", "_").replace("-", "_")

if not MILL or MILL not in MILL_CONFIGS:
    raise EnvironmentError(
        f"MILL env var must be one of {list(MILL_CONFIGS.keys())}. Got: '{os.environ.get('MILL', '')}'"
    )

_fecha_inicio_str = os.environ.get("FECHA_INICIO", "")
_fecha_fin_str    = os.environ.get("FECHA_FIN",    "")
_fechas_str       = os.environ.get("FECHAS",       "")

if not _fecha_inicio_str or not _fecha_fin_str:
    raise EnvironmentError("FECHA_INICIO and FECHA_FIN env vars are required (YYYY-MM-DD)")

if not _fechas_str:
    raise EnvironmentError("FECHAS env var is required (comma-separated dates, e.g. 2026-03-26)")

# ── Populate pipeline ENV vars ────────────────────────────────────────────────
cfg = MILL_CONFIGS[MILL]

AOI_GEOJSON_ENV  = Path(cfg["aoi_geojson"])
fecha_inicio_ENV = datetime.strptime(_fecha_inicio_str, "%Y-%m-%d")
fecha_fin_ENV    = datetime.strptime(_fecha_fin_str,    "%Y-%m-%d")
OUT_DIR_ENV      = Path(cfg["out_dir"])
NAME_CONFIG_ENV  = {
    "inference_folder": cfg["inference_folder"],
    "output_folder":    cfg["output_folder"],
    "prefix":           cfg["prefix"],
    "location":         cfg["location"],
}
INGENIO_ENV    = cfg["ingenio_name"]
FECHAS_ENV     = [d.strip() for d in _fechas_str.split(",")]
INPUT_DIR_ENV  = cfg["input_dir"]
OUTPUT_DIR_ENV = cfg["output_dir"]

print(f"✅ Config loaded for mill: {MILL}")
print(f"   AOI:         {AOI_GEOJSON_ENV}")
print(f"   Date range:  {fecha_inicio_ENV.date()} → {fecha_fin_ENV.date()}")
print(f"   FECHAS:      {FECHAS_ENV}")
print(f"   Out dir:     {OUT_DIR_ENV}")
print(f"   Input dir:   {INPUT_DIR_ENV}")
print(f"   Output dir:  {OUTPUT_DIR_ENV}")


In [2]:
# ============================================================
# Cell 1: S1/S2 Image Finder & Downloader - PAIRS ONLY
# ============================================================
# Key Logic:
# 1. Inference date = latest complete date in 5-day window
# 2. Previous 5 complete images from last 30 days (excl. inference)
# 3. NO cloud cover filtering - download if spatially complete
# 4. For inference range: allow broken image if no complete one
# 5. Track cloud% and nodata% per date
# 6. Downloads go DIRECTLY into pairs/ folder
# 7. On re-run: auto-rename inference→prev01, shift prevNN up
# 8. Check existing folders to avoid re-downloads
# 9. S2: batch job download with retry + progress tracking
# 10. S1: RAW GRD 60-day median composite (no sar_backscatter)
# ============================================================

import os
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict, OrderedDict
import re
import requests
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon, box, MultiPolygon, shape
from shapely.ops import unary_union
from shapely import wkt
import warnings
import json
import tempfile
import shutil
import copy
import time
import sys
warnings.filterwarnings('ignore')

try:
    import mgrs
    HAS_MGRS = True
except ImportError:
    HAS_MGRS = False
    print("⚠️  mgrs library not installed. Install with: pip install mgrs")

try:
    import openeo
    HAS_OPENEO = True
except ImportError:
    HAS_OPENEO = False
    print("⚠️  openeo library not installed")

try:
    import rasterio
    from rasterio.warp import calculate_default_transform, reproject, Resampling
    from rasterio.crs import CRS
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print("⚠️  rasterio library not installed")

# =============================================================================
# CONFIG
# =============================================================================
AOI_GEOJSON = AOI_GEOJSON_ENV

fecha_inicio = fecha_inicio_ENV
fecha_fin = fecha_fin_ENV

LOOKBACK_DAYS = 30
PREVIOUS_IMAGES_COUNT = 5

TARGET_CRS = "EPSG:4326"
MARGIN_DEGREES = 0.001

# SINGLE output directory - pairs only
DOWNLOAD_BASE_DIR = OUT_DIR_ENV
PAIRS_DIR = DOWNLOAD_BASE_DIR / "pairs"

# Coverage thresholds - NO cloud filtering
MIN_SPATIAL_COVERAGE_PCT = 95.0
MAX_NODATA_PCT = 10.0
PREVIEW_RESOLUTION = 100

OPENEO_BACKEND = "https://openeo.dataspace.copernicus.eu"

# S2 Config
S2_COLLECTION = "SENTINEL2_L2A"
S2_BANDS = [
    "B01", "B02", "B03", "B04", "B05", "B06", "B07",
    "B08", "B8A", "B09", "B11", "B12", "WVP", "AOT", "SCL"
]
S2_EXPECTED_BANDS = len(S2_BANDS)  # 15
TARGET_RESOLUTION = 10

# S1 Config
S1_COLLECTION = "SENTINEL1_GRD"
S1_BANDS = ["VV", "VH"]
S1_EXPECTED_BANDS = len(S1_BANDS)  # 2
S1_DATE_BUFFER = 60  # 60-day composite BEFORE the target date

# Download Config
MAX_RETRIES = 3
POLL_INTERVAL = 30
JOB_TIMEOUT = 7200
CHUNK_SIZE = 8 * 1024 * 1024
TIMEOUT = 300
RETRY_DELAY = 10


# =============================================================================
# BAND COUNT VALIDATION HELPER
# =============================================================================
def get_band_count(filepath: Path) -> int:
    """Get band count of a raster file. Returns 0 on error."""
    if not HAS_RASTERIO or not filepath or not filepath.exists():
        return 0
    try:
        with rasterio.open(filepath) as src:
            return src.count
    except Exception:
        return 0


def is_valid_s1_file(filepath: Path) -> bool:
    """Check if file is a valid S1 file (2 bands, reasonable size)."""
    if not filepath or not filepath.exists():
        return False
    if filepath.stat().st_size < 1000:
        return False
    bands = get_band_count(filepath)
    return bands == S1_EXPECTED_BANDS


def is_valid_s2_file(filepath: Path) -> bool:
    """Check if file is a valid S2 file (15 bands, reasonable size)."""
    if not filepath or not filepath.exists():
        return False
    if filepath.stat().st_size < 1000:
        return False
    bands = get_band_count(filepath)
    return bands == S2_EXPECTED_BANDS


# =============================================================================
# METADATA TRACKER
# =============================================================================
class DateMetadata:
    """Track metadata for each downloaded date."""
    def __init__(self, date_str: str, satellite: str):
        self.date = date_str
        self.satellite = satellite
        self.cloud_cover_pct = None
        self.nodata_pct = None
        self.spatial_coverage_pct = None
        self.tiles = []
        self.is_complete = None
        self.is_inference = False
        self.is_broken_allowed = False
        self.download_path = None
        self.file_size_mb = None
        self.valid_pixel_pct = None
        self.band_count = None

    def to_dict(self):
        return {k: (str(v) if isinstance(v, Path) else v)
                for k, v in self.__dict__.items()}

    def __repr__(self):
        status = "✅" if self.is_complete else "⚠️"
        cloud = f"Cloud:{self.cloud_cover_pct:.1f}%" if self.cloud_cover_pct is not None else "Cloud:N/A"
        nodata = f"NoData:{self.nodata_pct:.1f}%" if self.nodata_pct is not None else "NoData:N/A"
        bands = f"Bands:{self.band_count}" if self.band_count is not None else ""
        role = "[INFERENCE]" if self.is_inference else "[PREVIOUS]"
        return f"{status} {self.satellite} {self.date} {role} {cloud} {nodata} {bands}"


# =============================================================================
# PAIRS FOLDER MANAGEMENT
# =============================================================================
def parse_pair_folder_name(folder_name: str) -> dict:
    m = re.match(r'^inference_(\d{4}-\d{2}-\d{2})$', folder_name)
    if m:
        return {'role': 'inference', 'index': None, 'date': m.group(1)}
    m = re.match(r'^prev(\d{2})_(\d{4}-\d{2}-\d{2})$', folder_name)
    if m:
        return {'role': 'previous', 'index': int(m.group(1)), 'date': m.group(2)}
    return None


def find_s2_file_in_folder(folder: Path) -> Path:
    """Find a valid S2 file in folder. Returns None if not found."""
    if not folder or not folder.exists():
        return None
    for f in sorted(folder.glob("S2_*.tif"), key=lambda x: x.stat().st_mtime, reverse=True):
        if is_valid_s2_file(f):
            return f
    for f in sorted(folder.glob("openEO_*.tif"), key=lambda x: x.stat().st_mtime, reverse=True):
        if is_valid_s2_file(f):
            return f
    return None


def find_s1_file_in_folder(folder: Path) -> Path:
    """
    Find a valid S1 file in folder (2 bands only).
    NEVER matches openEO_*.tif - those are S2.
    """
    if not folder or not folder.exists():
        return None

    s1_candidates = list(folder.glob("s1_*.tif")) + list(folder.glob("S1_*.tif"))

    for f in s1_candidates:
        if is_valid_s1_file(f):
            return f

    return None


def find_s1_raw_file_in_folder(folder: Path) -> Path:
    """Find a valid non-filled S1 file in folder."""
    if not folder or not folder.exists():
        return None
    s1_candidates = list(folder.glob("s1_*.tif")) + list(folder.glob("S1_*.tif"))
    for f in s1_candidates:
        if "_filled" not in f.name and is_valid_s1_file(f):
            return f
    return None


def scan_existing_pairs(pairs_dir: Path) -> dict:
    result = {
        'inference': None, 'previous': [], 'all_dates': {},
        'all_s2_dates': set(), 'all_s1_dates': set(),
    }
    if not pairs_dir.exists():
        return result

    for folder in sorted(pairs_dir.iterdir()):
        if not folder.is_dir():
            continue
        parsed = parse_pair_folder_name(folder.name)
        if not parsed:
            continue

        date_str = parsed['date']

        s2_file = find_s2_file_in_folder(folder)
        s1_file = find_s1_file_in_folder(folder)
        s2_exists = s2_file is not None
        s1_exists = s1_file is not None

        s2_date = None
        if s2_file:
            m = re.search(r'(\d{4})-?(\d{2})-?(\d{2})', s2_file.name)
            if m:
                s2_date = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

        s1_date = None
        if s1_file:
            m = re.search(r'(\d{4})-?(\d{2})-?(\d{2})', s1_file.name)
            if m:
                s1_date = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

        entry = {
            'date': date_str, 'folder': folder,
            's2_exists': s2_exists, 's1_exists': s1_exists,
            's2_file': s2_file, 's1_file': s1_file,
            's2_date': s2_date or date_str, 's1_date': s1_date,
            'role': parsed['role'], 'index': parsed.get('index'),
        }

        if parsed['role'] == 'inference':
            result['inference'] = entry
        else:
            result['previous'].append(entry)

        result['all_dates'][date_str] = folder
        if s2_exists:
            result['all_s2_dates'].add(date_str)
        if s1_exists and s1_date:
            result['all_s1_dates'].add(s1_date)

    result['previous'].sort(key=lambda x: x.get('index', 99))
    return result


def reorganize_pairs_for_new_inference(
    pairs_dir: Path, new_inference_date: str,
    new_previous_dates: list, existing_info: dict
) -> dict:
    print(f"\n🔄 Reorganizing pairs folder for new inference: {new_inference_date}")

    existing_inference = existing_info.get('inference')
    existing_previous = existing_info.get('previous', [])

    if existing_inference and existing_inference['date'] != new_inference_date:
        old_inf_date = existing_inference['date']
        old_inf_folder = existing_inference['folder']

        max_idx = 0
        for entry in existing_previous:
            if entry['index'] is not None:
                max_idx = max(max_idx, entry['index'])

        for idx in range(max_idx, 0, -1):
            old_name = None
            for entry in existing_previous:
                if entry['index'] == idx:
                    old_name = entry['folder']
                    break
            if old_name and old_name.exists():
                new_idx = idx + 1
                entry_date = parse_pair_folder_name(old_name.name)['date']
                new_name = pairs_dir / f"prev{new_idx:02d}_{entry_date}"
                if old_name != new_name:
                    print(f"   📁 Rename: {old_name.name} → {new_name.name}")
                    if new_name.exists():
                        shutil.rmtree(new_name)
                    old_name.rename(new_name)

        new_prev01_name = pairs_dir / f"prev01_{old_inf_date}"
        if old_inf_folder.exists():
            print(f"   📁 Rename: {old_inf_folder.name} → {new_prev01_name.name}")
            if new_prev01_name.exists():
                shutil.rmtree(new_prev01_name)
            old_inf_folder.rename(new_prev01_name)

    elif existing_inference and existing_inference['date'] == new_inference_date:
        print(f"   ✅ Inference folder already correct: {new_inference_date}")

    current_info = scan_existing_pairs(pairs_dir)
    needed_dates = set([new_inference_date] + new_previous_dates)

    for entry in current_info.get('previous', []):
        if entry['date'] not in needed_dates:
            if entry.get('index', 0) > PREVIOUS_IMAGES_COUNT:
                print(f"   🗑️  Removing excess: {entry['folder'].name}")
                shutil.rmtree(entry['folder'])

    current_info = scan_existing_pairs(pairs_dir)

    target_folders = {}
    target_folders[new_inference_date] = pairs_dir / f"inference_{new_inference_date}"
    for i, d in enumerate(new_previous_dates, 1):
        target_folders[d] = pairs_dir / f"prev{i:02d}_{d}"

    dates_already_have_s2 = set()
    dates_already_have_s1 = set()

    for folder in pairs_dir.iterdir():
        if not folder.is_dir():
            continue
        parsed = parse_pair_folder_name(folder.name)
        if not parsed:
            continue

        folder_date = parsed['date']

        if folder_date in target_folders:
            target_path = target_folders[folder_date]

            if folder != target_path:
                print(f"   📁 Rename: {folder.name} → {target_path.name}")
                if target_path.exists() and target_path != folder:
                    shutil.rmtree(target_path)
                folder.rename(target_path)
                folder = target_path

            if find_s2_file_in_folder(folder) is not None:
                dates_already_have_s2.add(folder_date)
            if find_s1_file_in_folder(folder) is not None:
                dates_already_have_s1.add(folder_date)

    all_needed_dates = [new_inference_date] + new_previous_dates
    s2_to_download = [d for d in all_needed_dates if d not in dates_already_have_s2]

    return {
        'dates_already_have_s2': dates_already_have_s2,
        'dates_already_have_s1': dates_already_have_s1,
        's2_dates_to_download': s2_to_download,
        'target_folders': target_folders,
    }


# =============================================================================
# DATE RANGE FUNCTIONS
# =============================================================================
def validate_and_build_date_range(start: datetime, end: datetime) -> list:
    delta = (end - start).days
    if delta < 0:
        raise ValueError(f"End date must be after start date")
    if delta > 4:
        print(f"⚠️  Range is {delta+1} days. Clamping to 5 from start.")
        end = start + timedelta(days=4)
    dates = []
    current = start
    while current <= end:
        dates.append(current)
        current += timedelta(days=1)
    while len(dates) < 5:
        dates.append(dates[-1] + timedelta(days=1))
    return dates


def get_lookback_range(ref: datetime, days: int = 30):
    return ref - timedelta(days=days), ref


# =============================================================================
# AOI & SPATIAL
# =============================================================================
def load_aoi_geometry():
    if not AOI_GEOJSON.exists():
        raise FileNotFoundError(f"AOI not found: {AOI_GEOJSON}")

    print(f"📍 Loading AOI: {AOI_GEOJSON}")
    gdf = gpd.read_file(AOI_GEOJSON)
    if gdf.crs is None:
        gdf = gdf.set_crs(TARGET_CRS)
    elif gdf.crs.to_string() != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)

    geom = gdf.geometry.iloc[0]
    bounds = geom.bounds
    print(f"✅ CRS: {gdf.crs}")
    print(f"📍 Bounds: W={bounds[0]:.6f} S={bounds[1]:.6f} "
          f"E={bounds[2]:.6f} N={bounds[3]:.6f}")

    centroid = geom.centroid
    utm_zone = int((centroid.x + 180) / 6) + 1
    utm_epsg = 32600 + utm_zone if centroid.y >= 0 else 32700 + utm_zone
    area_km2 = gdf.to_crs(epsg=utm_epsg).geometry.area.iloc[0] / 1e6
    print(f"📏 Area: {area_km2:.4f} km²")

    geojson_geom = json.loads(gdf.to_json())["features"][0]["geometry"]

    return gdf, {
        "west": bounds[0], "south": bounds[1],
        "east": bounds[2], "north": bounds[3], "crs": TARGET_CRS,
    }, geojson_geom


def add_margin(extent: dict, margin: float) -> dict:
    result = {
        "west": extent["west"] - margin, "south": extent["south"] - margin,
        "east": extent["east"] + margin, "north": extent["north"] + margin,
        "crs": "EPSG:4326",
    }
    print(f"📐 MARGIN: {margin}° (~{margin*111:.0f}m)")
    return result


def get_mgrs_tiles(aoi_geom) -> list:
    if not HAS_MGRS:
        return []
    m = mgrs.MGRS()
    tiles = set()
    b = aoi_geom.bounds
    step = 0.05
    x = b[0]
    while x <= b[2]:
        y = b[1]
        while y <= b[3]:
            try:
                tiles.add(m.toMGRS(y, x, MGRSPrecision=0)[:5])
            except:
                pass
            y += step
        x += step
    for lon, lat in [(b[0],b[1]),(b[0],b[3]),(b[2],b[1]),(b[2],b[3]),
                     ((b[0]+b[2])/2,(b[1]+b[3])/2)]:
        try:
            tiles.add(m.toMGRS(lat, lon, MGRSPrecision=0)[:5])
        except:
            pass
    return sorted(tiles)


def parse_footprint(fp_str: str):
    if not fp_str:
        return None
    try:
        fp_str = fp_str.strip()
        if fp_str.startswith("geography'"):
            m = re.search(r"SRID=\d+;(.+)'$", fp_str)
            if m:
                fp_str = m.group(1)
        if fp_str.upper().startswith(('POLYGON', 'MULTIPOLYGON')):
            return wkt.loads(fp_str)
        if fp_str.startswith('{'):
            return shape(json.loads(fp_str))
    except:
        pass
    return None


# =============================================================================
# ODATA QUERIES
# =============================================================================
def query_s2_products(spatial_extent: dict, start: str, end: str) -> dict:
    ODATA = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
    w, s, e, n = spatial_extent["west"], spatial_extent["south"], spatial_extent["east"], spatial_extent["north"]

    filt = (
        "Collection/Name eq 'SENTINEL-2' and "
        "Attributes/OData.CSC.StringAttribute/any("
        "att:att/Name eq 'productType' and "
        "att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and "
        f"ContentDate/Start ge {start}T00:00:00.000Z and "
        f"ContentDate/Start le {end}T23:59:59.999Z and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;"
        f"POLYGON(({w} {s},{e} {s},{e} {n},{w} {n},{w} {s}))')"
    )

    print(f"\n📡 Querying S2 L2A: {start} → {end}")
    dates_products = defaultdict(dict)

    try:
        resp = requests.get(ODATA, params={
            "$filter": filt, "$top": 1000,
            "$orderby": "ContentDate/Start desc", "$expand": "Attributes",
        }, timeout=120)
        resp.raise_for_status()
        products = resp.json().get("value", [])
        print(f"📦 {len(products)} S2 products found")

        for prod in products:
            name = prod.get("Name", "")
            dm = re.search(r"_(\d{8})T\d{6}_", name)
            tm = re.search(r"_T(\d{2}[A-Z]{3})_", name)
            if not dm or not tm:
                continue
            d = dm.group(1)
            date_str = f"{d[:4]}-{d[4:6]}-{d[6:8]}"
            tile_id = tm.group(1)
            fp = parse_footprint(prod.get("Footprint","") or prod.get("GeoFootprint",""))

            cloud = None
            for attr in prod.get("Attributes", []):
                if "cloudcover" in attr.get("Name","").lower():
                    try:
                        cloud = float(attr.get("Value", 0))
                    except:
                        pass
                    break

            dates_products[date_str][tile_id] = {
                'product_name': name, 'product_id': prod.get("Id",""),
                'footprint': fp, 'cloud_cover': cloud,
            }
    except requests.exceptions.RequestException as ex:
        print(f"❌ OData error: {ex}")

    return dict(dates_products)


def query_s1_products(spatial_extent: dict, start: str, end: str) -> dict:
    ODATA = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
    w, s, e, n = spatial_extent["west"], spatial_extent["south"], spatial_extent["east"], spatial_extent["north"]

    filt = (
        "Collection/Name eq 'SENTINEL-1' and "
        "Attributes/OData.CSC.StringAttribute/any("
        "att:att/Name eq 'productType' and "
        "att/OData.CSC.StringAttribute/Value eq 'IW_GRDH_1S') and "
        f"ContentDate/Start ge {start}T00:00:00.000Z and "
        f"ContentDate/Start le {end}T23:59:59.999Z and "
        f"OData.CSC.Intersects(area=geography'SRID=4326;"
        f"POLYGON(({w} {s},{e} {s},{e} {n},{w} {n},{w} {s}))')"
    )

    print(f"\n📡 Querying S1 GRD: {start} → {end}")
    dates_products = defaultdict(list)

    try:
        resp = requests.get(ODATA, params={
            "$filter": filt, "$top": 1000,
            "$orderby": "ContentDate/Start desc",
        }, timeout=120)
        resp.raise_for_status()
        products = resp.json().get("value", [])
        print(f"📦 {len(products)} S1 products found")

        for prod in products:
            name = prod.get("Name", "")
            dm = re.search(r"_(\d{8})T\d{6}_", name)
            if dm:
                d = dm.group(1)
                date_str = f"{d[:4]}-{d[4:6]}-{d[6:8]}"
                dates_products[date_str].append({
                    'product_name': name, 'product_id': prod.get("Id",""),
                })
    except requests.exceptions.RequestException as ex:
        print(f"❌ OData error: {ex}")

    return dict(dates_products)


# =============================================================================
# COVERAGE CHECK (NO cloud filtering)
# =============================================================================
def check_spatial_coverage(date: str, products: dict, aoi_geom) -> dict:
    if not products:
        return {'date': date, 'spatial_coverage_pct': 0.0, 'is_complete': False,
                'tiles': [], 'avg_cloud_cover': None, 'per_tile_cloud': {},
                'reason': 'No products'}

    aoi_area = aoi_geom.area
    valid_fps, tiles, clouds, ptc = [], [], [], {}

    for tid, info in products.items():
        fp = info.get('footprint')
        cc = info.get('cloud_cover')
        if fp is not None and fp.intersects(aoi_geom):
            valid_fps.append(fp)
            tiles.append(tid)
            ptc[tid] = cc
            if cc is not None:
                clouds.append(cc)

    cov = 0.0
    if valid_fps:
        try:
            cov = (unary_union(valid_fps).intersection(aoi_geom).area / aoi_area) * 100
        except:
            pass

    avg_cloud = np.mean(clouds) if clouds else None
    complete = cov >= MIN_SPATIAL_COVERAGE_PCT

    return {
        'date': date, 'spatial_coverage_pct': cov, 'is_complete': complete,
        'tiles': tiles, 'avg_cloud_cover': avg_cloud, 'per_tile_cloud': ptc,
        'reason': 'OK' if complete else f'Spatial: {cov:.1f}%'
    }


# =============================================================================
# PREVIEW VALIDATION (nodata + cloud measurement, NO filtering on cloud)
# =============================================================================
def validate_nodata_preview(date: str, spatial_extent: dict,
                            connection=None, resolution: int = PREVIEW_RESOLUTION) -> dict:
    if not HAS_OPENEO or not HAS_RASTERIO:
        return {'date': date, 'validated': False, 'nodata_pct': None,
                'reason': 'openEO/rasterio N/A'}
    try:
        if connection is None:
            connection = openeo.connect(OPENEO_BACKEND)
            connection.authenticate_oidc()

        scl = connection.load_collection(
            "SENTINEL2_L2A", spatial_extent=spatial_extent,
            temporal_extent=[date, date], bands=["SCL"]
        ).resample_spatial(resolution=resolution)

        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
            tmp_path = tmp.name
        try:
            scl.download(tmp_path, format="GTiff")
            with rasterio.open(tmp_path) as src:
                data = src.read(1)
                total = data.size
                nd_mask = (data == 0) | (data == -32768) | (data == 255)
                nd_pct = (np.sum(nd_mask) / total) * 100
                cl_mask = (data == 8) | (data == 9) | (data == 10)
                cl_pct = (np.sum(cl_mask) / total) * 100
                valid_mask = (data >= 4) & (data <= 7)
                valid_pct = (np.sum(valid_mask) / total) * 100

            os.unlink(tmp_path)
            return {
                'date': date, 'validated': True,
                'nodata_pct': nd_pct, 'cloud_pct_scl': cl_pct,
                'valid_pct': valid_pct,
                'is_complete': nd_pct < MAX_NODATA_PCT,
                'reason': 'OK' if nd_pct < MAX_NODATA_PCT else f'NoData:{nd_pct:.1f}%'
            }
        except Exception as e:
            if os.path.exists(tmp_path):
                os.unlink(tmp_path)
            raise e
    except Exception as e:
        return {'date': date, 'validated': False, 'nodata_pct': None,
                'reason': f'Error: {str(e)[:80]}'}


# =============================================================================
# PROGRESS & DOWNLOAD HELPERS
# =============================================================================
def format_time(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds // 60:.0f}m {seconds % 60:.0f}s"
    else:
        return f"{seconds // 3600:.0f}h {(seconds % 3600) // 60:.0f}m"


def print_progress_bar(progress: float, width: int = 40, status: str = ""):
    filled = int(width * progress)
    bar = "█" * filled + "░" * (width - filled)
    percentage = progress * 100
    sys.stdout.write(f"\r   [{bar}] {percentage:5.1f}% {status}")
    sys.stdout.flush()


def wait_for_job_with_progress(job, job_name: str, timeout: int = JOB_TIMEOUT) -> bool:
    start_time = time.time()
    last_status = None
    progress_map = {
        "created": 0.05, "queued": 0.10, "running": 0.15,
        "finished": 1.0, "error": -1, "canceled": -1
    }

    print(f"\n   ⏳ Job submitted. Tracking progress...")
    print(f"   Job ID: {job.job_id}")

    while True:
        elapsed = time.time() - start_time
        if elapsed > timeout:
            print(f"\n   ❌ Timeout after {format_time(elapsed)}")
            return False

        try:
            status = job.status()
            job_info = job.describe()

            progress = job_info.get("progress", 0)
            if isinstance(progress, (int, float)):
                progress = progress / 100 if progress > 1 else progress
            else:
                progress = progress_map.get(status, 0.1)

            if status == "running":
                time_progress = min(0.9, elapsed / 2400)
                progress = max(0.15, min(0.95, time_progress))

            status_text = f"Status: {status} | Elapsed: {format_time(elapsed)}"

            if status != last_status:
                print()
                print(f"   📊 Status changed: {last_status} → {status}")
                last_status = status

            if status == "finished":
                print_progress_bar(1.0, status=status_text)
                print(f"\n   ✅ Job completed in {format_time(elapsed)}")
                return True

            elif status in ["error", "canceled"]:
                print(f"\n   ❌ Job {status}")
                try:
                    logs = job.logs()
                    if logs:
                        print("   Error logs:")
                        for log in logs[-5:]:
                            print(f"      {log}")
                except:
                    pass
                return False

            else:
                print_progress_bar(progress, status=status_text)

            time.sleep(POLL_INTERVAL)

        except Exception as e:
            print(f"\n   ⚠️ Error checking status: {e}")
            time.sleep(POLL_INTERVAL)


def download_with_retry_job(job, output_dir: Path, max_retries: int = MAX_RETRIES) -> bool:
    for attempt in range(max_retries):
        try:
            print(f"\n   📥 Downloading results to: {output_dir}")
            print(f"      Attempt {attempt + 1}/{max_retries}")

            start_time = time.time()
            results = job.get_results()
            results.download_files(output_dir)
            elapsed = time.time() - start_time

            downloaded = list(output_dir.glob("*.tif")) + list(output_dir.glob("*.nc"))
            if downloaded:
                for f in downloaded:
                    size_mb = f.stat().st_size / (1024 * 1024)
                    bands = get_band_count(f)
                    print(f"   ✅ Downloaded: {f.name} ({size_mb:.2f} MB, {bands} bands) in {format_time(elapsed)}")
                return True

            print(f"   ⚠️ No files found after download")

        except Exception as e:
            print(f"   ❌ Download error: {e}")
            if attempt < max_retries - 1:
                wait_time = 30 * (attempt + 1)
                print(f"   ⏳ Waiting {wait_time}s before retry...")
                time.sleep(wait_time)

    return False


def download_with_resume(url: str, output_path: Path, description: str = "", session=None):
    temp_path = output_path.with_suffix('.partial')
    resume_pos = 0
    if temp_path.exists():
        resume_pos = temp_path.stat().st_size
        print(f"            📥 Resuming from {resume_pos / 1024 / 1024:.1f} MB...")

    headers = {}
    if resume_pos > 0:
        headers['Range'] = f'bytes={resume_pos}-'
    mode = 'ab' if resume_pos > 0 else 'wb'

    try:
        if session is None:
            response = requests.get(url, headers=headers, stream=True, timeout=TIMEOUT)
        else:
            response = session.get(url, headers=headers, stream=True, timeout=TIMEOUT)
        response.raise_for_status()

        total_size = resume_pos
        if 'content-length' in response.headers:
            total_size += int(response.headers['content-length'])
        elif 'content-range' in response.headers:
            total_size = int(response.headers['content-range'].split('/')[-1])

        downloaded = resume_pos
        last_print = time.time()

        with open(temp_path, mode) as f:
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if time.time() - last_print > 2:
                        if total_size > 0:
                            percent = (downloaded / total_size) * 100
                            print(f"            📊 Progress: {downloaded/1024/1024:.1f}/{total_size/1024/1024:.1f} MB ({percent:.1f}%)")
                        else:
                            print(f"            📊 Downloaded: {downloaded/1024/1024:.1f} MB")
                        last_print = time.time()

        if temp_path.exists():
            temp_path.rename(output_path)
        file_size = output_path.stat().st_size
        print(f"            ✅ {description} complete ({file_size / 1024 / 1024:.1f} MB)")
        return True

    except (requests.exceptions.ChunkedEncodingError,
            requests.exceptions.ConnectionError,
            requests.exceptions.Timeout) as e:
        print(f"            ⚠️  Download interrupted: {type(e).__name__}")
        return False
    except Exception as e:
        print(f"            ❌ Download error: {type(e).__name__}: {str(e)}")
        if temp_path.exists():
            temp_path.unlink()
        raise


def download_cube_with_resume(conn, cube, output_path: Path, description: str = "", job_options: dict = None):
    print(f"         Creating batch job for {description}...")
    default_options = {}
    if job_options:
        default_options.update(job_options)

    job = cube.create_job(
        title=f"{description}_{output_path.stem}",
        out_format="GTiff",
        job_options=default_options if default_options else None
    )

    print(f"         Starting job {job.job_id}...")
    job.start_and_wait(
        max_poll_interval=60,
        connection_retry_interval=30,
        soft_error_max=10
    )

    print(f"         Job completed, downloading result...")
    results = job.get_results()
    assets = results.get_assets()

    if not assets:
        raise Exception("No assets found in job results")

    asset = assets[0]
    download_url = asset.href
    print(f"         Download URL obtained, starting chunked download...")

    max_attempts = MAX_RETRIES
    for attempt in range(1, max_attempts + 1):
        try:
            print(f"         Attempt {attempt}/{max_attempts}")
            success = download_with_resume(
                download_url, output_path, description,
                session=conn._session if hasattr(conn, '_session') else None
            )
            if success:
                if not output_path.exists():
                    raise Exception("Download reported success but file not found")
                file_size = output_path.stat().st_size
                if file_size == 0:
                    output_path.unlink()
                    raise Exception("Downloaded file is empty")
                return True

            if attempt < max_attempts:
                print(f"         🔄 Retrying in {RETRY_DELAY} seconds...")
                time.sleep(RETRY_DELAY)

        except Exception as e:
            if attempt >= max_attempts:
                raise
            print(f"         ⚠️  Error on attempt {attempt}: {type(e).__name__}")
            print(f"         🔄 Retrying in {RETRY_DELAY} seconds...")
            time.sleep(RETRY_DELAY)

    raise Exception(f"Failed after {max_attempts} attempts")


# =============================================================================
# RASTER VALIDATION
# =============================================================================
def validate_raster(filepath: Path, expected_crs: str = None,
                    expected_bands: int = None) -> dict:
    with rasterio.open(filepath) as src:
        bounds = src.bounds
        crs = src.crs
        res = src.res
        shape_hw = (src.height, src.width)
        data = src.read()
        non_zero_ratio = np.count_nonzero(data) / data.size

        info = {
            "path": filepath, "crs": str(crs), "bounds": bounds,
            "resolution": res, "shape": shape_hw, "bands": src.count,
            "dtype": str(src.dtypes[0]), "non_zero_ratio": non_zero_ratio,
            "min": float(np.nanmin(data)), "max": float(np.nanmax(data)),
            "mean": float(np.nanmean(data)), "valid": True, "issues": []
        }

        if expected_bands is not None and src.count != expected_bands:
            info["issues"].append(
                f"CRITICAL: Expected {expected_bands} bands, got {src.count}")
            info["valid"] = False
        if non_zero_ratio < 0.01:
            info["issues"].append("CRITICAL: >99% zeros - likely data loss")
            info["valid"] = False
        if crs is None:
            info["issues"].append("CRITICAL: No CRS defined")
            info["valid"] = False
        elif expected_crs and str(crs) != expected_crs:
            info["issues"].append(f"WARNING: CRS mismatch - expected {expected_crs}, got {crs}")
        if crs and crs.is_projected:
            if abs(bounds.left) < 180 and abs(bounds.right) < 180:
                if abs(bounds.left) < 1000:
                    info["issues"].append("CRITICAL: Bounds appear to be in degrees but CRS is projected")
                    info["valid"] = False
            if res[0] < 1 or res[1] < 1:
                info["issues"].append(f"CRITICAL: Sub-meter resolution ({res}) - likely transform error")
                info["valid"] = False

        return info


def print_raster_validation(info: dict):
    print(f"\n   📊 Raster Validation: {info['path'].name}")
    print(f"      CRS: {info['crs']}")
    print(f"      Bounds: {info['bounds']}")
    print(f"      Resolution: {info['resolution']}")
    print(f"      Shape: {info['shape']}")
    print(f"      Bands: {info['bands']}")
    print(f"      Non-zero: {info['non_zero_ratio']*100:.1f}%")
    print(f"      Values: min={info['min']:.6f}, max={info['max']:.6f}, mean={info['mean']:.6f}")
    if info['valid']:
        print(f"      ✅ VALID")
    else:
        print(f"      ❌ INVALID")
        for issue in info['issues']:
            print(f"         ⚠️  {issue}")


# =============================================================================
# S2 DOWNLOAD - BATCH JOB METHOD
# =============================================================================
def download_s2_to_folder(date: str, target_folder: Path,
                          spatial_extent: dict, connection=None,
                          date_index: int = 1, total_dates: int = 1) -> Path:
    """Download S2 data using batch job with progress tracking and retry."""
    target_folder.mkdir(parents=True, exist_ok=True)

    existing = find_s2_file_in_folder(target_folder)
    if existing:
        print(f"   ⏭️  S2 exists: {existing.name} ({get_band_count(existing)} bands)")
        return existing

    print("\n" + "=" * 70)
    print(f"🛰️  SENTINEL-2 L2A DOWNLOAD [{date_index}/{total_dates}]")
    print("=" * 70)
    print(f"   Date: {date}")
    print(f"   Output Dir: {target_folder}")
    print(f"   Bands: {S2_EXPECTED_BANDS} bands ({', '.join(S2_BANDS)})")
    print(f"   Resolution: {TARGET_RESOLUTION}m")
    print(f"   CRS: {TARGET_CRS}")
    print(f"   Margin: {MARGIN_DEGREES} degrees (~{MARGIN_DEGREES * 111:.0f}m)")

    temporal_extent = [date, date]

    print(f"\n   📍 Spatial Extent (with margin):")
    print(f"      West:  {spatial_extent['west']:.6f}")
    print(f"      South: {spatial_extent['south']:.6f}")
    print(f"      East:  {spatial_extent['east']:.6f}")
    print(f"      North: {spatial_extent['north']:.6f}")

    try:
        if connection is None:
            connection = openeo.connect(OPENEO_BACKEND)
            connection.authenticate_oidc()

        print("\n   📦 Building S2 data cube...")
        s2_cube = connection.load_collection(
            S2_COLLECTION,
            spatial_extent=spatial_extent,
            temporal_extent=temporal_extent,
            bands=S2_BANDS
        )

        print(f"   🔄 Resampling to {TARGET_RESOLUTION}m...")
        s2_cube = s2_cube.resample_spatial(resolution=TARGET_RESOLUTION)

        print("\n   🚀 Creating batch job...")
        job = s2_cube.create_job(
            title=f"S2_L2A_{date}",
            description=f"Sentinel-2 L2A {S2_EXPECTED_BANDS} bands for {date}",
            out_format="GTiff",
            job_options={
                "driver-memory": "4g",
                "executor-memory": "4g"
            }
        )

        job.start_job()
        print(f"   ✅ Job started: {job.job_id}")

        if not wait_for_job_with_progress(job, "S2"):
            return None

        if not download_with_retry_job(job, target_folder):
            return None

        actual_file = find_s2_file_in_folder(target_folder)
        if actual_file:
            bands = get_band_count(actual_file)
            print(f"\n   🎉 S2 download complete for {date}!")
            print(f"   📁 File: {actual_file.name} "
                  f"({actual_file.stat().st_size/(1024*1024):.2f} MB, {bands} bands)")
            if bands != S2_EXPECTED_BANDS:
                print(f"   ⚠️  WARNING: Expected {S2_EXPECTED_BANDS} bands, got {bands}")
            return actual_file
        else:
            all_tifs = list(target_folder.glob("*.tif"))
            for f in sorted(all_tifs, key=lambda x: x.stat().st_mtime, reverse=True):
                bands = get_band_count(f)
                if bands == S2_EXPECTED_BANDS:
                    print(f"\n   🎉 S2 download complete for {date}!")
                    print(f"   📁 File: {f.name} "
                          f"({f.stat().st_size/(1024*1024):.2f} MB, {bands} bands)")
                    return f
            print(f"   ❌ No valid S2 file ({S2_EXPECTED_BANDS} bands) found after download")
            return None

    except Exception as e:
        print(f"\n   ❌ S2 Error: {e}")
        import traceback
        traceback.print_exc()
        return None


# =============================================================================
# S1 DOWNLOAD - RAW GRD WITH 60-DAY MEDIAN COMPOSITE
# =============================================================================
def download_s1_to_folder(date: str, target_folder: Path,
                          spatial_extent: dict, connection=None,
                          date_index: int = 1, total_dates: int = 1) -> Path:
    """
    Download S1 data as RAW GRD with 60-day median composite.
    Uses ascending orbit, no sar_backscatter to avoid LUT errors.
    ONLY checks for s1_*/S1_* prefixed files. NEVER matches openEO_* (those are S2).
    """
    target_folder.mkdir(parents=True, exist_ok=True)

    existing = find_s1_raw_file_in_folder(target_folder)
    if existing:
        print(f"   ⏭️  S1 exists: {existing.name} ({get_band_count(existing)} bands)")
        return existing

    existing_any = find_s1_file_in_folder(target_folder)
    if existing_any:
        print(f"   ⏭️  S1 exists: {existing_any.name} ({get_band_count(existing_any)} bands)")
        return existing_any

    date_compact = date.replace('-', '')

    s1_dt = datetime.strptime(date, "%Y-%m-%d")
    s1_start = (s1_dt - timedelta(days=S1_DATE_BUFFER)).strftime("%Y-%m-%d")
    s1_end = date

    print(f"\n{'='*70}")
    print(f"📡 S1 Download - RAW GRD Method (no sar_backscatter) [{date_index}/{total_dates}]")
    print(f"{'='*70}")
    print(f"   ⚠️  Bypassing calibration to avoid LUT metadata errors")
    print(f"   📅 S1 Target Date: {date}")
    print(f"   📅 Search Window: {s1_start} to {s1_end} ({S1_DATE_BUFFER} days composite)")
    print(f"   📊 Bands: {S1_BANDS} ({S1_EXPECTED_BANDS} bands)")
    print(f"   🎯 Target Resolution: {TARGET_RESOLUTION}m")
    print(f"   📐 CRS: {TARGET_CRS}")
    print(f"   📁 Output Dir: {target_folder}")

    print(f"\n   📍 Spatial Extent (with margin):")
    print(f"      West:  {spatial_extent['west']:.6f}")
    print(f"      South: {spatial_extent['south']:.6f}")
    print(f"      East:  {spatial_extent['east']:.6f}")
    print(f"      North: {spatial_extent['north']:.6f}")

    final_output = target_folder / f"s1_{date_compact}.tif"

    if final_output.exists():
        print(f"   🗑️  Removing existing file: {final_output.name}")
        final_output.unlink()

    try:
        if connection is None:
            connection = openeo.connect(OPENEO_BACKEND)
            connection.authenticate_oidc()

        print(f"\n   📦 Loading S1 GRD collection (raw, no calibration)...")

        s1_cube = connection.load_collection(
            S1_COLLECTION,
            spatial_extent=spatial_extent,
            temporal_extent=[s1_start, s1_end],
            bands=S1_BANDS,
            properties={
                "sat:orbit_state": lambda x: x == "ascending"
            }
        )

        print(f"   🔄 Applying nodata mask and temporal median composite ({S1_DATE_BUFFER} days)...")
        s1_cube = s1_cube.reduce_dimension(
            dimension="t",
            reducer="median"
        )

        print(f"   🔄 Resampling to {TARGET_RESOLUTION}m...")
        s1_cube = s1_cube.resample_spatial(resolution=TARGET_RESOLUTION)

        job_options = {
            "soft-errors": "true",
            "tile-size": 512
        }

        download_cube_with_resume(
            connection, s1_cube, final_output,
            f"S1 RAW GRD {date}",
            job_options=job_options
        )

        if final_output.exists():
            info = validate_raster(final_output, expected_bands=S1_EXPECTED_BANDS)
            print_raster_validation(info)

            if info['bands'] != S1_EXPECTED_BANDS:
                print(f"   ❌ CRITICAL: Expected {S1_EXPECTED_BANDS} bands, got {info['bands']}")
                print(f"   ❌ This file is NOT a valid S1 product!")
                final_output.unlink()
                return None

            if info['non_zero_ratio'] < 0.01:
                print(f"   ⚠️  Very low valid data percentage")
                return final_output

            print(f"\n   🎉 S1 download complete for {date}!")
            print(f"   📁 File: {final_output.name} "
                  f"({final_output.stat().st_size/(1024*1024):.2f} MB, "
                  f"{info['bands']} bands)")
            return final_output
        else:
            print(f"   ❌ S1 file not found after download")
            return None

    except Exception as e:
        print(f"\n   ❌ S1 Error: {type(e).__name__}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


# =============================================================================
# NODATA MEASUREMENT
# =============================================================================
def measure_nodata(filepath: Path) -> dict:
    if not HAS_RASTERIO or not filepath or not filepath.exists():
        return {'nodata_pct': None, 'per_band': {}, 'band_count': 0}
    try:
        with rasterio.open(filepath) as src:
            total, total_nd, per_band = 0, 0, {}
            band_count = src.count
            for b in range(1, src.count + 1):
                data = src.read(b)
                n = data.size
                total += n
                nd_val = src.nodata
                if nd_val is not None:
                    bnd = np.sum(data == nd_val)
                else:
                    bnd = np.sum(np.isnan(data) | (data == 0) | (data == -9999))
                total_nd += bnd
                per_band[f"band_{b}"] = {
                    'nodata_pct': (bnd / n) * 100,
                    'nodata_pixels': int(bnd), 'total_pixels': int(n)
                }
            return {
                'nodata_pct': (total_nd / total) * 100 if total > 0 else 0,
                'per_band': per_band,
                'band_count': band_count
            }
    except Exception as e:
        print(f"   ⚠️  Nodata measure error: {e}")
        return {'nodata_pct': None, 'per_band': {}, 'band_count': 0}


# =============================================================================
# NEAREST S1
# =============================================================================
def find_nearest_s1(s1_dates: list, target: str):
    t = datetime.strptime(target, "%Y-%m-%d")
    best, best_diff = None, float("inf")
    for d in s1_dates:
        diff = abs((datetime.strptime(d, "%Y-%m-%d") - t).days)
        if diff < best_diff:
            best_diff = diff
            best = d
    return best, best_diff


# =============================================================================
# ======================== MAIN EXECUTION ====================================
# =============================================================================

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║  S1/S2 Downloader - PAIRS ONLY                                             ║
║                                                                             ║
║  • Downloads directly into pairs/ folder                                    ║
║  • Auto-renames on re-run (inference→prev01, shift up)                     ║
║  • Checks existing folders to skip re-downloads                             ║
║  • No cloud filtering - only spatial completeness                           ║
║  • Broken image allowed for inference if no complete exists                 ║
║  • S2: Batch job download with progress + retry                             ║
║  • S1: RAW GRD 60-day median composite (no sar_backscatter)                ║
║  • Band validation: S2=15 bands, S1=2 bands (VV+VH)                        ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# ---- STEP 0: Dates ----
print("=" * 80)
print("STEP 0: DATE CONFIGURATION")
print("=" * 80)

inference_window = validate_and_build_date_range(fecha_inicio, fecha_fin)
inference_window_strs = [d.strftime("%Y-%m-%d") for d in inference_window]

lookback_start, lookback_end = get_lookback_range(fecha_fin, LOOKBACK_DAYS)
lb_start_str = lookback_start.strftime("%Y-%m-%d")
lb_end_str = lookback_end.strftime("%Y-%m-%d")

print(f"\n📅 INFERENCE WINDOW (5 days):")
for d in inference_window_strs:
    print(f"   {d}")
print(f"\n🔍 LOOKBACK: {lb_start_str} → {lb_end_str}")

print(f"\n📊 BAND CONFIGURATION:")
print(f"   S2: {S2_EXPECTED_BANDS} bands → {', '.join(S2_BANDS)}")
print(f"   S1: {S1_EXPECTED_BANDS} bands → {', '.join(S1_BANDS)}")


# ---- STEP 1: AOI ----
print("\n" + "=" * 80)
print("STEP 1: LOAD AOI")
print("=" * 80)

gdf, spatial_extent, geojson_geom = load_aoi_geometry()
aoi_geom = gdf.geometry.iloc[0]

bounds_original = {
    "west": spatial_extent["west"], "south": spatial_extent["south"],
    "east": spatial_extent["east"], "north": spatial_extent["north"]
}

spatial_extent = add_margin(spatial_extent, MARGIN_DEGREES)
aoi_with_margin = box(
    spatial_extent["west"], spatial_extent["south"],
    spatial_extent["east"], spatial_extent["north"]
)

print(f"\n📋 BOUNDS COMPARISON ({TARGET_CRS}):")
print(f"   {'Coordinate':<12} {'Original':<18} {'With Margin':<18} {'Difference':<12}")
print(f"   {'-'*60}")
print(f"   {'West':<12} {bounds_original['west']:<18.6f} {spatial_extent['west']:<18.6f} {MARGIN_DEGREES:<12}")
print(f"   {'South':<12} {bounds_original['south']:<18.6f} {spatial_extent['south']:<18.6f} {MARGIN_DEGREES:<12}")
print(f"   {'East':<12} {bounds_original['east']:<18.6f} {spatial_extent['east']:<18.6f} {MARGIN_DEGREES:<12}")
print(f"   {'North':<12} {bounds_original['north']:<18.6f} {spatial_extent['north']:<18.6f} {MARGIN_DEGREES:<12}")


# ---- STEP 2: Scan Existing Pairs ----
print("\n" + "=" * 80)
print("STEP 2: SCAN EXISTING PAIRS FOLDER")
print("=" * 80)

existing_info = scan_existing_pairs(PAIRS_DIR)

print(f"\n📂 Pairs directory: {PAIRS_DIR.absolute()}")
if existing_info['inference']:
    ei = existing_info['inference']
    s2_bands = get_band_count(ei.get('s2_file')) if ei.get('s2_file') else 0
    s1_bands = get_band_count(ei.get('s1_file')) if ei.get('s1_file') else 0
    print(f"   Current inference: {ei['date']} "
          f"(S2:{'✅' if ei['s2_exists'] else '❌'}{f'[{s2_bands}b]' if s2_bands else ''} "
          f"S1:{'✅' if ei['s1_exists'] else '❌'}{f'[{s1_bands}b]' if s1_bands else ''})")
else:
    print("   No existing inference folder")

if existing_info['previous']:
    print(f"   Previous folders: {len(existing_info['previous'])}")
    for ep in existing_info['previous']:
        s2_bands = get_band_count(ep.get('s2_file')) if ep.get('s2_file') else 0
        s1_bands = get_band_count(ep.get('s1_file')) if ep.get('s1_file') else 0
        print(f"      prev{ep['index']:02d}_{ep['date']} "
              f"(S2:{'✅' if ep['s2_exists'] else '❌'}{f'[{s2_bands}b]' if s2_bands else ''} "
              f"S1:{'✅' if ep['s1_exists'] else '❌'}{f'[{s1_bands}b]' if s1_bands else ''})")
else:
    print("   No existing previous folders")

print(f"\n   All dates with valid S2 ({S2_EXPECTED_BANDS}b): {sorted(existing_info['all_s2_dates'])}")
print(f"   All dates with valid S1 ({S1_EXPECTED_BANDS}b): {sorted(existing_info['all_s1_dates'])}")


# ---- STEP 3: Query S2 ----
print("\n" + "=" * 80)
print("STEP 3: QUERY S2 PRODUCTS")
print("=" * 80)

s2_products = query_s2_products(spatial_extent, lb_start_str, lb_end_str)
if not s2_products:
    raise SystemExit("❌ No S2 products found!")
print(f"📊 S2 on {len(s2_products)} dates")


# ---- STEP 4: Spatial Check ----
print("\n" + "=" * 80)
print("STEP 4: SPATIAL COVERAGE (NO CLOUD FILTER)")
print("=" * 80)

spatial_results = {}
print(f"\n{'Date':<12} {'Coverage':<10} {'Tiles':<6} {'Cloud%':<10} "
      f"{'InWin':<7} {'Complete'}")
print("-" * 65)

for date in sorted(s2_products.keys(), reverse=True):
    r = check_spatial_coverage(date, s2_products[date], aoi_with_margin)
    spatial_results[date] = r
    iw = "✅" if date in inference_window_strs else "  "
    cl = f"{r['avg_cloud_cover']:.1f}%" if r['avg_cloud_cover'] is not None else "N/A"
    st = "✅" if r['is_complete'] else "❌"
    print(f"{date:<12} {r['spatial_coverage_pct']:.1f}%{'':>4} "
          f"{len(r['tiles']):<6} {cl:<10} {iw:<7} {st} {r['reason']}")

complete_dates = sorted([d for d, r in spatial_results.items() if r['is_complete']], reverse=True)
print(f"\n📊 Complete: {len(complete_dates)} | "
      f"Incomplete: {len(spatial_results) - len(complete_dates)}")


# ---- STEP 5: Determine Inference Date ----
print("\n" + "=" * 80)
print("STEP 5: DETERMINE INFERENCE DATE")
print("=" * 80)

inf_candidates = [d for d in complete_dates if d in inference_window_strs]
inference_date = None
inference_is_broken = False

if inf_candidates:
    inference_date = max(inf_candidates)
    print(f"\n✅ INFERENCE: {inference_date} (latest complete in window)")
else:
    print(f"\n⚠️  No complete image in inference window!")
    window_avail = [d for d in inference_window_strs if d in spatial_results]
    if window_avail:
        best = max(window_avail, key=lambda d: spatial_results[d]['spatial_coverage_pct'])
        inference_date = best
        inference_is_broken = True
        print(f"   🔧 Using BROKEN: {inference_date} "
              f"(coverage: {spatial_results[best]['spatial_coverage_pct']:.1f}%)")
    else:
        pre = [d for d in complete_dates if d < inference_window_strs[0]]
        if pre:
            inference_date = pre[0]
            print(f"   ⚠️  Using nearest before window: {inference_date}")
        else:
            raise SystemExit("❌ No inference date available!")

print(f"\n🎯 INFERENCE DATE: {inference_date} "
      f"{'⚠️ BROKEN' if inference_is_broken else '✅ COMPLETE'}")


# ---- STEP 6: Select Previous 5 ----
print("\n" + "=" * 80)
print("STEP 6: SELECT PREVIOUS 5 COMPLETE DATES")
print("=" * 80)

prev_candidates = [d for d in complete_dates if d != inference_date and d <= inference_date]
previous_dates = prev_candidates[:PREVIOUS_IMAGES_COUNT]

print(f"\n📋 Selected {len(previous_dates)} previous dates:")
for i, d in enumerate(previous_dates, 1):
    sr = spatial_results[d]
    cl = f"{sr['avg_cloud_cover']:.1f}%" if sr['avg_cloud_cover'] is not None else "N/A"
    print(f"   {i}. {d} | Coverage: {sr['spatial_coverage_pct']:.1f}% | Cloud: {cl}")


# ---- STEP 7: Preview Validation ----
print("\n" + "=" * 80)
print("STEP 7: OPENEO PREVIEW VALIDATION (NoData + Cloud %)")
print("=" * 80)

all_target_dates = [inference_date] + previous_dates

dates_needing_validation = [
    d for d in all_target_dates
    if d not in existing_info['all_s2_dates']
]
if inference_date not in dates_needing_validation:
    dates_needing_validation.insert(0, inference_date)
dates_needing_validation = sorted(list(set(dates_needing_validation)))

preview_results = {}
DATE_METADATA_S2 = {}

if HAS_OPENEO and dates_needing_validation:
    print(f"\n🔍 Validating {len(dates_needing_validation)} dates...")
    try:
        val_conn = openeo.connect(OPENEO_BACKEND)
        val_conn.authenticate_oidc()
        print("✅ Connected")

        for i, date in enumerate(dates_needing_validation, 1):
            print(f"   [{i}/{len(dates_needing_validation)}] {date}...", end=" ")
            r = validate_nodata_preview(date, spatial_extent, val_conn)
            preview_results[date] = r
            if r.get('validated'):
                nd = r.get('nodata_pct', 0)
                cl = r.get('cloud_pct_scl', 0)
                print(f"NoData:{nd:.1f}% Cloud:{cl:.1f}% "
                      f"{'✅' if r.get('is_complete') else '⚠️'}")
            else:
                print(f"⚠️  {r.get('reason')}")
    except Exception as e:
        print(f"\n❌ OpenEO error: {e}")
else:
    print("   Skipping validation (all validated or openEO N/A)")

# Build S2 metadata
for date in all_target_dates:
    meta = DateMetadata(date, "S2")
    meta.is_inference = (date == inference_date)
    meta.is_broken_allowed = (date == inference_date and inference_is_broken)
    meta.band_count = S2_EXPECTED_BANDS

    if date in spatial_results:
        sr = spatial_results[date]
        meta.spatial_coverage_pct = sr['spatial_coverage_pct']
        meta.tiles = sr['tiles']
        meta.cloud_cover_pct = sr['avg_cloud_cover']
        meta.is_complete = sr['is_complete']

    if date in preview_results:
        pr = preview_results[date]
        if pr.get('validated'):
            meta.nodata_pct = pr.get('nodata_pct')
            meta.valid_pixel_pct = pr.get('valid_pct')
            if pr.get('cloud_pct_scl') is not None:
                meta.cloud_cover_pct = pr.get('cloud_pct_scl')
            if not meta.is_inference:
                meta.is_complete = pr.get('is_complete', meta.is_complete)

    DATE_METADATA_S2[date] = meta

# Re-evaluate previous dates
valid_prev = [d for d in previous_dates
              if DATE_METADATA_S2[d].is_complete or DATE_METADATA_S2[d].is_complete is None]
invalid_prev = [d for d in previous_dates if d not in valid_prev]

if len(valid_prev) < PREVIOUS_IMAGES_COUNT:
    extra = [d for d in complete_dates
             if d != inference_date and d not in valid_prev
             and d not in invalid_prev and d <= inference_date]
    for d in extra[:PREVIOUS_IMAGES_COUNT - len(valid_prev)]:
        valid_prev.append(d)
        if d not in DATE_METADATA_S2:
            m = DateMetadata(d, "S2")
            m.band_count = S2_EXPECTED_BANDS
            sr = spatial_results.get(d, {})
            m.spatial_coverage_pct = sr.get('spatial_coverage_pct')
            m.tiles = sr.get('tiles', [])
            m.cloud_cover_pct = sr.get('avg_cloud_cover')
            m.is_complete = True
            DATE_METADATA_S2[d] = m

previous_dates = sorted(valid_prev, reverse=True)[:PREVIOUS_IMAGES_COUNT]

print(f"\n📋 FINAL SELECTION:")
print(f"   Inference: {inference_date} {'⚠️ BROKEN' if inference_is_broken else '✅'}")
for i, d in enumerate(previous_dates, 1):
    m = DATE_METADATA_S2.get(d)
    cl = f"{m.cloud_cover_pct:.1f}%" if m and m.cloud_cover_pct is not None else "N/A"
    nd = f"{m.nodata_pct:.1f}%" if m and m.nodata_pct is not None else "N/A"
    print(f"   Prev {i}: {d} | Cloud: {cl} | NoData: {nd}")


# ---- STEP 8: Query & Match S1 ----
print("\n" + "=" * 80)
print("STEP 8: QUERY & MATCH S1")
print("=" * 80)

s1_start = (lookback_start - timedelta(days=12)).strftime("%Y-%m-%d")
s1_end = (datetime.strptime(lb_end_str, "%Y-%m-%d") + timedelta(days=12)).strftime("%Y-%m-%d")

s1_products = query_s1_products(spatial_extent, s1_start, s1_end)
all_s1_dates = sorted(s1_products.keys(), reverse=True)
print(f"📊 S1 dates: {len(all_s1_dates)}")

# Build pairs
download_pairs = []
DATE_METADATA_S1 = {}

all_download_dates = [inference_date] + previous_dates

print(f"\n{'Role':<12} {'S2 Date':<12} {'S1 Date':<12} {'Δ Days'}")
print("-" * 48)

for i, s2d in enumerate(all_download_dates):
    s1d, s1diff = find_nearest_s1(all_s1_dates, s2d) if all_s1_dates else (None, None)
    role = "inference" if s2d == inference_date else "previous"
    pidx = 0 if role == "inference" else (previous_dates.index(s2d) + 1 if s2d in previous_dates else i)

    pair = {'s2_date': s2d, 's1_date': s1d, 's1_diff_days': s1diff,
            'role': role, 'prev_index': pidx}
    download_pairs.append(pair)

    if s1d and s1d not in DATE_METADATA_S1:
        m = DateMetadata(s1d, "S1")
        m.is_inference = (role == "inference")
        m.band_count = S1_EXPECTED_BANDS
        DATE_METADATA_S1[s1d] = m

    r = "INFERENCE" if role == "inference" else f"PREV {pidx:02d}"
    print(f"{r:<12} {s2d:<12} {s1d or 'N/A':<12} {s1diff if s1diff is not None else 'N/A'}")


# ---- STEP 9: Reorganize Existing Folders ----
print("\n" + "=" * 80)
print("STEP 9: REORGANIZE PAIRS FOLDER")
print("=" * 80)

PAIRS_DIR.mkdir(parents=True, exist_ok=True)

reorg = reorganize_pairs_for_new_inference(
    PAIRS_DIR, inference_date, previous_dates, existing_info
)

target_folders = reorg['target_folders']

# Determine S1 downloads needed - use validated file detection
s1_dates_needed = {}  # s1_date -> s2_date (for folder lookup)
for pair in download_pairs:
    if pair['s1_date']:
        s1_dates_needed[pair['s1_date']] = pair['s2_date']

# Check which S1 dates already have valid files (2 bands)
s1_already = set()
for s1_date, s2_date in s1_dates_needed.items():
    folder = target_folders.get(s2_date)
    if folder and find_s1_file_in_folder(folder) is not None:
        s1_already.add(s1_date)

# Also scan all pair folders for S1 files that might be in wrong folder
for folder in PAIRS_DIR.iterdir():
    if folder.is_dir():
        s1_file = find_s1_raw_file_in_folder(folder)
        if s1_file:
            m_match = re.search(r'(\d{4})-?(\d{2})-?(\d{2})', s1_file.name)
            if m_match:
                s1_date_str = f"{m_match.group(1)}-{m_match.group(2)}-{m_match.group(3)}"
                s1_already.add(s1_date_str)

s2_to_download = reorg['s2_dates_to_download']
s1_to_download = sorted([d for d in s1_dates_needed.keys() if d not in s1_already])

print(f"\n📋 DOWNLOAD PLAN:")
print(f"   S2 to download : {len(s2_to_download)} {s2_to_download}")
print(f"   S2 already have: {sorted(reorg['dates_already_have_s2'])}")
print(f"   S1 to download : {len(s1_to_download)} {s1_to_download}")
print(f"   S1 already have: {sorted(s1_already)}")
print(f"   S2 expected bands: {S2_EXPECTED_BANDS}")
print(f"   S1 expected bands: {S1_EXPECTED_BANDS}")


# ---- STEP 10: Execute Downloads ----
print("\n" + "=" * 80)
print("STEP 10: EXECUTE DOWNLOADS")
print("=" * 80)

download_results = {
    's2_downloaded': [], 's2_skipped': [], 's2_failed': [],
    's1_downloaded': [], 's1_skipped': [], 's1_failed': [],
}

total_start_time = time.time()

if s2_to_download or s1_to_download:
    print(f"\n🔌 Connecting to openEO...")
    try:
        dl_conn = openeo.connect(OPENEO_BACKEND)
        dl_conn.authenticate_oidc()
        print("✅ Connected and authenticated!")
        user_info = dl_conn.describe_account()
        print(f"   User: {user_info.get('user_id', 'N/A')}")

        # =========================================================
        # Download S2 using BATCH JOB method
        # =========================================================
        if s2_to_download:
            print(f"\n{'='*70}")
            print(f"📥 STARTING S2 DOWNLOADS ({len(s2_to_download)} dates)")
            print(f"   Expected: {S2_EXPECTED_BANDS} bands per file")
            print(f"{'='*70}")

            for i, date in enumerate(s2_to_download, 1):
                folder = target_folders.get(date)
                if not folder:
                    print(f"   ⚠️  No target folder for S2 {date}")
                    download_results['s2_failed'].append(date)
                    continue

                print(f"\n{'#'*70}")
                print(f"# Processing S2 date {i}/{len(s2_to_download)}: {date}")
                print(f"# Output folder: {folder}")
                print(f"{'#'*70}")

                result = download_s2_to_folder(
                    date, folder, spatial_extent, dl_conn,
                    date_index=i, total_dates=len(s2_to_download)
                )

                if result:
                    download_results['s2_downloaded'].append(date)
                    if date in DATE_METADATA_S2:
                        DATE_METADATA_S2[date].download_path = result
                        DATE_METADATA_S2[date].file_size_mb = result.stat().st_size/(1024*1024)
                        DATE_METADATA_S2[date].band_count = get_band_count(result)
                else:
                    download_results['s2_failed'].append(date)

                elapsed = time.time() - total_start_time
                print(f"\n   📊 S2 Progress: {i}/{len(s2_to_download)} | "
                      f"Success: {len(download_results['s2_downloaded'])} | "
                      f"Failed: {len(download_results['s2_failed'])} | "
                      f"Elapsed: {format_time(elapsed)}")

        # =========================================================
        # Download S1 using RAW GRD + 60-day median composite
        # =========================================================
        if s1_to_download:
            print(f"\n{'='*70}")
            print(f"📥 STARTING S1 DOWNLOADS ({len(s1_to_download)} dates)")
            print(f"   Method: RAW GRD with {S1_DATE_BUFFER}-day median composite")
            print(f"   No sar_backscatter (avoids LUT errors)")
            print(f"   Expected: {S1_EXPECTED_BANDS} bands per file (VV + VH)")
            print(f"{'='*70}")

            for i, s1_date in enumerate(s1_to_download, 1):
                # Find which pair folder this S1 goes into
                target_folder = None
                for pair in download_pairs:
                    if pair['s1_date'] == s1_date:
                        target_folder = target_folders.get(pair['s2_date'])
                        break

                if not target_folder:
                    print(f"   ⚠️  No target folder for S1 {s1_date}")
                    download_results['s1_failed'].append(s1_date)
                    continue

                print(f"\n{'#'*70}")
                print(f"# Processing S1 {i}/{len(s1_to_download)}: {s1_date}")
                print(f"# Output folder: {target_folder}")
                print(f"{'#'*70}")

                result = download_s1_to_folder(
                    s1_date, target_folder, spatial_extent, dl_conn,
                    date_index=i, total_dates=len(s1_to_download)
                )

                if result:
                    bands = get_band_count(result)
                    if bands == S1_EXPECTED_BANDS:
                        download_results['s1_downloaded'].append(s1_date)
                        if s1_date in DATE_METADATA_S1:
                            DATE_METADATA_S1[s1_date].download_path = result
                            DATE_METADATA_S1[s1_date].file_size_mb = result.stat().st_size/(1024*1024)
                            DATE_METADATA_S1[s1_date].band_count = bands
                    else:
                        print(f"   ❌ Downloaded file has {bands} bands, expected {S1_EXPECTED_BANDS}")
                        download_results['s1_failed'].append(s1_date)
                else:
                    download_results['s1_failed'].append(s1_date)

                elapsed = time.time() - total_start_time
                print(f"\n   📊 S1 Progress: {i}/{len(s1_to_download)} | "
                      f"Success: {len(download_results['s1_downloaded'])} | "
                      f"Failed: {len(download_results['s1_failed'])} | "
                      f"Elapsed: {format_time(elapsed)}")

    except Exception as e:
        print(f"\n❌ Connection error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n✅ All files already downloaded!")

total_download_time = time.time() - total_start_time

# Update paths for existing files using validated finders
for pair in download_pairs:
    folder = target_folders.get(pair['s2_date'])
    if not folder or not folder.exists():
        continue

    # S2 files - use validated finder
    s2_file = find_s2_file_in_folder(folder)
    if s2_file and pair['s2_date'] in DATE_METADATA_S2:
        DATE_METADATA_S2[pair['s2_date']].download_path = s2_file
        DATE_METADATA_S2[pair['s2_date']].file_size_mb = s2_file.stat().st_size/(1024*1024)
        DATE_METADATA_S2[pair['s2_date']].band_count = get_band_count(s2_file)

    # S1 files - use validated finder (NEVER matches openEO_*)
    if pair['s1_date']:
        s1_file = find_s1_file_in_folder(folder)
        if s1_file and pair['s1_date'] in DATE_METADATA_S1:
            DATE_METADATA_S1[pair['s1_date']].download_path = s1_file
            DATE_METADATA_S1[pair['s1_date']].file_size_mb = s1_file.stat().st_size/(1024*1024)
            DATE_METADATA_S1[pair['s1_date']].band_count = get_band_count(s1_file)


# ---- STEP 11: Measure NoData ----
print("\n" + "=" * 80)
print("STEP 11: MEASURE NODATA IN DOWNLOADED FILES")
print("=" * 80)

print("\n📊 S2 NoData:")
print(f"{'Date':<12} {'Role':<12} {'Bands':<7} {'NoData%':<10} {'Cloud%':<10} {'Size MB':<10} {'Status'}")
print("-" * 75)

for date in [inference_date] + previous_dates:
    meta = DATE_METADATA_S2.get(date)
    if not meta:
        continue
    role = "INFERENCE" if meta.is_inference else "PREVIOUS"

    if meta.download_path and meta.download_path.exists():
        nd = measure_nodata(meta.download_path)
        if nd['nodata_pct'] is not None:
            meta.nodata_pct = nd['nodata_pct']
        meta.band_count = nd.get('band_count', meta.band_count)

    bands_s = str(meta.band_count) if meta.band_count else "N/A"
    nd_s = f"{meta.nodata_pct:.2f}%" if meta.nodata_pct is not None else "N/A"
    cl_s = f"{meta.cloud_cover_pct:.1f}%" if meta.cloud_cover_pct is not None else "N/A"
    sz_s = f"{meta.file_size_mb:.1f}" if meta.file_size_mb else "N/A"
    st = "✅" if meta.is_complete else "⚠️ BROKEN"
    print(f"{date:<12} {role:<12} {bands_s:<7} {nd_s:<10} {cl_s:<10} {sz_s:<10} {st}")

print(f"\n📊 S1 NoData:")
print(f"{'Date':<12} {'Role':<12} {'Bands':<7} {'NoData%':<10} {'Size MB':<10} {'File':<30} {'Status'}")
print("-" * 95)

for date in sorted(DATE_METADATA_S1.keys(), reverse=True):
    meta = DATE_METADATA_S1[date]
    role = "INFERENCE" if meta.is_inference else "PREVIOUS"

    if meta.download_path and meta.download_path.exists():
        # Verify it's actually S1
        actual_bands = get_band_count(meta.download_path)
        if actual_bands != S1_EXPECTED_BANDS:
            print(f"{date:<12} {role:<12} {actual_bands:<7} {'N/A':<10} {'N/A':<10} "
                  f"{meta.download_path.name:<30} ❌ WRONG FILE ({actual_bands}b≠{S1_EXPECTED_BANDS}b)")
            meta.download_path = None
            meta.band_count = actual_bands
            continue

        nd = measure_nodata(meta.download_path)
        if nd['nodata_pct'] is not None:
            meta.nodata_pct = nd['nodata_pct']
        meta.band_count = nd.get('band_count', S1_EXPECTED_BANDS)
        filename = meta.download_path.name
    else:
        filename = "NOT FOUND"

    nd_s = f"{meta.nodata_pct:.2f}%" if meta.nodata_pct is not None else "N/A"
    sz_s = f"{meta.file_size_mb:.1f}" if meta.file_size_mb else "N/A"
    bands_s = str(meta.band_count) if meta.band_count else "N/A"
    has_nd = meta.nodata_pct is not None and meta.nodata_pct > 0.1
    has_file = meta.download_path is not None and meta.download_path.exists()
    meta.is_complete = has_file and not has_nd
    st = "⚠️ HAS NODATA" if has_nd else ("✅" if has_file else "❌ MISSING")
    print(f"{date:<12} {role:<12} {bands_s:<7} {nd_s:<10} {sz_s:<10} {filename:<30} {st}")


# ---- STEP 12: S1 NoData Summary (no fill) ----
print("\n" + "=" * 80)
print("STEP 12: S1 NODATA SUMMARY")
print("=" * 80)

s1_with_nodata = [
    d for d, m in DATE_METADATA_S1.items()
    if m.nodata_pct is not None and m.nodata_pct > 0.1
    and m.download_path and m.download_path.exists()
    and is_valid_s1_file(m.download_path)
]

if s1_with_nodata:
    print(f"\n📊 S1 dates with nodata > 0.1%: {s1_with_nodata}")
    for date in s1_with_nodata:
        meta = DATE_METADATA_S1[date]
        print(f"   • {date}: {meta.nodata_pct:.2f}% nodata ({meta.band_count} bands)")
    print(f"   ℹ️  No median fill applied (disabled)")
else:
    print("✅ All S1 files have minimal nodata")

s1_missing = [
    d for d, m in DATE_METADATA_S1.items()
    if m.download_path is None or not m.download_path.exists()
]
if s1_missing:
    print(f"\n⚠️  S1 dates with NO valid file ({S1_EXPECTED_BANDS} bands): {s1_missing}")


# ---- FINAL SUMMARY ----
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

s2_success_rate = (len(download_results['s2_downloaded']) / len(s2_to_download) * 100) if s2_to_download else 100
s1_success_rate = (len(download_results['s1_downloaded']) / len(s1_to_download) * 100) if s1_to_download else 100

# Count actual valid files
s2_valid_count = sum(1 for p in download_pairs if find_s2_file_in_folder(target_folders.get(p['s2_date'])))
s1_valid_count = sum(1 for p in download_pairs if p['s1_date'] and find_s1_file_in_folder(target_folders.get(p['s2_date'])))

print(f"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║  📊 COMPLETE                                                                    ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  Inference Window       : {inference_window_strs[0]} to {inference_window_strs[-1]:<33} ║
║  Inference Date         : {inference_date:<49} ║
║  Inference Broken?      : {'YES ⚠️' if inference_is_broken else 'NO  ✅':<49} ║
║  Previous Images        : {len(previous_dates):<49} ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  S2 Downloaded (new)    : {len(download_results['s2_downloaded']):<49} ║
║  S2 Skipped (existing)  : {len(all_target_dates) - len(s2_to_download):<49} ║
║  S2 Failed              : {len(download_results['s2_failed']):<49} ║
║  S2 Success Rate        : {f'{s2_success_rate:.1f}%':<49} ║
║  S2 Valid Files (total) : {f'{s2_valid_count}/{len(download_pairs)} ({S2_EXPECTED_BANDS} bands each)':<49} ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  S1 Downloaded (new)    : {len(download_results['s1_downloaded']):<49} ║
║  S1 Skipped (existing)  : {len(s1_dates_needed) - len(s1_to_download):<49} ║
║  S1 Failed              : {len(download_results['s1_failed']):<49} ║
║  S1 Success Rate        : {f'{s1_success_rate:.1f}%':<49} ║
║  S1 Valid Files (total) : {f'{s1_valid_count}/{len(download_pairs)} ({S1_EXPECTED_BANDS} bands each)':<49} ║
║  S1 NoData Filled       : {'N/A (disabled)':<49} ║
║  S1 Composite Window    : {f'{S1_DATE_BUFFER} days median':<49} ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  CRS                    : {TARGET_CRS:<49} ║
║  Margin                 : {f'{MARGIN_DEGREES} degrees (~{MARGIN_DEGREES * 111:.0f}m)':<49} ║
║  S1 Method              : {'RAW GRD (no sar_backscatter)':<49} ║
║  Total Download Time    : {format_time(total_download_time):<49} ║
╚══════════════════════════════════════════════════════════════════════════════════╝
""")

# Per-date metadata table
print("=" * 115)
print("📊 PER-DATE METADATA")
print("=" * 115)
print(f"\n{'Role':<12} {'S2 Date':<12} {'S2 Bands':<10} {'S2 Cloud%':<12} {'S2 NoData%':<12} "
      f"{'S1 Date':<12} {'S1 Bands':<10} {'S1 NoData%':<12} {'Status'}")
print("-" * 105)

for pair in download_pairs:
    s2d = pair['s2_date']
    s1d = pair.get('s1_date', 'N/A')
    role = pair['role'].upper()
    if role == "PREVIOUS":
        role = f"PREV {pair.get('prev_index', '?'):02d}"

    s2m = DATE_METADATA_S2.get(s2d)
    s1m = DATE_METADATA_S1.get(s1d) if s1d != 'N/A' else None

    s2b = str(s2m.band_count) if s2m and s2m.band_count else "N/A"
    s2cl = f"{s2m.cloud_cover_pct:.1f}%" if s2m and s2m.cloud_cover_pct is not None else "N/A"
    s2nd = f"{s2m.nodata_pct:.2f}%" if s2m and s2m.nodata_pct is not None else "N/A"

    s1b = str(s1m.band_count) if s1m and s1m.band_count else "N/A"
    s1nd = f"{s1m.nodata_pct:.2f}%" if s1m and s1m.nodata_pct is not None else "N/A"

    # Status check
    s2_ok = s2m and s2m.download_path and s2m.download_path.exists() and s2m.band_count == S2_EXPECTED_BANDS
    s1_ok = s1m and s1m.download_path and s1m.download_path.exists() and s1m.band_count == S1_EXPECTED_BANDS
    status = "✅" if (s2_ok and s1_ok) else ("⚠️ S1 MISSING" if s2_ok and not s1_ok else "❌")
    broken = " ⚠️BROKEN" if s2m and s2m.is_broken_allowed else ""

    print(f"{role:<12} {s2d:<12} {s2b:<10} {s2cl:<12} {s2nd:<12} "
          f"{s1d or 'N/A':<12} {s1b:<10} {s1nd:<12} {status}{broken}")

print("-" * 105)

# Print failed dates if any
if download_results['s2_failed']:
    print(f"\n❌ S2 FAILED DATES ({len(download_results['s2_failed'])}):")
    for fecha in download_results['s2_failed']:
        print(f"   - {fecha}")

if download_results['s1_failed']:
    print(f"\n❌ S1 FAILED DATES ({len(download_results['s1_failed'])}):")
    for fecha in download_results['s1_failed']:
        print(f"   - {fecha}")

# Folder structure
print("\n📁 FOLDER STRUCTURE:")
print(f"   {PAIRS_DIR}/")
if PAIRS_DIR.exists():
    for folder in sorted(PAIRS_DIR.iterdir()):
        if folder.is_dir():
            parsed = parse_pair_folder_name(folder.name)
            if parsed:
                files = list(folder.glob("*.tif"))
                print(f"   └── {folder.name}/")
                for f in sorted(files):
                    sz = f.stat().st_size / (1024*1024)
                    bands = get_band_count(f)
                    sat_type = "S2" if bands == S2_EXPECTED_BANDS else ("S1" if bands == S1_EXPECTED_BANDS else f"?{bands}b")
                    print(f"       └── {f.name} ({sz:.1f} MB, {bands}b={sat_type})")

# List all files summary
print(f"\n📁 ALL S2 FILES (expected {S2_EXPECTED_BANDS} bands):")
for pair in download_pairs:
    folder = target_folders.get(pair['s2_date'])
    if folder and folder.exists():
        s2_file = find_s2_file_in_folder(folder)
        if s2_file:
            size_mb = s2_file.stat().st_size / (1024 * 1024)
            bands = get_band_count(s2_file)
            ok = "✅" if bands == S2_EXPECTED_BANDS else f"⚠️ {bands}b"
            print(f"   {ok} {s2_file.relative_to(PAIRS_DIR)}: {size_mb:.2f} MB, {bands} bands")
        else:
            print(f"   ❌ {folder.name}/: No valid S2 file")
    else:
        print(f"   ❌ {pair['s2_date']}: Folder not found")

print(f"\n📁 ALL S1 FILES (expected {S1_EXPECTED_BANDS} bands):")
for pair in download_pairs:
    folder = target_folders.get(pair['s2_date'])
    if folder and folder.exists():
        s1_file = find_s1_file_in_folder(folder)
        if s1_file:
            size_mb = s1_file.stat().st_size / (1024 * 1024)
            bands = get_band_count(s1_file)
            ok = "✅" if bands == S1_EXPECTED_BANDS else f"⚠️ {bands}b"
            print(f"   {ok} {s1_file.relative_to(PAIRS_DIR)}: {size_mb:.2f} MB, {bands} bands")
        else:
            s1d = pair.get('s1_date', 'N/A')
            print(f"   ❌ {folder.name}/: No valid S1 file (S1 date: {s1d})")
    elif pair.get('s1_date'):
        print(f"   ❌ {pair['s2_date']}: Folder not found")


# =============================================================================
# STORE VARIABLES
# =============================================================================
INFERENCE_DATE = inference_date
INFERENCE_IS_BROKEN = inference_is_broken
PREVIOUS_DATES = previous_dates
DOWNLOAD_PAIRS = download_pairs
INFERENCE_WINDOW = inference_window_strs

CLOUD_PCT_PER_DATE = {d: m.cloud_cover_pct for d, m in DATE_METADATA_S2.items()}
NODATA_PCT_S2_PER_DATE = {d: m.nodata_pct for d, m in DATE_METADATA_S2.items()}
NODATA_PCT_S1_PER_DATE = {d: m.nodata_pct for d, m in DATE_METADATA_S1.items()}

ALL_S2_DATES = [inference_date] + previous_dates
ALL_S1_DATES = sorted(list(set([p['s1_date'] for p in download_pairs if p['s1_date']])), reverse=True)

SPATIAL_EXTENT = spatial_extent
AOI_GEOMETRY = aoi_geom
TARGET_FOLDERS = target_folders
DOWNLOAD_RESULTS = download_results

# Build file path dictionaries using VALIDATED finders
S2_FILE_PATHS = {}
S1_FILE_PATHS = {}
for pair in download_pairs:
    folder = target_folders.get(pair['s2_date'])
    if not folder or not folder.exists():
        continue

    # S2 files - validated finder
    s2_file = find_s2_file_in_folder(folder)
    if s2_file:
        S2_FILE_PATHS[pair['s2_date']] = s2_file

    # S1 files - validated finder (NEVER matches openEO_*)
    if pair['s1_date']:
        s1_file = find_s1_file_in_folder(folder)
        if s1_file:
            S1_FILE_PATHS[pair['s2_date']] = {
                's1_date': pair['s1_date'],
                's1_file': s1_file
            }

print("\n" + "=" * 80)
print("📦 VARIABLES FOR DOWNSTREAM USE")
print("=" * 80)
print(f"""
  INFERENCE_DATE         : {INFERENCE_DATE}
  INFERENCE_IS_BROKEN    : {INFERENCE_IS_BROKEN}
  PREVIOUS_DATES         : {PREVIOUS_DATES}
  INFERENCE_WINDOW       : {INFERENCE_WINDOW}

  CLOUD_PCT_PER_DATE     :""")
for d, v in sorted(CLOUD_PCT_PER_DATE.items()):
    r = "INF" if d == INFERENCE_DATE else "PRV"
    print(f"    {d}: {f'{v:.1f}%' if v is not None else 'N/A':>8}  [{r}]")

print(f"\n  NODATA_PCT_S2_PER_DATE :")
for d, v in sorted(NODATA_PCT_S2_PER_DATE.items()):
    r = "INF" if d == INFERENCE_DATE else "PRV"
    print(f"    {d}: {f'{v:.2f}%' if v is not None else 'N/A':>8}  [{r}]")

print(f"\n  NODATA_PCT_S1_PER_DATE :")
for d, v in sorted(NODATA_PCT_S1_PER_DATE.items()):
    print(f"    {d}: {f'{v:.2f}%' if v is not None else 'N/A':>8}")

print(f"""
  ALL_S2_DATES           : {ALL_S2_DATES}
  ALL_S1_DATES           : {ALL_S1_DATES}
  PAIRS_DIR              : {PAIRS_DIR.absolute()}
  TARGET_FOLDERS         :""")
for d, f in sorted(TARGET_FOLDERS.items()):
    r = "INF" if d == INFERENCE_DATE else "PRV"
    print(f"    {d} → {f.name}  [{r}]")

print(f"""
  DATE_METADATA_S2       : {len(DATE_METADATA_S2)} dates
  DATE_METADATA_S1       : {len(DATE_METADATA_S1)} dates
""")

for d in sorted(DATE_METADATA_S2.keys(), reverse=True):
    print(f"   {DATE_METADATA_S2[d]}")
for d in sorted(DATE_METADATA_S1.keys(), reverse=True):
    print(f"   {DATE_METADATA_S1[d]}")

# S2 and S1 file path summaries with band validation
print(f"\n📁 S2_FILE_PATHS dictionary ({len(S2_FILE_PATHS)} files):")
for fecha, path in S2_FILE_PATHS.items():
    bands = get_band_count(path)
    ok = "✅" if bands == S2_EXPECTED_BANDS else f"⚠️ {bands}b"
    print(f"   {ok} '{fecha}': '{path}'")

print(f"\n📁 S1_FILE_PATHS dictionary ({len(S1_FILE_PATHS)} files):")
for s2_date, info in S1_FILE_PATHS.items():
    bands = get_band_count(info['s1_file'])
    ok = "✅" if bands == S1_EXPECTED_BANDS else f"⚠️ {bands}b"
    print(f"   {ok} '{s2_date}': {{'s1_date': '{info['s1_date']}', 's1_file': '{info['s1_file']}'}}")

# Report completeness
print(f"\n📊 PAIR COMPLETENESS:")
print(f"   S2 files: {len(S2_FILE_PATHS)}/{len(download_pairs)} pairs")
print(f"   S1 files: {len(S1_FILE_PATHS)}/{len(download_pairs)} pairs")
missing_s1_pairs = [p['s2_date'] for p in download_pairs if p['s2_date'] not in S1_FILE_PATHS and p['s1_date']]
if missing_s1_pairs:
    print(f"   ⚠️  Pairs missing S1: {missing_s1_pairs}")
    print(f"   💡 These S1 files need to be downloaded (re-run may fix)")

print("\n✅ ALL DONE")
print("=" * 80)

# =============================================================================
# EXPLICIT GLOBAL EXPORT FOR CELL 2
# =============================================================================
print("\n" + "=" * 80)
print("🔗 EXPORTING VARIABLES TO GLOBAL SCOPE")
print("=" * 80)

globals().update({
    'INFERENCE_DATE': INFERENCE_DATE,
    'INFERENCE_IS_BROKEN': INFERENCE_IS_BROKEN,
    'PREVIOUS_DATES': PREVIOUS_DATES,
    'DOWNLOAD_PAIRS': DOWNLOAD_PAIRS,
    'INFERENCE_WINDOW': INFERENCE_WINDOW,
    'TARGET_FOLDERS': TARGET_FOLDERS,
    'PAIRS_DIR': PAIRS_DIR,
    'CLOUD_PCT_PER_DATE': CLOUD_PCT_PER_DATE,
    'NODATA_PCT_S2_PER_DATE': NODATA_PCT_S2_PER_DATE,
    'NODATA_PCT_S1_PER_DATE': NODATA_PCT_S1_PER_DATE,
    'ALL_S2_DATES': ALL_S2_DATES,
    'ALL_S1_DATES': ALL_S1_DATES,
    'SPATIAL_EXTENT': SPATIAL_EXTENT,
    'AOI_GEOMETRY': AOI_GEOMETRY,
    'DOWNLOAD_RESULTS': DOWNLOAD_RESULTS,
    'DATE_METADATA_S2': DATE_METADATA_S2,
    'DATE_METADATA_S1': DATE_METADATA_S1,
    'S2_FILE_PATHS': S2_FILE_PATHS,
    'S1_FILE_PATHS': S1_FILE_PATHS,
    'S2_EXPECTED_BANDS': S2_EXPECTED_BANDS,
    'S1_EXPECTED_BANDS': S1_EXPECTED_BANDS,
})

print(f"✅ Exported variables:")
print(f"   INFERENCE_DATE       = {INFERENCE_DATE}")
print(f"   PREVIOUS_DATES       = {len(PREVIOUS_DATES)} dates")
print(f"   DOWNLOAD_PAIRS       = {len(DOWNLOAD_PAIRS)} pairs")
print(f"   TARGET_FOLDERS       = {len(TARGET_FOLDERS)} folders")
print(f"   PAIRS_DIR            = {PAIRS_DIR}")
print(f"   DATE_METADATA_S2     = {len(DATE_METADATA_S2)} dates")
print(f"   DATE_METADATA_S1     = {len(DATE_METADATA_S1)} dates")
print(f"   S2_FILE_PATHS        = {len(S2_FILE_PATHS)} files (expected {S2_EXPECTED_BANDS}b each)")
print(f"   S1_FILE_PATHS        = {len(S1_FILE_PATHS)} files (expected {S1_EXPECTED_BANDS}b each)")
print("\n✅ Variables ready for Cell 2!")
print("=" * 80)

%store INFERENCE_DATE INFERENCE_IS_BROKEN PREVIOUS_DATES DOWNLOAD_PAIRS INFERENCE_WINDOW TARGET_FOLDERS PAIRS_DIR CLOUD_PCT_PER_DATE NODATA_PCT_S2_PER_DATE NODATA_PCT_S1_PER_DATE ALL_S2_DATES ALL_S1_DATES SPATIAL_EXTENT DOWNLOAD_RESULTS DATE_METADATA_S2 DATE_METADATA_S1 S2_FILE_PATHS S1_FILE_PATHS


╔══════════════════════════════════════════════════════════════════════════════╗
║  S1/S2 Downloader - PAIRS ONLY                                             ║
║                                                                             ║
║  • Downloads directly into pairs/ folder                                    ║
║  • Auto-renames on re-run (inference→prev01, shift up)                     ║
║  • Checks existing folders to skip re-downloads                             ║
║  • No cloud filtering - only spatial completeness                           ║
║  • Broken image allowed for inference if no complete exists                 ║
║  • S2: Batch job download with progress + retry                             ║
║  • S1: RAW GRD 60-day median composite (no sar_backscatter)                ║
║  • Band validation: S2=15 bands, S1=2 bands (VV+VH)                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

STEP 0: DATE CONFIGURATION

📅 INFERENCE 

# No data handled

In [3]:
%store -r
"""
Cloud Removal Pipeline - Compatible with Pairs Folder Download Structure
Uses INFERENCE_DATE, PREVIOUS_DATES, DOWNLOAD_PAIRS, TARGET_FOLDERS from Cell 1
Downloads go directly into pairs/ folder - no separate S1/S2 dirs
Includes confidence metrics based on prev image count, cloud %, nodata %
Never stops - always produces output + confidence values
"""
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from pathlib import Path
from tqdm import tqdm
import warnings
from scipy import ndimage
from sklearn.ensemble import RandomForestRegressor
import gc
import re

warnings.filterwarnings('ignore')

# Check for GPU availability
GPU_AVAILABLE = False
try:
    import cupy as cp
    import cupyx.scipy.ndimage as cp_ndimage
    GPU_AVAILABLE = True
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    gpu_mem = cp.cuda.runtime.getDeviceProperties(0)['totalGlobalMem'] / (1024**3)
    print(f"✅ GPU available via CuPy. Device: {gpu_name} ({gpu_mem:.1f} GB)")
    mempool = cp.get_default_memory_pool()
    mempool.set_limit(fraction=0.7)
except ImportError:
    print("ℹ️ CuPy not available. Using CPU only.")

print(f"NumPy version: {np.__version__}")
print(f"GPU acceleration (CuPy): {'Enabled' if GPU_AVAILABLE else 'Disabled'}")


# ============================================================================
# CONFIDENCE TRACKING
# ============================================================================
@dataclass
class PipelineConfidence:
    """
    Tracks confidence metrics for the cloud removal pipeline output.
    Confidence is based on:
      - Number of previous images available (within 30 days)
      - Cloud cover % of inference image
      - NoData % of inference image
      - Fill method distribution (temporal vs fusion vs spatial)
    """
    # Input metrics
    inference_date: str = ""
    num_previous_images: int = 0
    max_possible_previous: int = 5
    inference_cloud_cover_pct: float = 0.0
    inference_nodata_pct: float = 0.0
    inference_is_broken: bool = False

    # Per-previous-image info
    previous_image_dates: list = field(default_factory=list)
    previous_image_cloud_pcts: list = field(default_factory=list)
    previous_image_days_gap: list = field(default_factory=list)

    # Fill method breakdown
    total_pixels: int = 0
    clear_pixels: int = 0
    cloud_pixels: int = 0
    temporal_filled: int = 0
    fusion_filled: int = 0
    spatial_filled: int = 0
    unfilled_pixels: int = 0

    # Derived confidence scores (0-100)
    confidence_prev_images: float = 0.0    # Based on count of previous images
    confidence_cloud_cover: float = 0.0    # Based on inference cloud %
    confidence_nodata: float = 0.0         # Based on inference nodata %
    confidence_fill_quality: float = 0.0   # Based on fill method distribution
    confidence_overall: float = 0.0        # Weighted combination

    # Confidence level label
    confidence_level: str = "UNKNOWN"

    def calculate(self):
        """Calculate all confidence scores from collected metrics."""

        # 1. Previous images confidence (0-100)
        #    5 images = 100, 4=90, 3=75, 2=55, 1=30, 0=5
        prev_score_map = {5: 100, 4: 90, 3: 75, 2: 55, 1: 30, 0: 5}
        n = min(self.num_previous_images, 5)
        self.confidence_prev_images = prev_score_map.get(n, 5)

        # 2. Cloud cover confidence (0-100)
        #    0% cloud = 100 confidence, 100% cloud = 5 confidence
        if self.inference_cloud_cover_pct <= 0:
            self.confidence_cloud_cover = 100.0
        elif self.inference_cloud_cover_pct >= 100:
            self.confidence_cloud_cover = 5.0
        else:
            # Non-linear: low cloud = high confidence, degrades faster at high cloud
            self.confidence_cloud_cover = max(
                5.0,
                100.0 - (self.inference_cloud_cover_pct ** 1.3) * 0.8
            )

        # 3. NoData confidence (0-100)
        #    0% nodata = 100, >50% nodata = very low
        if self.inference_nodata_pct <= 0:
            self.confidence_nodata = 100.0
        elif self.inference_nodata_pct >= 50:
            self.confidence_nodata = 5.0
        else:
            self.confidence_nodata = max(
                5.0,
                100.0 - self.inference_nodata_pct * 2.0
            )

        # 4. Fill quality confidence (0-100)
        #    Based on HOW pixels were filled
        #    Temporal = best, Fusion = good, Spatial = fair, Unfilled = bad
        if self.cloud_pixels > 0:
            temporal_ratio = self.temporal_filled / self.cloud_pixels
            fusion_ratio = self.fusion_filled / self.cloud_pixels
            spatial_ratio = self.spatial_filled / self.cloud_pixels
            unfilled_ratio = self.unfilled_pixels / self.cloud_pixels

            # Weighted quality score
            self.confidence_fill_quality = (
                temporal_ratio * 100.0 +    # Temporal = full confidence
                fusion_ratio * 75.0 +       # Fusion = 75% confidence
                spatial_ratio * 40.0 +      # Spatial = 40% confidence
                unfilled_ratio * 0.0        # Unfilled = 0% confidence
            )
            self.confidence_fill_quality = max(5.0, min(100.0, self.confidence_fill_quality))
        else:
            # No clouds = perfect
            self.confidence_fill_quality = 100.0

        # 5. Overall confidence (weighted combination)
        #    Previous images: 30% weight (most important for quality)
        #    Cloud cover:     25% weight
        #    Fill quality:    30% weight
        #    NoData:          15% weight
        self.confidence_overall = (
            self.confidence_prev_images * 0.30 +
            self.confidence_cloud_cover * 0.25 +
            self.confidence_fill_quality * 0.30 +
            self.confidence_nodata * 0.15
        )
        self.confidence_overall = max(0.0, min(100.0, self.confidence_overall))

        # 6. Confidence level label
        if self.confidence_overall >= 85:
            self.confidence_level = "HIGH"
        elif self.confidence_overall >= 65:
            self.confidence_level = "MEDIUM"
        elif self.confidence_overall >= 40:
            self.confidence_level = "LOW"
        elif self.confidence_overall >= 20:
            self.confidence_level = "VERY LOW"
        else:
            self.confidence_level = "MINIMAL"

    def to_dict(self) -> dict:
        return {
            'inference_date': self.inference_date,
            'num_previous_images': self.num_previous_images,
            'max_possible_previous': self.max_possible_previous,
            'inference_cloud_cover_pct': round(self.inference_cloud_cover_pct, 2),
            'inference_nodata_pct': round(self.inference_nodata_pct, 2),
            'inference_is_broken': self.inference_is_broken,
            'previous_image_dates': self.previous_image_dates,
            'previous_image_cloud_pcts': [round(c, 2) for c in self.previous_image_cloud_pcts],
            'previous_image_days_gap': self.previous_image_days_gap,
            'total_pixels': self.total_pixels,
            'clear_pixels': self.clear_pixels,
            'cloud_pixels': self.cloud_pixels,
            'temporal_filled': self.temporal_filled,
            'fusion_filled': self.fusion_filled,
            'spatial_filled': self.spatial_filled,
            'unfilled_pixels': self.unfilled_pixels,
            'confidence_prev_images': round(self.confidence_prev_images, 1),
            'confidence_cloud_cover': round(self.confidence_cloud_cover, 1),
            'confidence_nodata': round(self.confidence_nodata, 1),
            'confidence_fill_quality': round(self.confidence_fill_quality, 1),
            'confidence_overall': round(self.confidence_overall, 1),
            'confidence_level': self.confidence_level,
        }

    def print_report(self):
        """Print a formatted confidence report."""
        level_emoji = {
            "HIGH": "🟢", "MEDIUM": "🟡", "LOW": "🟠",
            "VERY LOW": "🔴", "MINIMAL": "⚫", "UNKNOWN": "⚪"
        }
        emoji = level_emoji.get(self.confidence_level, "⚪")

        print(f"\n{'='*70}")
        print(f"📊 CONFIDENCE REPORT")
        print(f"{'='*70}")
        print(f"")
        print(f"  Inference Date     : {self.inference_date}")
        print(f"  Broken Image       : {'YES ⚠️' if self.inference_is_broken else 'NO ✅'}")
        print(f"")
        print(f"  ┌─────────────────────────────────────────────────────┐")
        print(f"  │  {emoji} OVERALL CONFIDENCE: {self.confidence_overall:.1f}/100 "
              f"({self.confidence_level}){' '*(22 - len(self.confidence_level))}│")
        print(f"  └─────────────────────────────────────────────────────┘")
        print(f"")
        print(f"  Component Scores:")
        print(f"  {'─'*55}")
        print(f"  {'Component':<25} {'Score':>8} {'Weight':>8} {'Contribution':>14}")
        print(f"  {'─'*55}")

        components = [
            ("Prev Images", self.confidence_prev_images, 0.30),
            ("Cloud Cover", self.confidence_cloud_cover, 0.25),
            ("Fill Quality", self.confidence_fill_quality, 0.30),
            ("NoData", self.confidence_nodata, 0.15),
        ]
        for name, score, weight in components:
            contrib = score * weight
            bar = "█" * int(score / 5) + "░" * (20 - int(score / 5))
            print(f"  {name:<25} {score:>7.1f} {weight:>7.0%} {contrib:>13.1f}")

        print(f"  {'─'*55}")
        print(f"  {'OVERALL':<25} {self.confidence_overall:>7.1f} {'100%':>8}")

        print(f"\n  Input Metrics:")
        print(f"    Previous images      : {self.num_previous_images} / "
              f"{self.max_possible_previous}")
        if self.previous_image_dates:
            for i, (d, c, g) in enumerate(zip(
                    self.previous_image_dates,
                    self.previous_image_cloud_pcts,
                    self.previous_image_days_gap)):
                c_str = f"{c:.1f}%" if c is not None else "N/A"
                print(f"      prev{i+1}: {d} (cloud: {c_str}, gap: {g}d)")
        print(f"    Inference cloud%     : {self.inference_cloud_cover_pct:.2f}%")
        print(f"    Inference nodata%    : {self.inference_nodata_pct:.2f}%")

        print(f"\n  Fill Breakdown:")
        print(f"    Total pixels         : {self.total_pixels:,}")
        print(f"    Clear (untouched)    : {self.clear_pixels:,} "
              f"({self.clear_pixels/max(self.total_pixels,1)*100:.1f}%)")
        print(f"    Cloud pixels         : {self.cloud_pixels:,} "
              f"({self.cloud_pixels/max(self.total_pixels,1)*100:.1f}%)")

        if self.cloud_pixels > 0:
            print(f"    ├── Temporal fill    : {self.temporal_filled:,} "
                  f"({self.temporal_filled/self.cloud_pixels*100:.1f}%)")
            print(f"    ├── S1-S2 fusion     : {self.fusion_filled:,} "
                  f"({self.fusion_filled/self.cloud_pixels*100:.1f}%)")
            print(f"    ├── Spatial fill     : {self.spatial_filled:,} "
                  f"({self.spatial_filled/self.cloud_pixels*100:.1f}%)")
            print(f"    └── Unfilled         : {self.unfilled_pixels:,} "
                  f"({self.unfilled_pixels/self.cloud_pixels*100:.1f}%)")

        print(f"\n{'='*70}")


# ============================================================================
# AUTO-DETECT PATHS FROM DOWNLOAD CELL VARIABLES
# ============================================================================
def build_paths_from_pairs(
    inference_date: str,
    previous_dates: list,
    download_pairs: list,
    target_folders: dict,
    pairs_dir: Path
) -> Tuple[dict, dict, str, list]:
    """
    Build S2_FILE_PATHS and S1_FILE_PATHS from the pairs folder structure.
    Handles multiple naming conventions from download code:
      S2: S2_YYYYMMDD.tif, S2_YYYY-MM-DD.tif, openEO_YYYY-MM-DDZ.tif
      S1: s1_YYYYMMDD.tif, S1_YYYYMMDD.tif, S1_YYYY-MM-DD.tif, openEO_YYYY-MM-DDZ.tif
    """
    print(f"\n{'='*70}")
    print(f"📂 BUILDING PATHS FROM PAIRS FOLDER")
    print(f"{'='*70}")

    s2_paths = {}
    s1_paths = {}

    for pair in download_pairs:
        s2_date = pair['s2_date']
        s1_date = pair.get('s1_date')
        date_compact = s2_date.replace('-', '')
        s1_date_compact = s1_date.replace('-', '') if s1_date else None

        folder = target_folders.get(s2_date)
        if folder is None:
            for candidate in pairs_dir.iterdir():
                if candidate.is_dir() and s2_date in candidate.name:
                    folder = candidate
                    break

        if folder is None or not folder.exists():
            print(f"   ⚠️  No folder for {s2_date}")
            continue

        # ---- Find S2 file (multiple naming conventions) ----
        s2_file = None
        s2_search_patterns = [
            f"S2_{date_compact}.tif",           # S2_20260222.tif
            f"S2_{s2_date}.tif",                 # S2_2026-02-22.tif
            f"openEO_{s2_date}Z.tif",            # openEO_2026-02-22Z.tif
        ]
        # Try exact matches first
        for pattern in s2_search_patterns:
            candidate = folder / pattern
            if candidate.exists() and candidate.stat().st_size >= 1000:
                s2_file = candidate
                break

        # Fallback: glob for any S2/openEO tif
        if s2_file is None:
            all_s2 = (list(folder.glob("S2_*.tif")) +
                      list(folder.glob("openEO_*.tif")))
            valid_s2 = [f for f in all_s2 if f.stat().st_size >= 1000]
            if valid_s2:
                s2_file = max(valid_s2, key=lambda f: f.stat().st_size)

        if s2_file:
            s2_paths[s2_date] = str(s2_file)
        else:
            print(f"   ⚠️  No S2 in {folder.name}/")

        # ---- Find S1 file (multiple naming conventions) ----
        if s1_date:
            s1_file = None
            s1_search_patterns = [
                f"s1_{s1_date_compact}.tif",             # s1_20260222.tif (lowercase)
                f"s1_{s1_date_compact}_filled.tif",      # s1_20260222_filled.tif
                f"S1_{s1_date_compact}.tif",             # S1_20260222.tif
                f"S1_{s1_date}.tif",                     # S1_2026-02-22.tif
                f"S1_{s1_date}_filled.tif",              # S1_2026-02-22_filled.tif
                f"openEO_{s1_date}Z.tif",                # openEO_2026-02-22Z.tif
            ]

            # Try exact matches first (prefer filled versions)
            filled_file = None
            regular_file = None
            for pattern in s1_search_patterns:
                candidate = folder / pattern
                if candidate.exists() and candidate.stat().st_size >= 1000:
                    if '_filled' in pattern:
                        filled_file = candidate
                    elif regular_file is None:
                        regular_file = candidate

            # Prefer filled over regular
            s1_file = filled_file or regular_file

            # Fallback: glob for any S1/s1/openEO tif
            if s1_file is None:
                all_s1 = (list(folder.glob("S1_*.tif")) +
                          list(folder.glob("s1_*.tif")) +
                          list(folder.glob("openEO_*.tif")))
                # Exclude files already matched as S2
                if s2_file:
                    all_s1 = [f for f in all_s1 if f != s2_file]
                # Prefer filled
                filled_s1 = [f for f in all_s1
                             if '_filled' in f.name and f.stat().st_size >= 1000]
                regular_s1 = [f for f in all_s1
                              if '_filled' not in f.name and f.stat().st_size >= 1000]
                if filled_s1:
                    s1_file = max(filled_s1, key=lambda f: f.stat().st_size)
                elif regular_s1:
                    s1_file = max(regular_s1, key=lambda f: f.stat().st_size)

            if s1_file:
                # Extract actual S1 date from filename
                actual_s1_date = s1_date
                m = re.match(r'(?:S1_|s1_)(\d{4}-?\d{2}-?\d{2})', s1_file.name)
                if m:
                    d = m.group(1).replace('-', '')
                    actual_s1_date = f"{d[:4]}-{d[4:6]}-{d[6:8]}"
                s1_paths[s2_date] = {
                    's1_date': actual_s1_date, 's1_file': str(s1_file)}
            else:
                print(f"   ⚠️  No valid S1 in {folder.name}/")
        else:
            print(f"   ⚠️  No S1 date for {s2_date}")

    sorted_dates = sorted(s2_paths.keys(), reverse=True)

    print(f"\n📊 Path Resolution:")
    print(f"   {'Date':<14} {'Role':<12} {'S2 File':<40} {'S1 File'}")
    print(f"   {'-'*100}")

    for date in sorted_dates:
        role = "INFERENCE" if date == inference_date else "PREVIOUS"
        s2_name = Path(s2_paths[date]).name
        s1_info = s1_paths.get(date)
        s1_name = Path(s1_info['s1_file']).name if s1_info else "MISSING"
        filled_tag = " [filled]" if s1_info and "filled" in s1_name else ""
        print(f"   {date:<14} {role:<12} {s2_name:<40} {s1_name}{filled_tag}")

    print(f"\n   S2: {len(s2_paths)} | S1: {len(s1_paths)} | Target: {inference_date}")
    return s2_paths, s1_paths, inference_date, sorted_dates


def detect_band_names(s2_filepath: str) -> Tuple[list, list, int]:
    """Detect band names from downloaded S2 file."""
    with rasterio.open(s2_filepath) as src:
        n_bands = src.count
        descriptions = [src.descriptions[i] if src.descriptions[i] else f"band_{i+1}"
                        for i in range(n_bands)]

    print(f"\n🔍 Detecting bands: {Path(s2_filepath).name}")
    print(f"   Bands: {n_bands}, Descriptions: {descriptions}")

    FULL_15 = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07',
               'B08', 'B8A', 'B09', 'B11', 'B12', 'WVP', 'AOT', 'SCL']

    CONFIGS = {
        15: FULL_15,
        13: ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A',
             'B09', 'B11', 'B12', 'AOT', 'SCL'],
        5:  ['B02', 'B03', 'B04', 'B08', 'SCL'],
        6:  ['B02', 'B03', 'B04', 'B08', 'B11', 'SCL'],
    }

    known = {'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07',
             'B08', 'B8A', 'B09', 'B11', 'B12', 'WVP', 'AOT', 'SCL'}

    detected = []
    for desc in descriptions:
        upper = desc.upper().strip()
        if upper in known:
            detected.append(upper)
        else:
            matched = False
            for kb in known:
                if kb in upper:
                    detected.append(kb)
                    matched = True
                    break
            if not matched:
                detected.append(desc)

    if len(set(detected) & known) >= n_bands * 0.5:
        band_names = detected
    elif n_bands in CONFIGS:
        band_names = CONFIGS[n_bands]
    else:
        band_names = FULL_15[:n_bands] if n_bands <= 15 else [
            f"band_{i+1}" for i in range(n_bands)]

    scl_idx = None
    for i, bn in enumerate(band_names):
        if bn.upper() == 'SCL':
            scl_idx = i
            break
    if scl_idx is None:
        scl_idx = n_bands - 1

    print(f"   Band names: {band_names}")
    print(f"   SCL index: {scl_idx}")
    return band_names, ['VV', 'VH'], scl_idx


# ============================================================================
# BUILD PATHS FROM CELL 1
# ============================================================================
print(f"\n{'='*70}")
print(f"🔗 CONNECTING TO DOWNLOAD CELL OUTPUTS")
print(f"{'='*70}")

required_vars = ['INFERENCE_DATE', 'PREVIOUS_DATES', 'DOWNLOAD_PAIRS',
                 'TARGET_FOLDERS', 'PAIRS_DIR']
# NEW (works)
missing_vars = []
for v in required_vars:
    try:
        eval(v)
    except NameError:
        missing_vars.append(v)

if missing_vars:
    print(f"❌ Missing: {missing_vars}")
    raise RuntimeError(f"Run Cell 1 first! Missing: {missing_vars}")

print(f"✅ Cell 1 variables found:")
print(f"   INFERENCE_DATE  : {INFERENCE_DATE}")
print(f"   PREVIOUS_DATES  : {PREVIOUS_DATES}")
print(f"   PAIRS_DIR       : {PAIRS_DIR}")
print(f"   DOWNLOAD_PAIRS  : {len(DOWNLOAD_PAIRS)} pairs")

S2_FILE_PATHS, S1_FILE_PATHS, TARGET_DATE, SORTED_DATES = build_paths_from_pairs(
    INFERENCE_DATE, PREVIOUS_DATES, DOWNLOAD_PAIRS, TARGET_FOLDERS, PAIRS_DIR)

first_s2_path = S2_FILE_PATHS[SORTED_DATES[0]]
BAND_NAMES, S1_BAND_NAMES, SCL_BAND_INDEX = detect_band_names(first_s2_path)

SPECTRAL_BAND_NAMES_FULL = ('B01', 'B02', 'B03', 'B04', 'B05', 'B06',
                             'B07', 'B08', 'B8A', 'B09', 'B11', 'B12')
AUX_BAND_NAMES = ('WVP', 'AOT', 'SCL')

SPECTRAL_BAND_INDICES = []
SPECTRAL_BANDS_AVAILABLE = []
for bn in SPECTRAL_BAND_NAMES_FULL:
    if bn in BAND_NAMES:
        SPECTRAL_BAND_INDICES.append(BAND_NAMES.index(bn))
        SPECTRAL_BANDS_AVAILABLE.append(bn)

AUX_BAND_INDICES = []
for bn in AUX_BAND_NAMES:
    if bn in BAND_NAMES:
        AUX_BAND_INDICES.append(BAND_NAMES.index(bn))

print(f"\n📊 Band Mapping:")
print(f"   Spectral: {len(SPECTRAL_BAND_INDICES)} → {SPECTRAL_BANDS_AVAILABLE}")
print(f"   Aux: {AUX_BAND_INDICES}")
print(f"   SCL: {SCL_BAND_INDEX}")

# Cell 1 metadata
try:
    if 'CLOUD_PCT_PER_DATE' in dir():
        print(f"\n📊 Cell 1 Metadata:")
        for d in SORTED_DATES:
            cl = CLOUD_PCT_PER_DATE.get(d)
            nd_s2 = NODATA_PCT_S2_PER_DATE.get(d)
            role = "INF" if d == INFERENCE_DATE else "PRV"
            cl_s = f"{cl:.1f}%" if cl is not None else "N/A"
            nd_s = f"{nd_s2:.2f}%" if nd_s2 is not None else "N/A"
            print(f"   {d} [{role}] Cloud:{cl_s} NoData:{nd_s}")
except:
    pass


# ============================================================================
# Configuration
# ============================================================================
@dataclass
class CloudRemovalConfig:
    """Configuration for cloud removal pipeline"""
    CLOUD_SCL_CLASSES: Tuple[int, ...] = (8, 9, 10, 3)
    SPECTRAL_BANDS: Tuple[str, ...] = tuple(SPECTRAL_BANDS_AVAILABLE)
    SPECTRAL_BAND_INDICES: Tuple[int, ...] = tuple(SPECTRAL_BAND_INDICES)
    AUX_BAND_INDICES: Tuple[int, ...] = tuple(AUX_BAND_INDICES)
    SCL_BAND_INDEX: int = SCL_BAND_INDEX
    MAX_PREVIOUS_IMAGES: int = 5
    CHUNK_SIZE: int = 1024
    BUFFER_PIXELS: int = 5
    MAX_CHANGE_RATE_PER_DAY: float = 500.0
    MIN_CLEAR_OBSERVATIONS: int = 2
    TRAINING_SAMPLE_FRACTION: float = 0.5
    TRAINING_MAX_SAMPLES: int = 200_000
    MAX_INTERPOLATION_DISTANCE: int = 50
    N_JOBS: int = -1

config = CloudRemovalConfig()

print(f"\n✅ Configuration loaded")
print(f" • Cloud SCL classes: {config.CLOUD_SCL_CLASSES}")
print(f" • Spectral bands: {len(config.SPECTRAL_BANDS)} → {config.SPECTRAL_BANDS}")
print(f" • SCL index: {config.SCL_BAND_INDEX}")
print(f" • Chunk size: {config.CHUNK_SIZE}x{config.CHUNK_SIZE}")


# ============================================================================
# Output Directory
# ============================================================================
OUTPUT_DIR = str(PAIRS_DIR.parent / "cloud_free_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f'cloud_free_{TARGET_DATE}.tif')

print(f"\n📁 Output: {OUTPUT_PATH}")


# ============================================================================
# Utility Functions
# ============================================================================
def parse_date(date_str: str) -> datetime:
    return datetime.strptime(date_str, '%Y-%m-%d')

def days_between(date1: str, date2: str) -> int:
    return abs((parse_date(date1) - parse_date(date2)).days)

def get_previous_dates(target_date: str, all_dates: List[str],
                       max_count: int = 5) -> List[str]:
    target_dt = parse_date(target_date)
    previous = []
    for date in sorted(all_dates, reverse=True):
        if parse_date(date) < target_dt:
            previous.append(date)
            if len(previous) >= max_count:
                break
    return previous

def validate_files_exist() -> bool:
    missing = []
    for date, path in S2_FILE_PATHS.items():
        if not os.path.exists(path):
            missing.append(f"S2 {date}: {path}")
    for date, info in S1_FILE_PATHS.items():
        if not os.path.exists(info['s1_file']):
            missing.append(f"S1 {date}: {info['s1_file']}")
    if missing:
        print("❌ Missing files:")
        for m in missing:
            print(f"   • {m}")
        return False
    print("✅ All files validated")
    return True

def get_image_info(filepath: str) -> dict:
    with rasterio.open(filepath) as src:
        return {
            'shape': (src.count, src.height, src.width),
            'dtype': src.dtypes[0], 'crs': src.crs,
            'transform': src.transform, 'bounds': src.bounds}

validate_files_exist()

print(f"\n📊 Image Info:")
sample_s2 = get_image_info(S2_FILE_PATHS[SORTED_DATES[0]])
print(f" S2 shape: {sample_s2['shape']}, dtype: {sample_s2['dtype']}")
if S1_FILE_PATHS:
    sample_s1 = get_image_info(list(S1_FILE_PATHS.values())[0]['s1_file'])
    print(f" S1 shape: {sample_s1['shape']}")


# ============================================================================
# Cloud Mask Generation
# ============================================================================
def create_cloud_mask(scl_band: np.ndarray, buffer_size: int = 5,
                      cloud_classes: tuple = (3, 8, 9, 10)) -> Tuple[np.ndarray, np.ndarray]:
    if GPU_AVAILABLE:
        scl_gpu = cp.asarray(scl_band)
        cm_gpu = cp.zeros_like(scl_gpu, dtype=cp.bool_)
        for cls in cloud_classes:
            cm_gpu |= (scl_gpu == cls)
        cloud_mask = cp.asnumpy(cm_gpu)
        del scl_gpu, cm_gpu
        cp.get_default_memory_pool().free_all_blocks()
    else:
        cloud_mask = np.isin(scl_band, cloud_classes)

    if buffer_size > 0:
        struct = ndimage.generate_binary_structure(2, 1)
        cloud_mask_buffered = ndimage.binary_dilation(
            cloud_mask, structure=struct, iterations=buffer_size)
    else:
        cloud_mask_buffered = cloud_mask.copy()

    return cloud_mask, cloud_mask_buffered

def analyze_cloud_coverage(scl_band: np.ndarray,
                           cloud_classes: tuple = (3, 8, 9, 10)) -> dict:
    total = scl_band.size
    cloud_mask = np.isin(scl_band, cloud_classes)
    cloud_px = np.sum(cloud_mask)
    return {
        'total_pixels': total, 'cloud_pixels': int(cloud_px),
        'clear_pixels': int(total - cloud_px),
        'cloud_percentage': (cloud_px / total) * 100}

print(f"\n🔍 Analyzing cloud coverage...")
cloud_stats = {}
for date in tqdm(SORTED_DATES, desc="Analyzing clouds"):
    with rasterio.open(S2_FILE_PATHS[date]) as src:
        scl = src.read(config.SCL_BAND_INDEX + 1)
        cloud_stats[date] = analyze_cloud_coverage(scl, config.CLOUD_SCL_CLASSES)
    del scl
    gc.collect()

print(f"\n📊 Cloud Coverage:")
print(f"{'Date':<15} {'Cloud %':>10} {'Pixels':>15} {'Status':<15}")
print("-" * 60)
for date in SORTED_DATES:
    s = cloud_stats[date]
    status = "🔴 High" if s['cloud_percentage'] > 30 else \
             "🟡 Medium" if s['cloud_percentage'] > 10 else "🟢 Low"
    role = " ← INFERENCE" if date == TARGET_DATE else ""
    print(f"{date:<15} {s['cloud_percentage']:>9.2f}% {s['cloud_pixels']:>15,} "
          f"{status}{role}")


# ============================================================================
# Image Loader
# ============================================================================
class ImageLoader:
    def __init__(self, s2_paths: dict, s1_paths: dict):
        self.s2_paths = s2_paths
        self.s1_paths = s1_paths
        self._metadata = None

    def get_metadata(self) -> dict:
        if self._metadata is None:
            first_path = list(self.s2_paths.values())[0]
            with rasterio.open(first_path) as src:
                self._metadata = {
                    'height': src.height, 'width': src.width,
                    'count': src.count, 'dtype': src.dtypes[0],
                    'crs': src.crs, 'transform': src.transform,
                    'profile': src.profile.copy()}
        return self._metadata

    def load_s2_full(self, date: str) -> np.ndarray:
        with rasterio.open(self.s2_paths[date]) as src:
            return src.read()

    def load_s1_full(self, date: str) -> np.ndarray:
        if date not in self.s1_paths:
            print(f"   ⚠️  No S1 for {date}, returning zeros")
            meta = self.get_metadata()
            return np.zeros((2, meta['height'], meta['width']), dtype=np.float32)
        with rasterio.open(self.s1_paths[date]['s1_file']) as src:
            return src.read()

    def load_s2_band(self, date: str, band_idx: int) -> np.ndarray:
        with rasterio.open(self.s2_paths[date]) as src:
            return src.read(band_idx + 1)

    def get_chunk_windows(self, chunk_size: int = 512) -> List[Window]:
        meta = self.get_metadata()
        h, w = meta['height'], meta['width']
        windows = []
        for r in range(0, h, chunk_size):
            for c in range(0, w, chunk_size):
                windows.append(Window(c, r, min(chunk_size, w - c),
                                      min(chunk_size, h - r)))
        return windows

loader = ImageLoader(S2_FILE_PATHS, S1_FILE_PATHS)
metadata = loader.get_metadata()

print(f"\n✅ Loader: {metadata['height']}x{metadata['width']}, "
      f"{metadata['count']} bands, {metadata['dtype']}")
chunks = loader.get_chunk_windows(config.CHUNK_SIZE)
print(f"   Chunks: {len(chunks)}")


# ============================================================================
# Temporal Rate of Change
# ============================================================================
def calculate_temporal_rate_of_change_fast(
    target_data, previous_data, target_cloud_mask, previous_cloud_masks,
    days_from_target, spectral_indices, max_rate=500.0
) -> Tuple[np.ndarray, np.ndarray]:

    n_bands, height, width = target_data.shape
    n_prev = previous_data.shape[0]

    filled_data = target_data.astype(np.float32).copy()
    fill_success = ~target_cloud_mask.copy()

    # Handle 0 previous images
    if n_prev == 0:
        print("   ⚠️  No previous images for temporal interpolation")
        return filled_data, fill_success

    clear_indicators = (~previous_cloud_masks).astype(np.float32)
    inv_days = 1.0 / np.maximum(days_from_target, 1.0)
    temporal_weights = clear_indicators * inv_days[:, np.newaxis, np.newaxis]
    weight_sum = np.sum(temporal_weights, axis=0)
    n_clear = np.sum(clear_indicators, axis=0)
    has_clear = n_clear >= 1
    cloudy_with_clear = target_cloud_mask & has_clear

    print(f"   Cloudy pixels with clear reference: {np.sum(cloudy_with_clear):,}")

    if GPU_AVAILABLE:
        print("   Using GPU...")
        days_gpu = cp.asarray(days_from_target)
        tw_gpu = cp.asarray(temporal_weights)
        ws_gpu = cp.asarray(weight_sum)
        nc_gpu = cp.asarray(n_clear)
        cm_gpu = cp.asarray(target_cloud_mask)

        for bi in tqdm(spectral_indices, desc="Bands (GPU)"):
            pbv = cp.asarray(previous_data[:, bi, :, :].astype(np.float32))

            single = (nc_gpu == 1) & cm_gpu
            if cp.any(single):
                ws_val = cp.sum(pbv * tw_gpu, axis=0)
                ws_d = cp.maximum(ws_gpu, 1e-10)
                fb = cp.asarray(filled_data[bi])
                fb[single] = (ws_val / ws_d)[single]
                filled_data[bi] = cp.asnumpy(fb)
                fill_success[cp.asnumpy(single)] = True
                del fb, ws_val, ws_d

            multi = (nc_gpu >= 2) & cm_gpu
            if cp.any(multi):
                mr_gpu, mc_gpu_idx = cp.where(multi)
                mr = cp.asnumpy(mr_gpu)
                mc = cp.asnumpy(mc_gpu_idx)

                csz = 100000
                for cs in range(0, len(mr), csz):
                    ce = min(cs + csz, len(mr))
                    cr, cc = mr[cs:ce], mc[cs:ce]

                    cv = pbv[:, cr, cc]
                    ctw = tw_gpu[:, cr, cc]
                    x = days_gpu[:, cp.newaxis]
                    w = ctw

                    sw = cp.sum(w, axis=0)
                    swx = cp.sum(w * x, axis=0)
                    swy = cp.sum(w * cv, axis=0)
                    swxx = cp.sum(w * x * x, axis=0)
                    swxy = cp.sum(w * x * cv, axis=0)

                    den = sw * swxx - swx * swx
                    valid = cp.abs(den) > 1e-10

                    slope = cp.zeros(len(cr), dtype=cp.float32)
                    intercept = cp.zeros(len(cr), dtype=cp.float32)

                    slope[valid] = (sw[valid] * swxy[valid] -
                                    swx[valid] * swy[valid]) / den[valid]
                    slope = cp.clip(slope, -max_rate, max_rate)
                    intercept[valid] = (swy[valid] -
                                        slope[valid] * swx[valid]) / sw[valid]
                    intercept[~valid] = swy[~valid] / cp.maximum(
                        sw[~valid], 1e-10)

                    filled_data[bi, cr, cc] = cp.asnumpy(
                        cp.maximum(intercept, 0))

                    del cv, ctw, w, x, sw, swx, swy, swxx, swxy
                    del den, valid, slope, intercept

                fill_success[cp.asnumpy(multi)] = True
                del mr_gpu, mc_gpu_idx

            del pbv
            cp.get_default_memory_pool().free_all_blocks()

        del days_gpu, tw_gpu, ws_gpu, nc_gpu, cm_gpu
        cp.get_default_memory_pool().free_all_blocks()

    else:
        for bi in tqdm(spectral_indices, desc="Bands (CPU)"):
            pbv = previous_data[:, bi, :, :].astype(np.float32)

            single = (n_clear == 1) & target_cloud_mask
            if np.any(single):
                ws_val = np.sum(pbv * temporal_weights, axis=0)
                ws_d = np.maximum(weight_sum, 1e-10)
                filled_data[bi, single] = (ws_val / ws_d)[single]
                fill_success[single] = True
                del ws_val, ws_d

            multi = (n_clear >= 2) & target_cloud_mask
            if np.any(multi):
                mr, mc = np.where(multi)
                csz = 100000
                for cs in range(0, len(mr), csz):
                    ce = min(cs + csz, len(mr))
                    cr, cc = mr[cs:ce], mc[cs:ce]

                    cv = pbv[:, cr, cc]
                    ctw = temporal_weights[:, cr, cc]
                    x = days_from_target[:, np.newaxis]
                    w = ctw

                    sw = np.sum(w, axis=0)
                    swx = np.sum(w * x, axis=0)
                    swy = np.sum(w * cv, axis=0)
                    swxx = np.sum(w * x * x, axis=0)
                    swxy = np.sum(w * x * cv, axis=0)

                    den = sw * swxx - swx * swx
                    valid = np.abs(den) > 1e-10

                    slope = np.zeros(len(cr), dtype=np.float32)
                    intercept = np.zeros(len(cr), dtype=np.float32)

                    slope[valid] = (sw[valid] * swxy[valid] -
                                    swx[valid] * swy[valid]) / den[valid]
                    slope = np.clip(slope, -max_rate, max_rate)
                    intercept[valid] = (swy[valid] -
                                        slope[valid] * swx[valid]) / sw[valid]
                    intercept[~valid] = swy[~valid] / np.maximum(
                        sw[~valid], 1e-10)

                    filled_data[bi, cr, cc] = np.maximum(intercept, 0)

                    del cv, ctw, w, x, sw, swx, swy, swxx, swxy

                fill_success[multi] = True

            del pbv
            gc.collect()

    return filled_data, fill_success

print("✅ Temporal functions ready")


# ============================================================================
# S1-S2 Fusion Model
# ============================================================================
class S1S2FusionModel:
    def __init__(self, n_estimators=100, sample_fraction=0.5,
                 max_samples=200_000):
        self.n_estimators = n_estimators
        self.sample_fraction = sample_fraction
        self.max_samples = max_samples
        self.models = {}
        self.is_trained = False
        self.band_stats = {}

    def _get_s1_valid_mask(self, s1_data):
        vv, vh = s1_data[0], s1_data[1]
        return (np.isfinite(vv) & np.isfinite(vh) &
                (vv != 0) & (vh != 0) &
                (np.abs(vv) < 1e6) & (np.abs(vh) < 1e6))

    def _prepare_features(self, s1_data, rows, cols):
        n = len(rows)
        vv_f = s1_data[0].astype(np.float32)
        vh_f = s1_data[1].astype(np.float32)
        vv, vh = vv_f[rows, cols], vh_f[rows, cols]
        vv_s = np.maximum(np.abs(vv), 1e-10)
        vh_s = np.maximum(np.abs(vh), 1e-10)

        vvm3 = ndimage.uniform_filter(vv_f, size=3)
        vhm3 = ndimage.uniform_filter(vh_f, size=3)
        vvm7 = ndimage.uniform_filter(vv_f, size=7)
        vhm7 = ndimage.uniform_filter(vh_f, size=7)
        vv_var = np.maximum(
            ndimage.uniform_filter(vv_f**2, 5) -
            ndimage.uniform_filter(vv_f, 5)**2, 0)
        vh_var = np.maximum(
            ndimage.uniform_filter(vh_f**2, 5) -
            ndimage.uniform_filter(vh_f, 5)**2, 0)

        X = np.zeros((n, 14), dtype=np.float32)
        X[:, 0] = vv;  X[:, 1] = vh
        X[:, 2] = vv / vh_s;  X[:, 3] = vh / vv_s
        X[:, 4] = vv * vh
        X[:, 5] = np.log10(vv_s);  X[:, 6] = np.log10(vh_s)
        X[:, 7] = vvm3[rows, cols]; X[:, 8] = vhm3[rows, cols]
        X[:, 9] = vvm7[rows, cols]; X[:, 10] = vhm7[rows, cols]
        X[:, 11] = np.sqrt(vv_var[rows, cols])
        X[:, 12] = np.sqrt(vh_var[rows, cols])
        X[:, 13] = (vv - vh) / (vv_s + vh_s)

        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        del vv_f, vh_f, vvm3, vhm3, vvm7, vhm7, vv_var, vh_var
        gc.collect()
        return X

    def train_multi_date(self, s2_list, s1_list, cm_list, spectral_idx):
        if not s2_list:
            print("   ⚠️  No training data for S1-S2 fusion")
            self.is_trained = False
            return

        all_rows, all_cols = [], []
        all_s1_feat = []
        all_s2_vals = {bi: [] for bi in spectral_idx}
        date_idx = []

        for i, (s2, s1, cm) in enumerate(zip(s2_list, s1_list, cm_list)):
            s1v = self._get_s1_valid_mask(s1)
            trainable = (~cm) & s1v
            cr, cc = np.where(trainable)
            na = len(cr)

            if na < 500:
                print(f"   Date {i}: skip ({na} trainable)")
                continue

            budget = self.max_samples // max(len(s2_list), 1)
            nt = min(int(na * self.sample_fraction), budget)
            nt = max(nt, 500)

            idx = np.random.choice(na, nt, replace=False)
            sr, sc = cr[idx], cc[idx]

            all_rows.append(sr)
            all_cols.append(sc)
            all_s1_feat.append((s1, sr, sc))
            date_idx.append(i)

            for bi in spectral_idx:
                all_s2_vals[bi].append(s2[bi, sr, sc].astype(np.float32))

            print(f"   Date {i}: {nt:,} / {na:,}")

        if not all_rows:
            print("   ⚠️  No trainable pairs")
            self.is_trained = False
            return

        X_parts = [self._prepare_features(s1, r, c)
                    for s1, r, c in all_s1_feat]
        X = np.vstack(X_parts)
        print(f"   Total samples: {X.shape[0]:,}")
        del X_parts, all_s1_feat
        gc.collect()

        for bi in tqdm(spectral_idx, desc="Training fusion"):
            y = np.nan_to_num(np.concatenate(all_s2_vals[bi]), nan=0.0)
            self.band_stats[bi] = {
                'mean': float(np.mean(y)), 'std': float(np.std(y)),
                'p1': float(np.percentile(y, 1)),
                'p99': float(np.percentile(y, 99))}

            model = RandomForestRegressor(
                n_estimators=self.n_estimators, max_depth=15,
                min_samples_leaf=5, max_features='sqrt',
                n_jobs=-1, random_state=42)
            model.fit(X, y)
            self.models[bi] = model

        self.is_trained = True
        print(f"   ✅ Trained on {len(date_idx)} dates, {X.shape[0]:,} samples")

    def predict(self, s1_data, mask, spectral_idx):
        if not self.is_trained:
            return {}

        s1v = self._get_s1_valid_mask(s1_data)
        predictable = mask & s1v
        pr, pc = np.where(predictable)
        n = len(pr)

        if n == 0:
            print(f"   ⚠️  No valid S1 in mask")
            return {}

        print(f"   Pixels to predict: {n:,}")
        predictions = {}
        csz = 500_000

        for bi in tqdm(spectral_idx, desc="Predicting", leave=False):
            if bi not in self.models:
                continue
            all_p = np.zeros(n, dtype=np.float32)
            for cs in range(0, n, csz):
                ce = min(cs + csz, n)
                X_c = self._prepare_features(s1_data, pr[cs:ce], pc[cs:ce])
                p_c = self.models[bi].predict(X_c)
                p1 = self.band_stats[bi]['p1']
                p99 = self.band_stats[bi]['p99']
                margin = (p99 - p1) * 0.1
                all_p[cs:ce] = np.clip(
                    p_c, max(0, p1 - margin), p99 + margin).astype(np.float32)
                del X_c, p_c
                gc.collect()
            predictions[bi] = (pr.copy(), pc.copy(), all_p)

        return predictions

print("✅ S1-S2 Fusion ready")


# ============================================================================
# Spatial Interpolation
# ============================================================================
def spatial_interpolate_band_fast(data, mask, max_distance=50):
    result = data.copy().astype(np.float32)
    success = np.zeros_like(mask, dtype=bool)
    if not np.any(mask):
        success[:] = True
        return result, success
    if not np.any(~mask):
        return result, success
    dist, indices = ndimage.distance_transform_edt(
        mask, return_distances=True, return_indices=True)
    fillable = mask & (dist <= max_distance)
    if np.any(fillable):
        result[fillable] = data[indices[0][fillable], indices[1][fillable]]
        success[fillable] = True
    success[~mask] = True
    return result, success

def apply_spatial_interpolation(data, mask, spectral_idx, max_distance=50):
    result = data.copy()
    overall = np.zeros(mask.shape, dtype=bool)
    for bi in tqdm(spectral_idx, desc="Spatial interpolation"):
        br, bs = spatial_interpolate_band_fast(data[bi], mask, max_distance)
        result[bi] = br
        overall |= bs
    return result, overall

print("✅ Spatial interpolation ready")


# ============================================================================
# Edge Blending
# ============================================================================
def create_blend_weights(cloud_mask, cloud_mask_buffered, buffer_size=5):
    if buffer_size <= 0:
        return cloud_mask.astype(np.float32)
    weights = np.zeros_like(cloud_mask, dtype=np.float32)
    weights[cloud_mask] = 1.0
    bz = cloud_mask_buffered & ~cloud_mask
    if np.any(bz):
        d = ndimage.distance_transform_edt(~cloud_mask)
        bw = np.maximum(1.0 - d / buffer_size, 0.0)
        weights[bz] = bw[bz]
        del d, bw
    return weights

def blend_images(original, filled, weights, spectral_idx):
    result = original.copy().astype(np.float32)
    if GPU_AVAILABLE:
        w_gpu = cp.asarray(weights)
        ow = 1.0 - w_gpu
        for bi in spectral_idx:
            o = cp.asarray(original[bi].astype(np.float32))
            f = cp.asarray(filled[bi].astype(np.float32))
            result[bi] = cp.asnumpy(o * ow + f * w_gpu)
            del o, f
        del w_gpu, ow
        cp.get_default_memory_pool().free_all_blocks()
    else:
        bm = weights > 0
        if np.any(bm):
            for bi in spectral_idx:
                result[bi][bm] = (
                    original[bi].astype(np.float32)[bm] * (1 - weights[bm]) +
                    filled[bi].astype(np.float32)[bm] * weights[bm])
    return result

print("✅ Blending ready")


# ============================================================================
# Vegetation Indices (NDRE and NDWI only)
# ============================================================================
def generate_vegetation_indices(cloud_free_path, output_dir, target_date, band_names):
    print(f"\n{'='*70}")
    print(f"🌿 GENERATING VEGETATION INDICES (NDRE, NDWI)")
    print(f"{'='*70}")

    CLEAN = Path(cloud_free_path)
    OUT = Path(output_dir)
    bu = [b.upper() for b in band_names]

    def find_band(name):
        try:
            return bu.index(name.upper())
        except ValueError:
            return None

    B05 = find_band('B05')
    B08 = find_band('B08')
    B11 = find_band('B11')

    avail = []
    if B05 is not None and B08 is not None:
        avail.append('NDRE')
    if B08 is not None and B11 is not None:
        avail.append('NDWI')

    print(f"   Available: {avail}")

    if not CLEAN.exists():
        print(f"   ❌ File not found: {CLEAN}")
        return

    dc = target_date.replace('-', '')

    with rasterio.open(CLEAN) as src:
        profile = src.profile.copy()
        height = src.height
        width = src.width
        dtype = src.dtypes[0]
        nir = src.read(B08 + 1).astype(np.float32) if B08 is not None else None
        re1 = src.read(B05 + 1).astype(np.float32) if B05 is not None else None
        swir = src.read(B11 + 1).astype(np.float32) if B11 is not None else None

    print(f"   Image size: {width} x {height}")
    print(f"   Bands: {src.count if False else 'read'}")
    print(f"   Dtype: {dtype}")

    ip = profile.copy()
    ip.update(dtype='float32', count=1, nodata=None, compress='lzw')

    # ---- Compute NDRE = (NIR - Red Edge 1) / (NIR + Red Edge 1) ----
    if 'NDRE' in avail:
        print(f"\n🔬 Computing NDRE...")

        ndre = (nir - re1) / (nir + re1 + 1e-6)
        ndre = np.clip(ndre, -1.0, 1.0).astype(np.float32)

        print(f"   NDRE range: [{ndre.min():.4f}, {ndre.max():.4f}]")
        print(f"   NDRE mean:  {ndre.mean():.4f}")

        # Save NDRE
        ndre_output_path = OUT / f"ndre_PROD_{dc}.tif"

        print(f"\n💾 Saving NDRE...")

        ndre_profile = profile.copy()
        ndre_profile.update(
            dtype='float32',
            count=1,
            nodata=None,
            compress='lzw'
        )

        with rasterio.open(ndre_output_path, 'w', **ndre_profile) as dst:
            dst.write(ndre, 1)
            dst.set_band_description(1, "NDRE = (B08_NIR - B05_RedEdge1) / (B08_NIR + B05_RedEdge1)")

        ndre_size_mb = ndre_output_path.stat().st_size / (1024 * 1024)
        print(f"   ✅ Saved: {ndre_output_path}")
        print(f"   File size: {ndre_size_mb:.2f} MB")

        del ndre
        gc.collect()

    # ---- Compute NDWI (Gao) = (NIR - SWIR1) / (NIR + SWIR1) ----
    if 'NDWI' in avail:
        print(f"\n🔬 Computing NDWI...")

        ndwi = (nir - swir) / (nir + swir + 1e-6)
        ndwi = np.clip(ndwi, -1.0, 1.0).astype(np.float32)

        print(f"   NDWI range: [{ndwi.min():.4f}, {ndwi.max():.4f}]")
        print(f"   NDWI mean:  {ndwi.mean():.4f}")

        # Save NDWI
        ndwi_output_path = OUT / f"ndwi_PROD_{dc}.tif"

        print(f"\n💾 Saving NDWI...")

        ndwi_profile = profile.copy()
        ndwi_profile.update(
            dtype='float32',
            count=1,
            nodata=None,
            compress='lzw'
        )

        with rasterio.open(ndwi_output_path, 'w', **ndwi_profile) as dst:
            dst.write(ndwi, 1)
            dst.set_band_description(1, "NDWI = (B08_NIR - B11_SWIR1) / (B08_NIR + B11_SWIR1)")

        ndwi_size_mb = ndwi_output_path.stat().st_size / (1024 * 1024)
        print(f"   ✅ Saved: {ndwi_output_path}")
        print(f"   File size: {ndwi_size_mb:.2f} MB")

        del ndwi
        gc.collect()

    # Summary
    print(f"\n{'='*70}")
    print(f"📊 VEGETATION INDICES SUMMARY")
    print(f"{'='*70}")
    print(f"  Input:  {CLEAN.name}")
    print(f"  Date:   {target_date}")
    print(f"")
    print(f"  Output files:")
    if 'NDRE' in avail:
        print(f"    NDRE: {OUT / f'ndre_PROD_{dc}.tif'}")
        print(f"           Size:  {(OUT / f'ndre_PROD_{dc}.tif').stat().st_size / (1024*1024):.2f} MB")
        print(f"")
    if 'NDWI' in avail:
        print(f"    NDWI: {OUT / f'ndwi_PROD_{dc}.tif'}")
        print(f"           Size:  {(OUT / f'ndwi_PROD_{dc}.tif').stat().st_size / (1024*1024):.2f} MB")
    print(f"{'='*70}")
    print(f"✅ Vegetation indices generation complete!")

    # Free memory
    del nir, re1, swir
    gc.collect()

print("✅ Vegetation indices generation function ready (NDRE, NDWI)")


# ============================================================================
# Main Pipeline
# ============================================================================
class CloudRemovalPipeline:
    def __init__(self, config, loader, s2_paths, s1_paths, band_names):
        self.config = config
        self.loader = loader
        self.s2_paths = s2_paths
        self.s1_paths = s1_paths
        self.band_names = band_names
        self.fusion_model = S1S2FusionModel(
            n_estimators=100,
            sample_fraction=config.TRAINING_SAMPLE_FRACTION,
            max_samples=config.TRAINING_MAX_SAMPLES)
        self.stats = {}
        self.confidence = None  # Will be set after processing

    def process_date(self, target_date, output_path=None):
        print(f"\n{'='*70}")
        print(f"🚀 PROCESSING: {target_date}")
        print(f"{'='*70}")

        # Initialize confidence tracker
        conf = PipelineConfidence()
        conf.inference_date = target_date
        conf.inference_is_broken = INFERENCE_IS_BROKEN if 'INFERENCE_IS_BROKEN' in dir() else False

        # Step 1: Load target
        print("\n📥 Step 1: Loading target image...")
        target_data = self.loader.load_s2_full(target_date)
        target_s1 = self.loader.load_s1_full(target_date)
        original_dtype = target_data.dtype
        print(f"   Shape: {target_data.shape}, dtype: {original_dtype}")

        # Step 2: Cloud mask
        print("\n☁️ Step 2: Creating cloud mask...")
        scl_band = target_data[self.config.SCL_BAND_INDEX]
        cloud_mask, cloud_mask_buffered = create_cloud_mask(
            scl_band, buffer_size=self.config.BUFFER_PIXELS,
            cloud_classes=self.config.CLOUD_SCL_CLASSES)

        initial_cloud = int(np.sum(cloud_mask))
        total_px = cloud_mask.size
        cloud_pct = (initial_cloud / total_px) * 100
        buffered_px = int(np.sum(cloud_mask_buffered))

        print(f"   • Cloud: {initial_cloud:,} ({cloud_pct:.2f}%)")
        print(f"   • Buffered: {buffered_px:,}")

        # Populate confidence input metrics
        conf.total_pixels = total_px
        conf.cloud_pixels = initial_cloud
        conf.clear_pixels = total_px - initial_cloud
        conf.inference_cloud_cover_pct = cloud_pct

        # Get nodata from Cell 1 if available
        try:
            conf.inference_nodata_pct = NODATA_PCT_S2_PER_DATE.get(
                target_date, 0.0) or 0.0
        except:
            conf.inference_nodata_pct = 0.0

        # No clouds case
        if initial_cloud == 0:
            print("✅ No clouds! Returning original.")
            if output_path:
                self._save_result(target_data, target_date, output_path)

            conf.temporal_filled = 0
            conf.fusion_filled = 0
            conf.spatial_filled = 0
            conf.unfilled_pixels = 0

            # Count previous images (even though not used)
            prev_dates = get_previous_dates(
                target_date, list(self.s2_paths.keys()),
                self.config.MAX_PREVIOUS_IMAGES)
            conf.num_previous_images = len(prev_dates)
            conf.previous_image_dates = prev_dates
            conf.previous_image_cloud_pcts = [
                cloud_stats.get(d, {}).get('cloud_percentage', 0)
                for d in prev_dates]
            conf.previous_image_days_gap = [
                days_between(target_date, d) for d in prev_dates]

            conf.calculate()
            self.confidence = conf
            self.stats[target_date] = {
                'initial_cloud_pixels': 0,
                'initial_cloud_percentage': 0,
                'temporal_filled': 0, 'fusion_filled': 0,
                'spatial_filled': 0, 'unfilled': 0,
                'success_rate': 100.0}
            return target_data

        # Step 3: Load previous
        print("\n📥 Step 3: Loading previous images...")
        previous_dates = get_previous_dates(
            target_date, list(self.s2_paths.keys()),
            self.config.MAX_PREVIOUS_IMAGES)

        if not previous_dates:
            print("   ⚠️  No previous dates! Using all other dates...")
            all_other = [d for d in self.s2_paths.keys() if d != target_date]
            previous_dates = sorted(all_other, reverse=True)[
                :self.config.MAX_PREVIOUS_IMAGES]

        # Populate confidence with previous image info
        conf.num_previous_images = len(previous_dates)
        conf.previous_image_dates = list(previous_dates)
        conf.previous_image_cloud_pcts = [
            cloud_stats.get(d, {}).get('cloud_percentage', 0)
            for d in previous_dates]
        conf.previous_image_days_gap = [
            days_between(target_date, d) for d in previous_dates]

        print(f"   • Previous dates ({len(previous_dates)}): {previous_dates}")
        if not previous_dates:
            print("   ⚠️  ZERO previous images available!")

        prev_data_list, prev_masks_list = [], []
        prev_s1_list, prev_s2_fusion, prev_cm_fusion = [], [], []
        days_list = []

        for pd in tqdm(previous_dates, desc="Loading previous"):
            pi = self.loader.load_s2_full(pd)
            ps1 = self.loader.load_s1_full(pd)
            pscl = pi[self.config.SCL_BAND_INDEX]
            pcm, _ = create_cloud_mask(
                pscl, buffer_size=0,
                cloud_classes=self.config.CLOUD_SCL_CLASSES)

            prev_data_list.append(pi)
            prev_masks_list.append(pcm)
            prev_s1_list.append(ps1)
            prev_s2_fusion.append(pi)
            prev_cm_fusion.append(pcm)
            days_list.append(days_between(target_date, pd))

            del pscl
            gc.collect()

        # Safe array creation for 0 previous images
        if prev_data_list:
            prev_arr = np.array(prev_data_list)
            prev_masks_arr = np.array(prev_masks_list)
        else:
            # Create empty arrays with correct shape
            n_bands = target_data.shape[0]
            h, w = target_data.shape[1], target_data.shape[2]
            prev_arr = np.empty((0, n_bands, h, w), dtype=target_data.dtype)
            prev_masks_arr = np.empty((0, h, w), dtype=bool)

        days_arr = np.array(days_list, dtype=np.float32) if days_list else np.array([], dtype=np.float32)

        del prev_data_list, prev_masks_list
        gc.collect()

        print(f"   • Days from target: {days_list}")

        # Step 4: Temporal fill
        print("\n⏱️ Step 4: Temporal rate-of-change interpolation...")
        spectral_idx = list(self.config.SPECTRAL_BAND_INDICES)

        filled_data, fill_success = calculate_temporal_rate_of_change_fast(
            target_data.astype(np.float32),
            prev_arr.astype(np.float32),
            cloud_mask, prev_masks_arr,
            days_arr, spectral_idx,
            self.config.MAX_CHANGE_RATE_PER_DAY)

        temporal_filled = int(np.sum(cloud_mask & fill_success))
        remaining_mask = cloud_mask & ~fill_success
        remaining = int(np.sum(remaining_mask))

        print(f"   ✅ Temporal: {temporal_filled:,}")
        print(f"   • Remaining: {remaining:,}")

        del prev_arr, prev_masks_arr
        gc.collect()
        if GPU_AVAILABLE:
            cp.get_default_memory_pool().free_all_blocks()

        # Step 5: S1-S2 Fusion
        fusion_filled = 0
        if remaining > 0 and prev_s2_fusion:
            print("\n🛰️ Step 5: S1-S2 Fusion...")

            self.fusion_model.train_multi_date(
                prev_s2_fusion, prev_s1_list,
                prev_cm_fusion, spectral_idx)

            if self.fusion_model.is_trained:
                predictions = self.fusion_model.predict(
                    target_s1, remaining_mask, spectral_idx)

                if predictions:
                    for bi, (rows, cols, vals) in predictions.items():
                        filled_data[bi, rows, cols] = vals

                    first_bi = list(predictions.keys())[0]
                    pr, pc, _ = predictions[first_bi]
                    fusion_filled = len(pr)
                    fill_success[pr, pc] = True
                    remaining_mask = cloud_mask & ~fill_success
                    remaining = int(np.sum(remaining_mask))
                    print(f"   ✅ Fusion: {fusion_filled:,}")

                del predictions
                gc.collect()
        elif remaining > 0:
            print("\n🛰️ Step 5: S1-S2 Fusion → SKIPPED (no training data)")

        del prev_s1_list, prev_s2_fusion, prev_cm_fusion, target_s1
        gc.collect()

        # Step 6: Spatial interpolation
        spatial_filled = 0
        if remaining > 0:
            print(f"\n🔲 Step 6: Spatial interpolation ({remaining:,} remaining)...")

            filled_spatial, spatial_success = apply_spatial_interpolation(
                filled_data, remaining_mask, spectral_idx,
                self.config.MAX_INTERPOLATION_DISTANCE)
            filled_data = filled_spatial
            spatial_filled = int(np.sum(remaining_mask & spatial_success))
            fill_success |= spatial_success

            del filled_spatial, spatial_success
            gc.collect()
            print(f"   ✅ Spatial: {spatial_filled:,}")

        # Step 7: Blend
        print("\n🎨 Step 7: Edge blending...")
        blend_w = create_blend_weights(
            cloud_mask, cloud_mask_buffered, self.config.BUFFER_PIXELS)

        result = blend_images(
            target_data, filled_data, blend_w, spectral_idx)

        del filled_data, blend_w
        gc.collect()

        # Ensure clear pixels untouched
        clear_mask = ~cloud_mask_buffered
        for bi in spectral_idx:
            result[bi][clear_mask] = target_data[bi][clear_mask].astype(np.float32)

        # Clip & cast
        if np.issubdtype(original_dtype, np.integer):
            max_val = np.iinfo(original_dtype).max
        else:
            max_val = np.finfo(original_dtype).max
        result = np.clip(result, 0, max_val).astype(original_dtype)

        # Keep aux bands
        for ai in self.config.AUX_BAND_INDICES:
            result[ai] = target_data[ai]

        # Save
        if output_path:
            print(f"\n💾 Step 8: Saving...")
            self._save_result(result, target_date, output_path)

        # Verify
        sample_bi = spectral_idx[0]
        clear_diff = np.sum(
            result[sample_bi][clear_mask].astype(np.float64) -
            target_data[sample_bi][clear_mask].astype(np.float64))
        print(f"   🔍 Clear pixel check: diff = {clear_diff}")

        # Stats
        final_unfilled = int(np.sum(cloud_mask & ~fill_success))
        self.stats[target_date] = {
            'initial_cloud_pixels': initial_cloud,
            'initial_cloud_percentage': cloud_pct,
            'temporal_filled': temporal_filled,
            'fusion_filled': fusion_filled,
            'spatial_filled': spatial_filled,
            'unfilled': final_unfilled,
            'success_rate': (
                ((initial_cloud - final_unfilled) / initial_cloud * 100)
                if initial_cloud > 0 else 100)}

        # Populate confidence fill metrics
        conf.temporal_filled = temporal_filled
        conf.fusion_filled = fusion_filled
        conf.spatial_filled = spatial_filled
        conf.unfilled_pixels = final_unfilled

        # Calculate confidence
        conf.calculate()
        self.confidence = conf

        print(f"\n{'='*70}")
        print(f"📊 PIPELINE SUMMARY:")
        print(f"   • Initial clouds: {initial_cloud:,} ({cloud_pct:.2f}%)")
        print(f"   • Temporal fill:  {temporal_filled:,}")
        print(f"   • S1-S2 fusion:   {fusion_filled:,}")
        print(f"   • Spatial fill:   {spatial_filled:,}")
        print(f"   • Unfilled:       {final_unfilled:,}")
        print(f"   • Success rate:   {self.stats[target_date]['success_rate']:.2f}%")
        print(f"{'='*70}")

        del target_data, cloud_mask, cloud_mask_buffered, fill_success, clear_mask
        gc.collect()
        if GPU_AVAILABLE:
            cp.get_default_memory_pool().free_all_blocks()

        return result

    def _save_result(self, data, date, output_path):
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with rasterio.open(self.s2_paths[date]) as src:
            profile = src.profile.copy()
        profile.update(dtype=data.dtype)
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(data)
        print(f"   ✅ Saved: {output_path}")


# ============================================================================
# INITIALIZE AND RUN
# ============================================================================
pipeline = CloudRemovalPipeline(
    config, loader, S2_FILE_PATHS, S1_FILE_PATHS, BAND_NAMES)

print(f"\n✅ Pipeline initialized!")
print(f"\n🎯 Target: {TARGET_DATE}")
print(f"📁 Output: {OUTPUT_PATH}")
print(f"☁️ Cloud: {cloud_stats[TARGET_DATE]['cloud_percentage']:.2f}%")

try:
    if 'CLOUD_PCT_PER_DATE' in dir():
        cl1 = CLOUD_PCT_PER_DATE.get(TARGET_DATE)
        nd1 = NODATA_PCT_S2_PER_DATE.get(TARGET_DATE)
        print(f"📊 Cell 1: Cloud={cl1}%, NoData={nd1}%")
    if 'INFERENCE_IS_BROKEN' in dir() and INFERENCE_IS_BROKEN:
        print(f"   ⚠️  BROKEN INFERENCE IMAGE")
except:
    pass

# RUN PIPELINE
result = pipeline.process_date(TARGET_DATE, OUTPUT_PATH)

print(f"\n✅ Processing complete!")
print(f"   Result shape: {result.shape}")
print(f"   Result dtype: {result.dtype}")

# ============================================================================
# Generate Vegetation Indices (NDRE and NDWI only)
# ============================================================================
generate_vegetation_indices(OUTPUT_PATH, OUTPUT_DIR, TARGET_DATE, BAND_NAMES)

# ============================================================================
# PRINT CONFIDENCE REPORT
# ============================================================================
if pipeline.confidence:
    pipeline.confidence.print_report()

# ============================================================================
# STORE CONFIDENCE VARIABLES
# ============================================================================
PIPELINE_CONFIDENCE = pipeline.confidence
PIPELINE_STATS = pipeline.stats

# Convenience variables
CONFIDENCE_OVERALL = pipeline.confidence.confidence_overall if pipeline.confidence else 0.0
CONFIDENCE_LEVEL = pipeline.confidence.confidence_level if pipeline.confidence else "UNKNOWN"
CONFIDENCE_PREV_IMAGES = pipeline.confidence.confidence_prev_images if pipeline.confidence else 0.0
CONFIDENCE_CLOUD_COVER = pipeline.confidence.confidence_cloud_cover if pipeline.confidence else 0.0
CONFIDENCE_NODATA = pipeline.confidence.confidence_nodata if pipeline.confidence else 0.0
CONFIDENCE_FILL_QUALITY = pipeline.confidence.confidence_fill_quality if pipeline.confidence else 0.0

NUM_PREVIOUS_IMAGES = pipeline.confidence.num_previous_images if pipeline.confidence else 0
INFERENCE_CLOUD_PCT = pipeline.confidence.inference_cloud_cover_pct if pipeline.confidence else 0.0
INFERENCE_NODATA_PCT = pipeline.confidence.inference_nodata_pct if pipeline.confidence else 0.0

FILL_TEMPORAL_PCT = (
    (pipeline.confidence.temporal_filled / max(pipeline.confidence.cloud_pixels, 1) * 100)
    if pipeline.confidence and pipeline.confidence.cloud_pixels > 0 else 0.0
)
FILL_FUSION_PCT = (
    (pipeline.confidence.fusion_filled / max(pipeline.confidence.cloud_pixels, 1) * 100)
    if pipeline.confidence and pipeline.confidence.cloud_pixels > 0 else 0.0
)
FILL_SPATIAL_PCT = (
    (pipeline.confidence.spatial_filled / max(pipeline.confidence.cloud_pixels, 1) * 100)
    if pipeline.confidence and pipeline.confidence.cloud_pixels > 0 else 0.0
)
FILL_UNFILLED_PCT = (
    (pipeline.confidence.unfilled_pixels / max(pipeline.confidence.cloud_pixels, 1) * 100)
    if pipeline.confidence and pipeline.confidence.cloud_pixels > 0 else 0.0
)

# ============================================================================
# FINAL OUTPUT SUMMARY
# ============================================================================
print(f"\n{'='*70}")
print(f"📦 FINAL OUTPUT")
print(f"{'='*70}")

output_dir_path = Path(OUTPUT_DIR)
print(f"\n📁 {output_dir_path}/")
for f in sorted(output_dir_path.glob("*.tif")):
    sz = f.stat().st_size / (1024 * 1024)
    print(f"   └── {f.name} ({sz:.1f} MB)")

print(f"\n📁 {PAIRS_DIR}/")
for folder in sorted(PAIRS_DIR.iterdir()):
    if folder.is_dir():
        print(f"   └── {folder.name}/")
        for f in sorted(folder.glob("*.tif")):
            sz = f.stat().st_size / (1024 * 1024)
            print(f"       └── {f.name} ({sz:.1f} MB)")

print(f"\n{'='*70}")
print(f"📦 CONFIDENCE VARIABLES")
print(f"{'='*70}")
print(f"""
  CONFIDENCE_OVERALL      : {CONFIDENCE_OVERALL:.1f}/100
  CONFIDENCE_LEVEL        : {CONFIDENCE_LEVEL}
  CONFIDENCE_PREV_IMAGES  : {CONFIDENCE_PREV_IMAGES:.1f}/100
  CONFIDENCE_CLOUD_COVER  : {CONFIDENCE_CLOUD_COVER:.1f}/100
  CONFIDENCE_NODATA       : {CONFIDENCE_NODATA:.1f}/100
  CONFIDENCE_FILL_QUALITY : {CONFIDENCE_FILL_QUALITY:.1f}/100

  NUM_PREVIOUS_IMAGES     : {NUM_PREVIOUS_IMAGES}
  INFERENCE_CLOUD_PCT     : {INFERENCE_CLOUD_PCT:.2f}%
  INFERENCE_NODATA_PCT    : {INFERENCE_NODATA_PCT:.2f}%

  FILL_TEMPORAL_PCT       : {FILL_TEMPORAL_PCT:.2f}%
  FILL_FUSION_PCT         : {FILL_FUSION_PCT:.2f}%
  FILL_SPATIAL_PCT        : {FILL_SPATIAL_PCT:.2f}%
  FILL_UNFILLED_PCT       : {FILL_UNFILLED_PCT:.2f}%

  PIPELINE_CONFIDENCE     : PipelineConfidence object (full details)
  PIPELINE_STATS          : {{date: stats_dict}}
""")

print("✅ ALL DONE")
print(f"{'='*70}")

ℹ️ CuPy not available. Using CPU only.
NumPy version: 2.4.3
GPU acceleration (CuPy): Disabled

🔗 CONNECTING TO DOWNLOAD CELL OUTPUTS
✅ Cell 1 variables found:
   INFERENCE_DATE  : 2026-03-26
   PREVIOUS_DATES  : ['2026-03-21', '2026-03-16', '2026-03-13', '2026-03-11', '2026-03-06']
   PAIRS_DIR       : emsa-auto/pairs
   DOWNLOAD_PAIRS  : 6 pairs

📂 BUILDING PATHS FROM PAIRS FOLDER

📊 Path Resolution:
   Date           Role         S2 File                                  S1 File
   ----------------------------------------------------------------------------------------------------
   2026-03-26     INFERENCE    openEO_2026-03-26Z.tif                   s1_20260325.tif
   2026-03-21     PREVIOUS     openEO_2026-03-21Z.tif                   s1_20260322.tif
   2026-03-16     PREVIOUS     openEO_2026-03-16Z.tif                   s1_20260315.tif
   2026-03-13     PREVIOUS     openEO_2026-03-13Z.tif                   s1_20260313.tif
   2026-03-11     PREVIOUS     openEO_2026-03-11Z.tif      

Analyzing clouds: 100%|██████████| 6/6 [00:05<00:00,  1.03it/s]



📊 Cloud Coverage:
Date               Cloud %          Pixels Status         
------------------------------------------------------------
2026-03-26          47.63%      43,776,972 🔴 High ← INFERENCE
2026-03-21           0.08%          72,076 🟢 Low
2026-03-16          16.52%      15,183,844 🟡 Medium
2026-03-13          16.11%      14,812,016 🟡 Medium
2026-03-11          14.93%      13,722,864 🟡 Medium
2026-03-06          60.69%      55,786,764 🔴 High

✅ Loader: 10000x9192, 15 bands, int32
   Chunks: 90
✅ Temporal functions ready
✅ S1-S2 Fusion ready
✅ Spatial interpolation ready
✅ Blending ready
✅ Vegetation indices generation function ready (NDRE, NDWI)

✅ Pipeline initialized!

🎯 Target: 2026-03-26
📁 Output: emsa-auto/cloud_free_output/cloud_free_2026-03-26.tif
☁️ Cloud: 47.63%
📊 Cell 1: Cloud=36.54467391304348%, NoData=0.0%

🚀 PROCESSING: 2026-03-26

📥 Step 1: Loading target image...
   Shape: (15, 10000, 9192), dtype: int32

☁️ Step 2: Creating cloud mask...
   • Cloud: 43,776,972

Loading previous: 100%|██████████| 5/5 [00:57<00:00, 11.43s/it]


   • Days from target: [5, 10, 13, 15, 20]

⏱️ Step 4: Temporal rate-of-change interpolation...
   Cloudy pixels with clear reference: 43,776,888


...

Training fusion: 100%|██████████| 12/12 [00:51<00:00,  4.31s/it]


   ✅ Trained on 5 dates, 200,000 samples
   Pixels to predict: 48


   ✅ Fusion: 48

🔲 Step 6: Spatial interpolation (36 remaining)...


Spatial interpolation: 100%|██████████| 12/12 [02:13<00:00, 11.12s/it]


   ✅ Spatial: 36

🎨 Step 7: Edge blending...

💾 Step 8: Saving...
   ✅ Saved: emsa-auto/cloud_free_output/cloud_free_2026-03-26.tif
   🔍 Clear pixel check: diff = 2698393.0

📊 PIPELINE SUMMARY:
   • Initial clouds: 43,776,972 (47.63%)
   • Temporal fill:  43,776,888
   • S1-S2 fusion:   48
   • Spatial fill:   36
   • Unfilled:       0
   • Success rate:   100.00%

✅ Processing complete!
   Result shape: (15, 10000, 9192)
   Result dtype: int32

🌿 GENERATING VEGETATION INDICES (NDRE, NDWI)
   Available: ['NDRE', 'NDWI']
   Image size: 9192 x 10000
   Bands: read
   Dtype: int32

🔬 Computing NDRE...
   NDRE range: [-1.0000, 1.0000]
   NDRE mean:  0.2387

💾 Saving NDRE...
   ✅ Saved: emsa-auto/cloud_free_output/ndre_PROD_20260326.tif
   File size: 379.13 MB

🔬 Computing NDWI...
   NDWI range: [-1.0000, 1.0000]
   NDWI mean:  -0.0964

💾 Saving NDWI...
   ✅ Saved: emsa-auto/cloud_free_output/ndwi_PROD_20260326.tif
   File size: 397.36 MB

📊 VEGETATION INDICES SUMMARY
  Input:  cloud_free_2

In [4]:
# Improved
import rasterio
import numpy as np

# =============================================================================
# Fix NoData value for NDRE and NDWI to match cloud_free (-32768)
# + Clamp pixel values to valid ranges
# =============================================================================

NODATA_VALUE = -32768.0

cloud_free_dir = DOWNLOAD_BASE_DIR / "cloud_free_output"

# Find NDRE and NDWI files for inference date
inference_compact = INFERENCE_DATE.replace('-', '')

ndre_path = cloud_free_dir / f"ndre_PROD_{inference_compact}.tif"
ndwi_path = cloud_free_dir / f"ndwi_PROD_{inference_compact}.tif"

files_to_fix = []
if ndre_path.exists():
    files_to_fix.append(("NDRE", ndre_path))
if ndwi_path.exists():
    files_to_fix.append(("NDWI", ndwi_path))

print(f"🔧 Setting NoData={NODATA_VALUE} for index rasters + clamping values")
print(f"   Inference date: {INFERENCE_DATE}")
print(f"   Output dir: {cloud_free_dir}")
print()

for name, path in files_to_fix:
    with rasterio.open(path) as src:
        old_nodata = src.nodata
        profile = src.profile.copy()
        data = src.read()

    print(f"   {name}: {path.name}")
    print(f"      Before: nodata={old_nodata}")

    # Mark existing NaN as nodata
    nodata_mask = np.isnan(data)
    data[nodata_mask] = NODATA_VALUE

    # Create mask of valid (non-nodata) pixels
    valid_mask = data != NODATA_VALUE

    # Clamp pixel values based on index type
    if name == "NDWI":
        # Values < -0.9 become -0.9
        clamp_low = valid_mask & (data < -0.9)
        data[clamp_low] = -0.9
        print(f"      Clamped {np.count_nonzero(clamp_low)} pixels < -0.9 → -0.9")

        # Values > 0.349 become 0.31
        clamp_high = valid_mask & (data > 0.349)
        data[clamp_high] = 0.31
        print(f"      Clamped {np.count_nonzero(clamp_high)} pixels > 0.349 → 0.31")

    elif name == "NDRE":
        # Values < 0 become 0.1
        clamp_low = valid_mask & (data < 0)
        data[clamp_low] = 0.1
        print(f"      Clamped {np.count_nonzero(clamp_low)} pixels < 0 → 0.1")

        # Values > 0.7 become 0.69
        clamp_high = valid_mask & (data > 0.7)
        data[clamp_high] = 0.69
        print(f"      Clamped {np.count_nonzero(clamp_high)} pixels > 0.7 → 0.69")

    profile.update(nodata=NODATA_VALUE)

    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(data)

    # Verify
    with rasterio.open(path) as src:
        print(f"      After:  nodata={src.nodata}")

    print(f"      ✅ Done")

print(f"\n✅ All index rasters updated with nodata={NODATA_VALUE} and values clamped")
    

🔧 Setting NoData=-32768.0 for index rasters + clamping values
   Inference date: 2026-03-26
   Output dir: emsa-auto/cloud_free_output

   NDRE: ndre_PROD_20260326.tif
      Before: nodata=None
      Clamped 1494303 pixels < 0 → 0.1
      Clamped 254766 pixels > 0.7 → 0.69
      After:  nodata=-32768.0
      ✅ Done
   NDWI: ndwi_PROD_20260326.tif
      Before: nodata=None
      Clamped 30194 pixels < -0.9 → -0.9
      Clamped 831752 pixels > 0.349 → 0.31
      After:  nodata=-32768.0
      ✅ Done

✅ All index rasters updated with nodata=-32768.0 and values clamped


In [5]:
import os
import shutil
from pathlib import Path
from datetime import datetime

def rename_and_move_results(
    inference_folder="amajac-auto",
    output_folder="inputs-amajac-auto",
    prefix="MX06_Grupo Pantaleon",
    location="Amajac"
):
    """
    Rename and move result files from cloud_free_output folder to output folder.
    
    NEW STRUCTURE:
    inference_folder/
    ├── pairs/
    └── cloud_free_output/  <- Files are here now
        ├── cloud_free_2026-02-19.tif
        ├── ndvi_PROD_20260219.tif
        ├── ndre_PROD_20260219.tif
        └── ndwi_PROD_20260219.tif
    
    Example transformation:
    ndvi_PROD_20260219.tif -> MX06_Grupo Pantaleon_NDVI_Amajac_2026_02_19_cloudfill.tif
    """
    
    # Paths
    inference_path = Path(inference_folder)
    cloud_free_path = inference_path / "cloud_free_output"
    output_path = Path(output_folder)
    output_path.mkdir(exist_ok=True)
    
    print(f"Processing files from: {cloud_free_path}")
    print(f"Output folder: {output_path}\n")
    
    if not cloud_free_path.exists():
        print(f"❌ Cloud-free output folder not found: {cloud_free_path}")
        print("   Run Cell 2 (cloud removal) first!")
        return
    
    moved_count = 0
    skipped_count = 0
    
    # Process all .tif files in cloud_free_output folder
    for file in sorted(cloud_free_path.glob("*.tif")):
        filename = file.stem  # filename without extension
        
        # Skip the main cloud_free file (we only want vegetation indices)
        if filename.startswith("cloud_free"):
            print(f"⊙ Skipping cloud-free base file: {file.name}")
            continue
        
        # Extract index type and date from filename
        # Format: ndvi_PROD_20260219.tif or ndre_PROD_20260219.tif
        index_type = None
        date_str = None
        
        for idx in ["ndvi", "ndwi", "ndre", "evi", "savi"]:
            if filename.lower().startswith(idx):
                index_type = idx.upper()
                # Extract date: ndvi_PROD_20260219 -> 20260219
                parts = filename.split("_")
                if len(parts) >= 3 and parts[1].lower() == "prod":
                    date_str = parts[2]  # 20260219
                break
        
        if not index_type or not date_str:
            print(f"⊘ Skipping unknown file format: {file.name}")
            continue
        
        # Convert date: 20260219 -> 2026_02_19
        try:
            if len(date_str) == 8:  # YYYYMMDD
                date_formatted = f"{date_str[0:4]}_{date_str[4:6]}_{date_str[6:8]}"
            else:
                print(f"⊘ Invalid date format in: {file.name}")
                continue
        except:
            print(f"⊘ Could not parse date from: {file.name}")
            continue
        
        # Create new filename
        new_filename = f"{prefix}_{index_type}_{location}_{date_formatted}_cloudfill.tif"
        new_filepath = output_path / new_filename
        
        # Check if file already exists
        if new_filepath.exists():
            print(f"⊙ Already exists, skipping: {new_filename}")
            skipped_count += 1
            continue
        
        # Copy file (use shutil.move() to move instead of copy)
        shutil.copy2(file, new_filepath)
        print(f"✓ {file.name} -> {new_filename}")
        moved_count += 1
    
    print(f"\n✅ Processing complete!")
    print(f"   Files moved: {moved_count}")
    print(f"   Files skipped (already exist): {skipped_count}")
    print(f"   Output folder: {output_path}")

if __name__ == "__main__":
    # Run the script
    rename_and_move_results(**NAME_CONFIG_ENV)

Processing files from: emsa-auto/cloud_free_output
Output folder: inputs-emsa-auto

⊙ Skipping cloud-free base file: cloud_free_2026-03-26.tif
✓ ndre_PROD_20260326.tif -> MX07_Grupo Pantaleon_NDRE_EMSA_2026_03_26_cloudfill.tif
✓ ndwi_PROD_20260326.tif -> MX07_Grupo Pantaleon_NDWI_EMSA_2026_03_26_cloudfill.tif

✅ Processing complete!
   Files moved: 2
   Files skipped (already exist): 0
   Output folder: inputs-emsa-auto


In [6]:
# -*- coding: utf-8 -*-
"""
excel_to_supabase_sync.py - INCREMENTAL UPDATE WITH ZAFRA LOGIC
Implements progressive sync with fecha_fin/fecha_inicio logic and ciclo validation

SYNC MODES:
  - INCREMENTAL (default): Only processes records within the last N days window
                           Set SYNC_INTERVAL_DAYS to match your run frequency (default: 5)
  - LEGACY:                Processes all records up to processing_date (original behavior)
                           Set SYNC_MODE = 'legacy'
"""

import pandas as pd
import os
import logging
from datetime import datetime, timedelta
from supabase import create_client, Client
import warnings
warnings.filterwarnings('ignore')


# =======================================================
# ⚙️  SYNC CONFIGURATION — Edit these as needed
# =======================================================

SYNC_MODE = 'incremental'   # 'incremental' | 'legacy'
SYNC_INTERVAL_DAYS = 25      # Only used when SYNC_MODE = 'incremental'
                             # Set this to match how often this script runs
                             # e.g. 5 = script runs every 5 days

# =======================================================
# TABLE / COLUMN CONFIGURATION
# =======================================================

TABLE_NAME = 'parcelas_ingenios_reprocess'

ALL_COLUMNS = [
    'id_parcela', 'zafra', 'temporada_activa', 'company', 'ingenio',
    'area_calculada', 'area_cosechada', 'area_estimada', 'area_potencial',
    'area_real', 'azucar_estimada', 'azucar_potencial', 'azucar_real',
    'tch_cosechado', 'tch_estimado', 'tch_potencial', 'tch_real',
    'tah_estimado', 'tah_real', 'ton_cosechadas', 'ton_estimada',
    'ton_potencial', 'ton_real', 'ciclo', 'variedad', 'textura_suelo',
    'tipo_riego', 'tipo_corte', 'tipo_cosecha_estimado',
    'tipo_cosecha_real', 'distancia_surco', 'fecha_inicio', 'INICIO',
    'fecha_fin_estimada', 'fecha_fin', 'division_01', 'division_02',
    'division_03', 'division_04', 'division_05', 'division_06',
    'division_07', 'division_08', 'division_09', 'division_10',
    'division_11', 'division_12', 'division_13', 'division_14',
    'division_15', 'division_16', 'division_17', 'division_18',
    'division_19', 'division_20', 'division_21', 'division_22',
    'division_23', 'division_24', 'division_25', 'division_26',
    'division_27', 'division_28', 'division_29', 'division_30',
    'division_31', 'division_32', 'division_33', 'division_34',
    'division_35', 'geometry_polygon', 'geometry_centroid',
    'ingenio_id', 'company_id', 'kg_t_core_zp', 'zafra_st',
]


# =======================================================
# LOGGING CONFIGURATION
# =======================================================

def setup_logging(log_file: str = None):
    logger = logging.getLogger('excel_sync')
    logger.setLevel(logging.DEBUG)
    logger.handlers = []

    if log_file:
        fh = logging.FileHandler(log_file, encoding='utf-8')
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(logging.Formatter(
            '%(asctime)s | %(levelname)-8s | %(funcName)-25s | %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        ))
        logger.addHandler(fh)

    return logger


log_filename = f'sync_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
logger = setup_logging(log_file=log_filename)


# =======================================================
# HELPERS
# =======================================================

def get_date_window(processing_date: str) -> tuple:
    """
    Returns (window_start, window_end) based on SYNC_MODE.

    incremental → window_start = processing_date - SYNC_INTERVAL_DAYS
    legacy      → window_start = None  (no lower bound)
    """
    proc_date = pd.to_datetime(processing_date)

    if SYNC_MODE == 'incremental':
        window_start = proc_date - timedelta(days=SYNC_INTERVAL_DAYS)
        logger.info(f"Mode: INCREMENTAL | Window: {window_start.date()} → {proc_date.date()} ({SYNC_INTERVAL_DAYS}d)")
        return window_start, proc_date
    else:
        logger.info(f"Mode: LEGACY | Window: (all) → {proc_date.date()}")
        return None, proc_date


def get_supabase_client() -> Client:
    url = os.getenv("SUPABASE_URL")
    key = os.getenv("SUPABASE_KEY")
    if not url or not key:
        raise ValueError("SUPABASE_URL and SUPABASE_KEY environment variables required")
    logger.info(f"Supabase connected: {url[:40]}...")
    return create_client(url, key)


def load_excel_data(excel_path: str) -> dict:
    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"Excel not found: {excel_path}")

    data = {}
    for sheet in ['2026', '2027']:
        try:
            df = pd.read_excel(excel_path, sheet_name=sheet)
            df.columns = df.columns.str.strip().str.lower()
            df['zafra'] = int(sheet)
            logger.info(f"Sheet '{sheet}': {len(df)} rows | cols: {list(df.columns)}")
            data[sheet] = df
        except Exception as e:
            logger.warning(f"Sheet '{sheet}' not loaded: {e}")
            data[sheet] = pd.DataFrame()

    return data


def get_past_zafra_record(supabase: Client, id_parcela: str, ingenio: str, current_zafra: int) -> dict:
    try:
        resp = (
            supabase.table(TABLE_NAME)
            .select('*')
            .eq('id_parcela', id_parcela)
            .eq('ingenio', ingenio)
            .lt('zafra', current_zafra)
            .order('zafra', desc=True)
            .limit(1)
            .execute()
        )
        if resp.data:
            rec = resp.data[0]
            logger.debug(f"Past record: {id_parcela} zafra={rec.get('zafra')} ciclo={rec.get('ciclo')}")
            return rec
    except Exception as e:
        logger.error(f"Error fetching past record for {id_parcela}: {e}")
    return None


def validate_ciclo(past_ciclo, new_ciclo, id_parcela: str, past_zafra=None, new_zafra=None) -> tuple:
    logger.info(f"CICLO CHECK: {id_parcela} | zafra {past_zafra}→{new_zafra} | ciclo {past_ciclo}→{new_ciclo}")

    if pd.isna(new_ciclo) if new_ciclo is not None else True:
        return True, "No ciclo provided"

    try:
        new_int = int(float(new_ciclo))
    except (ValueError, TypeError):
        return False, f"Cannot convert new ciclo: {new_ciclo}"

    if past_ciclo is None or (isinstance(past_ciclo, float) and pd.isna(past_ciclo)):
        return True, f"New parcel, ciclo={new_int}"

    try:
        past_int = int(float(past_ciclo))
    except (ValueError, TypeError):
        return True, f"Invalid past ciclo, skipping validation"

    if new_int == past_int + 1:
        return True, f"Ciclo incremented: {past_int}→{new_int}"
    elif new_int == 0:
        return True, f"Ciclo reset: {past_int}→0"
    else:
        return False, f"Invalid transition: {past_int}→{new_int} (expected {past_int + 1} or 0)"


def prepare_record_data(row: pd.Series, excel_cols: list) -> dict:
    NUMERIC_COLS = {
        'area_calculada', 'area_cosechada', 'area_estimada', 'area_potencial',
        'area_real', 'azucar_estimada', 'azucar_potencial', 'azucar_real',
        'tch_cosechado', 'tch_estimado', 'tch_potencial', 'tch_real',
        'tah_estimado', 'tah_real', 'ton_cosechadas', 'ton_estimada',
        'ton_potencial', 'ton_real', 'ciclo', 'distancia_surco', 'kg_t_core_zp',
    }
    NUMERIC_COLS.update({f'division_{str(i).zfill(2)}' for i in range(1, 36)})

    DATE_COLS   = {'fecha_inicio', 'fecha_fin', 'fecha_fin_estimada', 'INICIO'}
    STRING_COLS = {
        'id_parcela', 'zafra', 'company', 'ingenio', 'variedad',
        'textura_suelo', 'tipo_riego', 'tipo_corte', 'tipo_cosecha_estimado',
        'tipo_cosecha_real', 'geometry_polygon', 'geometry_centroid',
        'ingenio_id', 'company_id', 'zafra_st',
    }
    GARBAGE = {'', '-', '—', 'N/A', 'n/a', 'NA', 'None', 'nan', '#N/A', '#VALUE!', '#REF!'}

    data = {}
    for col in excel_cols:
        if col not in row.index:
            continue
        value = row[col]

        if value is None:
            continue
        try:
            if pd.isna(value):
                continue
        except (ValueError, TypeError):
            pass

        if isinstance(value, str):
            value = value.strip()
            if value in GARBAGE:
                continue

        if col == 'zafra':
            try:
                data[col] = str(int(float(value)))
            except (ValueError, TypeError):
                pass
            continue

        if col == 'temporada_activa':
            if isinstance(value, bool):
                data[col] = value
            elif isinstance(value, str):
                data[col] = value.lower() in ('true', '1', 'yes')
            else:
                data[col] = bool(value)
            continue

        if col in NUMERIC_COLS:
            try:
                num = float(value)
                data[col] = int(num) if num == int(num) else num
            except (ValueError, TypeError):
                logger.warning(f"  Non-numeric in {col}: {repr(value)}")
            continue

        if col in DATE_COLS:
            if isinstance(value, pd.Timestamp):
                data[col] = value.strftime('%Y-%m-%d')
            elif isinstance(value, str):
                try:
                    pd.to_datetime(value)
                    data[col] = value
                except (ValueError, TypeError):
                    logger.warning(f"  Invalid date in {col}: {repr(value)}")
            continue

        if col in STRING_COLS:
            data[col] = str(value)
            continue

        # Fallback
        if isinstance(value, pd.Timestamp):
            data[col] = value.strftime('%Y-%m-%d')
        elif isinstance(value, float):
            data[col] = int(value) if value.is_integer() else value
        elif isinstance(value, int):
            data[col] = value
        else:
            data[col] = str(value)

    return data


def filter_by_ingenio(df: pd.DataFrame, ingenio: str) -> pd.DataFrame:
    if 'ingenio' not in df.columns or df.empty:
        return df
    normalized = ingenio.lower().replace('_', ' ')
    return df[df['ingenio'].str.lower().str.replace('_', ' ') == normalized].copy()


# =======================================================
# MAIN SYNC
# =======================================================

def sync_excel_to_supabase(
    excel_path: str,
    ingenio: str,
    processing_date: str,
    dry_run: bool = False
) -> dict:
    """
    Synchronize Excel → Supabase with configurable date windowing.

    SYNC_MODE = 'incremental':
        Step 1: Close 2026 parcels where fecha_fin falls in [processing_date - SYNC_INTERVAL_DAYS, processing_date]
        Step 2: Process 2027 parcels where fecha_inicio falls in same window

    SYNC_MODE = 'legacy':
        Step 1: Close 2026 parcels where fecha_fin <= processing_date  (all up to date)
        Step 2: Process 2027 parcels where fecha_inicio <= processing_date
    """

    window_start, window_end = get_date_window(processing_date)
    mode_label = f"INCREMENTAL ({SYNC_INTERVAL_DAYS}d window)" if SYNC_MODE == 'incremental' else "LEGACY (all up to date)"

    # ── Console header ──────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"  SYNC: {ingenio} | Date: {processing_date} | Mode: {mode_label}")
    if SYNC_MODE == 'incremental':
        print(f"  Window: {window_start.date()} → {window_end.date()}")
    print(f"  {'[DRY RUN]' if dry_run else '[PRODUCTION]'}")
    print(f"{'='*70}")

    logger.info("=" * 80)
    logger.info(f"SYNC START | ingenio={ingenio} | date={processing_date} | mode={SYNC_MODE}")
    logger.info(f"  Table: {TABLE_NAME} | dry_run={dry_run}")
    logger.info("=" * 80)

    excel_data   = load_excel_data(excel_path)
    df_2026      = filter_by_ingenio(excel_data.get('2026', pd.DataFrame()), ingenio)
    df_2027      = filter_by_ingenio(excel_data.get('2027', pd.DataFrame()), ingenio)

    print(f"  📊 Loaded: 2026={len(df_2026)} | 2027={len(df_2027)} records (after ingenio filter)")
    logger.info(f"After filter: 2026={len(df_2026)}, 2027={len(df_2027)}")

    stats = {
        'closed_2026': 0, 'inserted_2027': 0, 'updated_2027': 0,
        'ciclo_errors': 0, 'errors': 0, 'skipped': 0,
        'columns_updated_2026': set(), 'columns_updated_2027': set(),
    }
    ciclo_errors_log = []
    update_log_2026  = []
    update_log_2027  = []

    supabase = None if dry_run else get_supabase_client()

    # ================================================================
    # STEP 1: Close finished 2026 parcels
    # ================================================================
    print(f"\n  📌 Step 1: Closing 2026 parcels...")

    if not df_2026.empty and 'fecha_fin' in df_2026.columns:
        df_2026['fecha_fin'] = pd.to_datetime(df_2026['fecha_fin'], errors='coerce')

        if SYNC_MODE == 'incremental':
            # Only the parcels whose fecha_fin falls inside the current window
            mask = (
                df_2026['fecha_fin'].notna() &
                (df_2026['fecha_fin'] >= window_start) &
                (df_2026['fecha_fin'] <= window_end)
            )
            print(f"     Window: {window_start.date()} ≤ fecha_fin ≤ {window_end.date()}")
        else:
            # Legacy: everything up to processing_date
            mask = df_2026['fecha_fin'].notna() & (df_2026['fecha_fin'] <= window_end)
            print(f"     Window: fecha_fin ≤ {window_end.date()} (all)")

        df_to_close = df_2026[mask].copy()
        logger.info(f"Parcels to close: {len(df_to_close)}")

        if not dry_run:
            for _, row in df_to_close.iterrows():
                id_parcela = str(row['id_parcela'])
                try:
                    excel_cols  = [c for c in df_2026.columns if c in ALL_COLUMNS]
                    update_data = prepare_record_data(row, excel_cols)
                    update_data['temporada_activa'] = False
                    stats['columns_updated_2026'].update(update_data.keys())

                    resp = (
                        supabase.table(TABLE_NAME)
                        .update(update_data)
                        .eq('id_parcela', id_parcela)
                        .eq('ingenio', ingenio)
                        .eq('zafra', '2026')
                        .execute()
                    )
                    if resp.data:
                        stats['closed_2026'] += 1
                        update_log_2026.append({
                            'id_parcela': id_parcela,
                            'fecha_fin': str(row['fecha_fin']),
                            'columns_updated': list(update_data.keys()),
                        })
                    else:
                        stats['skipped'] += 1

                except Exception as e:
                    stats['errors'] += 1
                    logger.error(f"Error closing {id_parcela}: {e}")
                    print(f"  ❌ Error closing {id_parcela}: {str(e)[:60]}")
        else:
            stats['closed_2026'] = len(df_to_close)

        print(f"     ✓ Closed: {stats['closed_2026']} parcels")
    else:
        print(f"     ⚠  No fecha_fin column or empty 2026 sheet")
        logger.warning("Skipping Step 1: no fecha_fin or empty sheet")

    # ================================================================
    # STEP 2: Process 2027 parcels
    # ================================================================
    print(f"\n  📌 Step 2: Processing 2027 parcels...")

    if not df_2027.empty and 'fecha_inicio' in df_2027.columns:
        df_2027['fecha_inicio'] = pd.to_datetime(df_2027['fecha_inicio'], errors='coerce')

        if SYNC_MODE == 'incremental':
            mask = (
                df_2027['fecha_inicio'].notna() &
                (df_2027['fecha_inicio'] >= window_start) &
                (df_2027['fecha_inicio'] <= window_end)
            )
            print(f"     Window: {window_start.date()} ≤ fecha_inicio ≤ {window_end.date()}")
        else:
            mask = df_2027['fecha_inicio'].notna() & (df_2027['fecha_inicio'] <= window_end)
            print(f"     Window: fecha_inicio ≤ {window_end.date()} (all)")

        df_to_process = df_2027[mask].copy()
        logger.info(f"Parcels to process: {len(df_to_process)}")

        if not dry_run:
            for _, row in df_to_process.iterrows():
                id_parcela = str(row['id_parcela'])
                try:
                    past_record = get_past_zafra_record(supabase, id_parcela, ingenio, 2027)

                    if past_record:
                        # ── Existing parcel: validate ciclo ──────────────
                        past_zafra = past_record.get('zafra')
                        past_ciclo = past_record.get('ciclo')
                        new_ciclo  = row.get('ciclo') if 'ciclo' in row.index else None
                        if new_ciclo is not None and pd.isna(new_ciclo):
                            new_ciclo = None

                        is_valid, message = validate_ciclo(
                            past_ciclo, new_ciclo, id_parcela,
                            past_zafra=past_zafra, new_zafra=2027
                        )

                        if not is_valid:
                            stats['ciclo_errors'] += 1
                            ciclo_errors_log.append({
                                'id_parcela': id_parcela,
                                'past_zafra': past_zafra, 'new_zafra': 2027,
                                'past_ciclo': past_ciclo, 'new_ciclo': new_ciclo,
                                'message': message,
                            })
                            print(f"  ⚠  Ciclo error: {id_parcela} ({past_ciclo}→{new_ciclo})")
                            continue

                        # Inherit past record, overwrite with Excel values
                        new_record = {
                            k: v for k, v in past_record.items()
                            if k in ALL_COLUMNS and k not in ('id', 'created_at', 'updated_at')
                        }
                        excel_cols = [c for c in df_2027.columns if c in ALL_COLUMNS]
                        new_record.update(prepare_record_data(row, excel_cols))
                        new_record.update({
                            'zafra': '2027',
                            'temporada_activa': True,
                            'ingenio': ingenio,
                            'id_parcela': id_parcela,
                        })
                        stats['columns_updated_2027'].update(new_record.keys())

                        # Upsert
                        existing = (
                            supabase.table(TABLE_NAME)
                            .select('id')
                            .eq('id_parcela', id_parcela)
                            .eq('ingenio', ingenio)
                            .eq('zafra', '2027')
                            .execute()
                        )
                        if existing.data:
                            supabase.table(TABLE_NAME) \
                                .update(new_record) \
                                .eq('id_parcela', id_parcela) \
                                .eq('ingenio', ingenio) \
                                .eq('zafra', '2027') \
                                .execute()
                            stats['updated_2027'] += 1
                            action = 'UPDATE'
                        else:
                            supabase.table(TABLE_NAME).insert(new_record).execute()
                            stats['inserted_2027'] += 1
                            action = 'INSERT'

                        update_log_2027.append({
                            'id_parcela': id_parcela, 'action': action,
                            'inherited_from_zafra': past_zafra,
                            'ciclo_transition': f"{past_ciclo}→{new_ciclo}",
                            'column_count': len(new_record),
                        })

                    else:
                        # ── Brand new parcel ─────────────────────────────
                        excel_cols = [c for c in df_2027.columns if c in ALL_COLUMNS]
                        new_record = prepare_record_data(row, excel_cols)
                        new_record.update({
                            'id_parcela': id_parcela,
                            'ingenio': ingenio,
                            'zafra': '2027',
                            'temporada_activa': True,
                        })
                        stats['columns_updated_2027'].update(new_record.keys())
                        supabase.table(TABLE_NAME).insert(new_record).execute()
                        stats['inserted_2027'] += 1

                        update_log_2027.append({
                            'id_parcela': id_parcela, 'action': 'INSERT_NEW',
                            'inherited_from_zafra': None,
                            'ciclo_transition': f"N/A→{new_record.get('ciclo', 'N/A')}",
                            'column_count': len(new_record),
                        })

                except Exception as e:
                    stats['errors'] += 1
                    logger.error(f"Error processing {id_parcela}: {e}")
                    print(f"  ❌ Error: {id_parcela}: {str(e)[:60]}")
        else:
            stats['inserted_2027'] = len(df_to_process)
    else:
        print(f"     ⚠  No fecha_inicio column or empty 2027 sheet")
        logger.warning("Skipping Step 2: no fecha_inicio or empty sheet")

    # ================================================================
    # SUMMARY
    # ================================================================
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    print(f"\n{'─'*70}")
    print(f"  📊 SUMMARY  ({mode_label})")
    print(f"{'─'*70}")
    print(f"     2026 Closed:    {stats['closed_2026']}")
    print(f"     2027 Inserted:  {stats['inserted_2027']}")
    print(f"     2027 Updated:   {stats['updated_2027']}")
    if stats['ciclo_errors']:
        print(f"     ⚠  Ciclo Errors: {stats['ciclo_errors']}")
    if stats['errors']:
        print(f"     ❌ Errors:        {stats['errors']}")
    print(f"{'─'*70}")
    print(f"  📄 Log: {log_filename}")
    print(f"{'='*70}\n")

    logger.info("SYNC COMPLETE")
    logger.info(f"  closed_2026={stats['closed_2026']} inserted_2027={stats['inserted_2027']} "
                f"updated_2027={stats['updated_2027']} ciclo_errors={stats['ciclo_errors']} errors={stats['errors']}")

    # Save CSV logs
    if update_log_2026:
        p = f'update_log_2026_{timestamp}.csv'
        pd.DataFrame(update_log_2026).to_csv(p, index=False)
        logger.info(f"2026 log: {p}")

    if update_log_2027:
        p = f'update_log_2027_{timestamp}.csv'
        pd.DataFrame(update_log_2027).to_csv(p, index=False)
        logger.info(f"2027 log: {p}")

    if ciclo_errors_log:
        p = f'ciclo_errors_{timestamp}.csv'
        pd.DataFrame(ciclo_errors_log).to_csv(p, index=False)
        logger.info(f"Ciclo errors: {p}")
        print(f"  ⚠  Ciclo errors saved: {p}")

    stats['columns_updated_2026'] = sorted(stats['columns_updated_2026'])
    stats['columns_updated_2027'] = sorted(stats['columns_updated_2027'])

    return stats


# =======================================================
# MULTI-DATE PROCESSING
# =======================================================

def procesar_con_sync(
    excel_path: str,
    ingenio: str,
    fechas: list,
    productos: list,
    input_dir: str,
    output_dir: str,
    bd_insert: bool = False,
    dry_run_sync: bool = False,
):
    """
    Process multiple dates with incremental sync.
    Import procesar_todos_productos from your main_processing module.
    """

    if isinstance(fechas, str):
        fechas = [fechas]

    print(f"\n{'#'*70}")
    print(f"  MULTI-DATE: {ingenio} | Dates: {fechas} | Mode: {SYNC_MODE.upper()}")
    print(f"{'#'*70}")

    logger.info(f"MULTI-DATE | ingenio={ingenio} | dates={fechas} | products={productos}")

    all_sync_stats = {}
    all_results    = {}

    for idx, fecha in enumerate(fechas, 1):
        print(f"\n{'─'*70}")
        print(f"  [{idx}/{len(fechas)}] {fecha}")
        print(f"{'─'*70}")

        # Sync
        try:
            all_sync_stats[fecha] = sync_excel_to_supabase(
                excel_path=excel_path,
                ingenio=ingenio,
                processing_date=fecha,
                dry_run=dry_run_sync,
            )
        except Exception as e:
            logger.error(f"Sync error {fecha}: {e}")
            print(f"  ❌ Sync error: {e}")
            all_sync_stats[fecha] = {'error': str(e)}

        # Product processing
        try:
            resultados = procesar_todos_productos(
                ingenio=ingenio, fecha=fecha, productos=productos,
                input_dir=input_dir, output_dir=output_dir,
                zafras=None, bd_insert=bd_insert, id_field='id_parcela',
            )
            all_results[fecha] = resultados
            ok = sum(1 for df in resultados.values() if df is not None)
            print(f"  ✓ Products: {ok}/{len(productos)}")
        except Exception as e:
            logger.error(f"Processing error {fecha}: {e}")
            print(f"  ❌ Processing error: {e}")
            all_results[fecha] = None

    # Final summary
    print(f"\n{'#'*70}")
    print(f"  FINAL SUMMARY")
    print(f"{'#'*70}")
    for fecha in fechas:
        sync    = all_sync_stats.get(fecha, {})
        results = all_results.get(fecha)
        if 'error' in sync:
            print(f"  {fecha}: ❌ Sync Error — {sync['error']}")
        elif results is None:
            print(f"  {fecha}: ❌ Processing Error")
        else:
            ok = sum(1 for df in results.values() if df is not None)
            print(f"  {fecha}: ✓ Sync(closed={sync.get('closed_2026',0)} "
                  f"ins={sync.get('inserted_2027',0)} upd={sync.get('updated_2027',0)}) "
                  f"| Products({ok}/{len(productos)})")
    print(f"{'#'*70}")
    print(f"  📄 Log: {log_filename}")
    print(f"{'#'*70}\n")

    return all_results

In [7]:
# -*- coding: utf-8 -*-
"""
Procesamiento de productos finales usando bibliotecas open source
Procesa: NDVI, NDWI, Smart Growth y Weed
Reemplazo completo de ArcGIS ModelBuilder con geopandas, rasterio, numpy, pandas
"""

import geopandas as gpd
from pathlib import Path
import gc
# from supabase.lib.client_options import ClientOptions 
from supabase import create_client, Client
import rasterio
from rasterio.mask import mask
from rasterio.features import shapes, rasterize
import numpy as np
import pandas as pd
from shapely.geometry import shape, mapping, box 
from datetime import datetime
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from shapely import wkb, wkt
import json

# Supabase (solo importar si BD_INSERT=True)
try:
    from supabase import create_client, Client
    SUPABASE_AVAILABLE = True
except ImportError:
    SUPABASE_AVAILABLE = False

# # =======================================================
# # CARGAR CURVAS POTENCIALES
# # =======================================================
# print("📊 Cargando curvas potenciales...")

from pathlib import Path

# Ruta a las curvas potenciales
BASE_DIR = Path.cwd()  # or Path(__file__).resolve().parent

CURVAS_DIR = Path(OUTPUT_DIR_ENV)
if not CURVAS_DIR.is_absolute():
    CURVAS_DIR = BASE_DIR / CURVAS_DIR


# Diccionario para almacenar curvas por ingenio
CURVAS_POTENCIALES = {}

# Función para obtener valor potencial de la curva
def obtener_potencial_curva(ingenio, edad_dias, indice='NDRE'):
    """
    Obtiene el valor potencial para una edad dada desde la curva cargada.
    
    Args:
        ingenio: Nombre del ingenio ('CAC', 'SANTA_CLARA', etc.)
        edad_dias: Edad en días
        indice: 'NDRE' o 'NDWI'
    
    Returns:
        Valor potencial para esa edad, o None si no hay curva disponible
    """
    if ingenio not in CURVAS_POTENCIALES:
        return None
    
    df_curva = CURVAS_POTENCIALES[ingenio]
    col = f'{indice}_potencial'
    
    if col not in df_curva.columns:
        return None
    
    # Si la edad está en la tabla, retornar directamente
    match = df_curva[df_curva['edad_dias'] == edad_dias]
    if len(match) > 0:
        return match.iloc[0][col]
    
    # Si no está, interpolar
    edad_min = df_curva['edad_dias'].min()
    edad_max = df_curva['edad_dias'].max()
    
    if edad_dias < edad_min:
        return df_curva.iloc[0][col]
    elif edad_dias > edad_max:
        return df_curva.iloc[-1][col]
    else:
        # Interpolación lineal
        df_sorted = df_curva.sort_values('edad_dias')
        return np.interp(edad_dias, df_sorted['edad_dias'], df_sorted[col])

# =======================================================
# DICCIONARIO MAESTRO DE INGENIOS
# =======================================================
INGENIOS_META = {
    "Pantaleon": { # Use the name you filter by in the CSV
        "EMPRESA": "Grupo Pantaleon", #"Pantaleon", # Use the code from the file name convention # GP
        "PAIS": "GT",      # Based on file prefix / bounds context
        "AOI_BOUNDS": {"west": -91.530544, "south": 13.947144, "east": -90.590395, "north": 14.445327}, # Your provided bounds
        "CRS": "EPSG:32615",  # **CRITICAL: Must match the raster CRS you confirmed!** #/ EPSG:4326
        "TILE_SIZE_KM": 20 # Dummy value, as you suspected
    },
    "Monte_Rosa": {
        "EMPRESA": "Grupo Pantaleon",
        "PAIS": "NI",
        "AOI_BOUNDS": {
            "west": -87.502239,
            "south": 12.279927,
            "east": -86.79237,
            "north": 12.948203
        },
        "CRS": "EPSG:32616",
        "TILE_SIZE_KM": 20
    },

    "Amajac": {
        "EMPRESA": "Grupo Pantaleon",
        "PAIS": "MX06",
        "AOI_BOUNDS": {
            "west": -98.543185,
            "south": 21.662828,
            "east": -98.021094,
            "north": 22.280718
        },
        "CRS": "EPSG:32614",
        "TILE_SIZE_KM": 20
    },

    "EMSA": {
        "EMPRESA": "Grupo Pantaleon",
        "PAIS": "MX07",
        "AOI_BOUNDS": {
            "west": -99.353038,
            "south": 22.437112,
            "east": -98.49084,
            "north": 23.31672
        },
        "CRS": "EPSG:32614",
        "TILE_SIZE_KM": 20
    },

    "IPSA": {
        "EMPRESA": "Grupo Pantaleon",
        "PAIS": "MX02",
        "AOI_BOUNDS": {
            "west": -98.623859,
            "south": 21.65155,
            "east": -97.987145,
            "north": 22.508884
        },
        "CRS": "EPSG:32614",
        "TILE_SIZE_KM": 20
    }
}

print("✅ Diccionario de ingenios cargado")
print(f"   Total ingenios configurados: {len(INGENIOS_META)}")
for ing, meta in INGENIOS_META.items():
    bounds = meta['AOI_BOUNDS']
    area_km2 = (bounds['east'] - bounds['west']) * 111 * (bounds['north'] - bounds['south']) * 111
    print(f"   - {ing:15} ({meta['PAIS']}/{meta['EMPRESA']}): ~{area_km2:.0f} km²")

# =======================================================
# FORMULAS POLINOMICAS (FALLBACK)
# =======================================================
# Se usan solo si no existe el archivo parquet de curva.
# x = edad en dias
FORMULAS_POTENCIALES = {
    "EMSA": {
        "NDRE": lambda x: (
            0.000000000000098 * (x**5) - 
            0.000000000051161 * (x**4) - 
            0.000000012525467 * (x**3) + 
            0.000005959336438 * (x**2) + 
            0.000739203312554 * x + 
            0.199493195410753
        ),
        "NDWI": lambda x: (
            -0.000000000000268 * (x**5) + 
            0.000000000365123 * (x**4) - 
            0.000000187095158 * (x**3) + 
            0.000037974035589 * (x**2) - 
            0.001211159730837 * x - 
            0.103648224595588
        )
    },
    "Amajac": {
        "NDRE": lambda x: (
            -0.000000000000656 * (x**5) + 
            0.000000000626723 * (x**4) - 
            0.000000228534347 * (x**3) + 
            0.000033607870585 * (x**2) - 
            0.000346568129761 * x + 
            0.221016483453894
        ),
        "NDWI": lambda x: (
            -0.000000000001057 * (x**5) + 
            0.000000001000451 * (x**4) - 
            0.000000355156276 * (x**3) + 
            0.000051721075749 * (x**2) - 
            0.001023270894495 * x - 
            0.087364430989297
        )
    },
    "IPSA": {
        "NDRE": lambda x: (
            -0.000000000000676 * (x**5) + 
            0.000000000533030 * (x**4) - 
            0.000000155353342 * (x**3) + 
            0.000017696957070 * (x**2) + 
            0.000556844849307 * x + 
            0.214562788170895
        ),
        "NDWI": lambda x: (
            -0.000000000001388 * (x**5) + 
            0.000000001173416 * (x**4) - 
            0.000000344287333 * (x**3) + 
            0.000035529951513 * (x**2) + 
            0.000857207455698 * x - 
            0.087356195373687
        )
    },
    "Pantaleon": {
        "NDRE": lambda x: (
            -0.000000000000047 * (x**5) + 
            0.000000000019120 * (x**4) + 
            0.000000005022705 * (x**3) - 
            0.000010922540182 * (x**2) + 
            0.003816187604734 * x + 
            0.149990990306719
        ),
        "NDWI": lambda x: (
            -0.000000000001043 * (x**5) + 
            0.000000000994730 * (x**4) - 
            0.000000335383715 * (x**3) + 
            0.000037700434707 * (x**2) + 
            0.001739266187149 * x - 
            0.134608730808750
        )
    },
    "Monte Rosa": {
        "NDRE": lambda x: (
            0.000000000000096 * (x**5) + 
            0.000000000055232 * (x**4) - 
            0.000000086619735 * (x**3) + 
            0.000018148797811 * (x**2) + 
            0.001270882911438 * x + 
            0.178959414402129
        ),
        "NDWI": lambda x: (
            -0.000000000001572 * (x**5) + 
            0.000000001636878 * (x**4) - 
            0.000000621366573 * (x**3) + 
            0.000092411080706 * (x**2) - 
            0.001803549089450 * x - 
            0.113900219568478
        )
    }
}


# =======================================================
# CONFIGURACIONES DE PRODUCTOS
# =======================================================
PRODUCTOS_CONFIG = {
    "NDVI": {
        "tipo": "simple",  # Procesamiento estándar
        "input_suffix": "NDRE",
        "potencial_formula": lambda edad, ingenio: (
            # 1. Try loading parquet file
            obtener_potencial_curva(ingenio, edad, 'NDRE') 
            # 2. If it is None, search in the formula dictionary
            if obtener_potencial_curva(ingenio, edad, 'NDRE') is not None
            else (
                FORMULAS_POTENCIALES[ingenio]['NDRE'](edad) 
                if ingenio in FORMULAS_POTENCIALES 
                else None # Si no hay ni archivo ni formula, error controlado
            )
        ),
        
        # "potencial_formula": lambda edad, ingenio='CAC': (
        #     # Intentar usar curva cargada, si no existe usar fórmula hardcodeada
        #     obtener_potencial_curva(ingenio, edad, 'NDRE') or (
        #         0.000000000000028 * (edad**6) - 
        #         0.000000000031353 * (edad**5) + 
        #         0.000000013102096 * (edad**4) - 
        #         0.000002457541752 * (edad**3) + 
        #         0.000175180254613 * (edad**2) + 
        #         0.002445789888924 * edad + 
        #         0.188780255063534
        #     )
        # ),

        "reclass_ranges": [
            (-1, 0.1, 1), (0.1, 0.2, 2), (0.2, 0.3, 3), (0.3, 0.4, 4),
            (0.40, 0.5, 5), (0.5, 0.55, 6), (0.55, 0.6, 7), (0.60, 0.625, 8),
            (0.625, 0.65, 9), (0.65, 0.675, 10), (0.675, 0.70, 11), (0.70, 0.725, 12),
            (0.725, 0.75, 13), (0.75, 1.0, 14)
        ],
        "reclass_potencial": [
            (-9000, 0.6, 1), (0.6, 0.9, 2), (0.9, 1.1, 3),
            (1.1, 1.3, 4), (1.3, 9000, 5)
        ],
        "cosechado_classes": [1, 2, 3]
    },
    "NDWI": {
        "tipo": "simple",  # Procesamiento estándar
        "input_suffix": "NDWI",
        "potencial_formula": lambda edad, ingenio: (
            # 1. Try loading parquet file
            obtener_potencial_curva(ingenio, edad, 'NDWI') 
            # 2. If it is None, search in the formula dictionary
            if obtener_potencial_curva(ingenio, edad, 'NDWI') is not None
            else (
                FORMULAS_POTENCIALES[ingenio]['NDWI'](edad) 
                if ingenio in FORMULAS_POTENCIALES 
                else None
            )
        ),

        # "potencial_formula": lambda edad, ingenio='CAC': (
        #     # Intentar usar curva cargada, si no existe usar fórmula hardcodeada
        #     obtener_potencial_curva(ingenio, edad, 'NDWI') or (
        #         0.0000046831 * (edad**2) - 0.0023419769 * edad - 0.2889615388
        #     )
        # ),
        "reclass_ranges": [
            (-1, -0.15, 1), (-0.15, -0.05, 2), (-0.05, 0.05, 3), (0.05, 0.15, 4),
            (0.15, 0.20, 5), (0.20, 0.25, 6), (0.25, 0.30, 7), (0.30, 0.325, 8),
            (0.325, 0.35, 9), (0.35, 0.375, 10), (0.375, 0.40, 11), (0.40, 0.425, 12),
            (0.425, 0.45, 13), (0.45, 1, 14)
        ],
        "reclass_potencial": [
            (-9000, 0.6, 1), (0.6, 0.9, 2), (0.9, 1.1, 3),
            (1.1, 1.3, 4), (1.3, 9000, 5)
        ],
        "cosechado_classes": []
    },
    "SMART_GROWTH": {
        "tipo": "combinado",  # Combina potenciales NDVI y NDWI
        "input_suffix": None,
        "requires": ["NDVI", "NDWI"],
        "potencial_formula": None,
        "reclass_ranges": None,
        "reclass_potencial": None,
        "cosechado_classes": []
    },
    "WEED": {
        "tipo": "complejo",  # Procesamiento especial para detección de malezas
        "input_suffix": "NDRE",
        "potencial_formula": None,
        "reclass_ranges": [
            (-10, 0, 2),  # ≤ 0: NO maleza (NDVI ≤ umbral = normal)
            (0, 10, 1)    # > 0: SÍ maleza (NDVI > umbral = exceso de vigor)
        ],
        "reclass_potencial": None,
        "cosechado_classes": [],
        "buffer_distance": 0,  # Sin buffer
        "maleza_threshold_formula": lambda mean, std: mean + (1.5 * std),  # Umbral ALTO para detectar exceso
        "maleza_min": 0,
        "maleza_max": 1,
        "maleza_critica_percent": 15
    }
}

# =======================================================
# FUNCIONES AUXILIARES
# =======================================================

def get_supabase_client():
    """Obtiene cliente de Supabase desde variables de entorno"""
    if not SUPABASE_AVAILABLE:
        raise ImportError("Librería 'supabase' no instalada. Ejecute: pip install supabase")
    
    url = os.getenv("SUPABASE_URL")
    key = os.getenv("SUPABASE_KEY")
    
    if not url or not key:
        raise ValueError("Variables de entorno SUPABASE_URL y SUPABASE_KEY requeridas")
    
    # REMOVE ALL ClientOptions code - just return simple client
    return create_client(url, key)

from pathlib import Path

# 🔧 MANUALLY HARDCODE CURVE FILE PATHS HERE
CURVAS_PARQUET_PATHS = {
    "EMSA": Path("Output-emsa/EMSA_curvas/curva_global_EMSA.parquet"),
    "Monte Rosa": Path("Output-monterosa/Monte Rosa_curvas/curva_global_Monte Rosa.parquet"),
    "IPSA": Path("Output-ipsa/IPSA_curvas/curva_global_IPSA.parquet"),
    "Amajac": Path("Output-amajac/Amajac_curvas/curva_global_Amajac.parquet"),
    "Pantaleon": Path("Output-gt/Pantaleon_curvas/curva_global_Pantaleon.parquet")
}


def cargar_curva_dinamica(ingenio):
    """
    Carga la curva potencial desde una ruta hardcodeada por ingenio.
    Si falla, usa fórmulas de respaldo.
    """
    print(f"📊 Buscando curvas potenciales para: {ingenio}...")

    ruta_completa = CURVAS_PARQUET_PATHS.get(ingenio)

    if ruta_completa is not None:
        try:
            if ruta_completa.exists():
                df_curva = pd.read_parquet(ruta_completa)
                CURVAS_POTENCIALES[ingenio] = df_curva
                print(f"   ✅ Curva Parquet cargada exitosamente: {len(df_curva)} edades")
                return True
            else:
                print(f"   ⚠️  Archivo no encontrado: {ruta_completa}")
        except Exception as e:
            print(f"   ❌ Error leyendo archivo parquet: {str(e)}")
    else:
        print(f"   ⚠️  No hay ruta hardcodeada para {ingenio}")

    # 🔁 Fallback a fórmulas
    print(f"   ⚠️  No se cargó archivo de curva. Verificando fórmulas de respaldo...")

    if ingenio in FORMULAS_POTENCIALES:
        print(f"   ✅ Fórmulas polinómicas encontradas para {ingenio}.")
        print(f"      -> Se usarán ecuaciones hardcodeadas (R² ~ 0.90+)")
        return False
    else:
        print(f"   ❌ CRITICO: No hay ni archivo Parquet ni Fórmula para {ingenio}!")
        print(f"      -> El cálculo de potencial fallará (NaN).")
        return False


def construir_lookup_potencial(ingenio, input_dir):
    """
    Loads the consolidated curve parquet and JSON mapping file,
    builds group names for matching, runs the matching ladder,
    and returns a lookup dictionary keyed by id_parcela.
    
    Rules:
    - ciclo and tipo_riego: must be numeric digits only → binary (0/1) → else OTRAS
    - mes_nombre: from fecha_fin_estimada (sane year) → fecha_inicio+360d → else OTRAS
    - division_03: if present use as-is → if NULL or garbage → OTRAS
    - Matching ladder: always full (4var → 3var → GLOBAL), no cheating
    - OTRAS in group name does NOT skip matching — ladder runs in full always
    
    Args:
        ingenio: Original ingenio name e.g. "Monte Rosa"
        input_dir: Path to inputs folder where JSON and parquet live
    
    Returns:
        dict keyed by id_parcela:
        {
            id_parcela: {
                'matched_curve_id': str,
                'match_tier': str,
            }
        }
        Also returns df_curvas (the full parquet DataFrame) for age lookups later.
    """
    import json
    
    ingenio_key = ingenio.replace(' ', '_')
    input_path = Path(input_dir)
    
    # ------------------------------------------------------------------
    # STEP 1: Determine country for this ingenio
    # ------------------------------------------------------------------
    meta = get_ingenio_meta(ingenio_key)
    pais = meta['PAIS']
    is_ni_gt = pais in ['NI', 'GT']
    
    print(f"\n📐 Construyendo lookup potencial para: {ingenio} ({pais})")
    print(f"   Lógica de agrupación: {'4-var (Ciclo+Mes+Riego+Estrato)' if is_ni_gt else '3-var (Ciclo+Mes+Riego)'}")
    
    # ------------------------------------------------------------------
    # STEP 2: Load JSON mapping file
    # ------------------------------------------------------------------
    json_filename = f"mapping_production_{ingenio_key}.json"
    json_path = input_path / json_filename
    
    if not json_path.exists():
        print(f"   ⚠ JSON mapping file not found: {json_filename}")
        print(f"   → Potencial lookup will NOT be available. Falling back to formula/global.")
        return None, None
    
    with open(json_path, 'r', encoding='utf-8') as f:
        production_map = json.load(f)
    
    lookup_json = production_map.get('lookup', {})
    print(f"   ✅ JSON loaded: {json_filename} ({len(lookup_json)} entries)")
    
    # ------------------------------------------------------------------
    # STEP 3: Load consolidated parquet
    # ------------------------------------------------------------------
    parquet_filename = f"curvas_potenciales_{ingenio_key}_consolidado.parquet"
    parquet_path = input_path / parquet_filename
    
    if not parquet_path.exists():
        print(f"   ⚠ Consolidated parquet not found: {parquet_filename}")
        print(f"   → Potencial lookup will NOT be available. Falling back to formula/global.")
        return None, None
    
    df_curvas = pd.read_parquet(parquet_path)
    available_curve_ids = set(df_curvas['curve_id'].unique())
    print(f"   ✅ Parquet loaded: {parquet_filename}")
    print(f"   ✅ Unique curves available: {len(available_curve_ids)}")
    
    # ------------------------------------------------------------------
    # STEP 4: Return assets (parcelas not available yet at this stage)
    # We return df_curvas and available_curve_ids for use later
    # ------------------------------------------------------------------
    print(f"   ✅ Lookup assets ready. Matching will run per parcel after loading.")
    
    return df_curvas, available_curve_ids, lookup_json, is_ni_gt


def aplicar_matching_curvas(parcelas_gdf, df_curvas, available_curve_ids, lookup_json, is_ni_gt, fecha_dt):
    """
    Takes the loaded parcelas GeoDataFrame and runs the full matching engine.
    Builds group names using Cell 19 rules and runs the matching ladder.
    Adds matched_curve_id and match_tier as columns to parcelas_gdf.
    Also adds potencial column per parcel based on their edad and matched curve.
    
    Args:
        parcelas_gdf: GeoDataFrame with parcels (must have ciclo, tipo_riego, 
                      division_03, fecha_fin_estimada, fecha_inicio, edad)
        df_curvas: Full consolidated parquet DataFrame
        available_curve_ids: Set of curve_ids present in parquet
        lookup_json: JSON lookup dictionary
        is_ni_gt: Boolean, True if country is NI or GT
        fecha_dt: datetime object of the image date
    
    Returns:
        parcelas_gdf with new columns:
            matched_curve_id, match_tier
        AND a potencial lookup dict:
            { id_parcela: {'NDRE_p80': float, 'NDWI_p80': float} }
    """
    
    VALID_YEAR_MIN = 2015
    VALID_YEAR_MAX = 2035
    
    meses_nombres = {
        1:'ENE', 2:'FEB', 3:'MAR', 4:'ABR', 5:'MAY', 6:'JUN',
        7:'JUL', 8:'AGO', 9:'SEP', 10:'OCT', 11:'NOV', 12:'DIC'
    }
    
    # ------------------------------------------------------------------
    # HELPER: Safe binary conversion (digits only → 0/1, else OTRAS)
    # ------------------------------------------------------------------
    def safe_binary(val):
        """
        Converts numeric value to binary.
        Only accepts actual numeric digits (int or float).
        Anything else (None, NaN, string, garbage) → OTRAS.
        """
        if val is None:
            return 'OTRAS'
        # Check for pandas/numpy NaN
        try:
            if pd.isna(val):
                return 'OTRAS'
        except:
            pass
        # Check if it is numeric
        try:
            numeric_val = float(val)
            return 1 if numeric_val > 0 else 0
        except (ValueError, TypeError):
            return 'OTRAS'
    
    # ------------------------------------------------------------------
    # HELPER: Resolve harvest month name
    # ------------------------------------------------------------------
    def resolve_mes_nombre(row):
        """
        Strategy A: fecha_fin_estimada (sane year 2015-2035) → month name
        Strategy B: fecha_inicio + 360 days → month name
        Strategy C: OTRAS
        """
        # Strategy A
        try:
            ffe = pd.to_datetime(row.get('fecha_fin_estimada'), errors='coerce')
            if pd.notna(ffe) and VALID_YEAR_MIN <= ffe.year <= VALID_YEAR_MAX:
                return meses_nombres.get(ffe.month, 'OTRAS')
        except:
            pass
        
        # Strategy B
        try:
            fi = pd.to_datetime(row.get('fecha_inicio'), errors='coerce')
            if pd.notna(fi):
                estimated = fi + pd.to_timedelta(360, unit='D')
                return meses_nombres.get(estimated.month, 'OTRAS')
        except:
            pass
        
        # Strategy C
        return 'OTRAS'
    
    # ------------------------------------------------------------------
    # HELPER: Safe stratum resolution
    # ------------------------------------------------------------------
    def resolve_strat(val):
        """
        If present and not null → use as-is (string).
        If None, NaN, or unreadable → OTRAS.
        """
        if val is None:
            return 'OTRAS'
        try:
            if pd.isna(val):
                return 'OTRAS'
        except:
            pass
        # It exists and is not null — use as-is
        val_str = str(val).strip()
        if val_str == '' or val_str.lower() == 'nan' or val_str.lower() == 'none':
            return 'OTRAS'
        return val_str
    
    # ------------------------------------------------------------------
    # STEP 1: Build binary and resolved columns for all parcels
    # ------------------------------------------------------------------
    print(f"\n   [MATCHING] Building group names for {len(parcelas_gdf)} parcels...")
    
    parcelas_gdf = parcelas_gdf.copy()
    
    parcelas_gdf['ciclo_binary']        = parcelas_gdf['ciclo'].apply(safe_binary)
    parcelas_gdf['riego_binary']        = parcelas_gdf['tipo_riego'].apply(safe_binary)
    parcelas_gdf['mes_nombre_resuelto'] = parcelas_gdf.apply(resolve_mes_nombre, axis=1)
    
    if is_ni_gt and 'division_03' in parcelas_gdf.columns:
        parcelas_gdf['strat_resolved'] = parcelas_gdf['division_03'].apply(resolve_strat)
    else:
        parcelas_gdf['strat_resolved'] = 'N/A'
    
    # ------------------------------------------------------------------
    # STEP 2: Build group names (4-var and 3-var)
    # ------------------------------------------------------------------
    def build_4var(row):
        c = str(row['ciclo_binary'])
        m = str(row['mes_nombre_resuelto'])
        r = str(row['riego_binary'])
        e = str(row['strat_resolved'])
        return f"{c}_{m}_{r}_{e}"
    
    def build_3var(row):
        c = str(row['ciclo_binary'])
        m = str(row['mes_nombre_resuelto'])
        r = str(row['riego_binary'])
        return f"{c}_{m}_{r}"
    
    if is_ni_gt:
        parcelas_gdf['grupo_4var'] = parcelas_gdf.apply(build_4var, axis=1)
        parcelas_gdf['grupo_3var'] = parcelas_gdf.apply(build_3var, axis=1)
    else:
        # MX: only 3-var
        parcelas_gdf['grupo_4var'] = parcelas_gdf.apply(build_3var, axis=1)
        parcelas_gdf['grupo_3var'] = parcelas_gdf.apply(build_3var, axis=1)
    
    # ------------------------------------------------------------------
    # STEP 3: Run matching ladder for every parcel
    # Always full ladder — no cheating — OTRAS goes through ladder too
    # ------------------------------------------------------------------
    print(f"   [MATCHING] Running matching ladder...")
    
    matched_curve_ids = []
    match_tiers = []
    
    for idx, row in parcelas_gdf.iterrows():
        group_4var = row['grupo_4var']
        group_3var = row['grupo_3var']
        
        matched_curve_id = None
        match_tier = None
        
        # TIER 1: Exact match (4-var for NI/GT, 3-var for MX)
        if group_4var in available_curve_ids:
            matched_curve_id = group_4var
            match_tier = 'EXACT'
        
        # TIER 2: Remove stratum (NI/GT only)
        elif is_ni_gt and group_3var in available_curve_ids:
            matched_curve_id = group_3var
            match_tier = 'FALLBACK_NO_STRATUM'
        
        # TIER 3: Global — always available
        else:
            matched_curve_id = 'MASTER_GLOBAL'
            match_tier = 'GLOBAL'
        
        matched_curve_ids.append(matched_curve_id)
        match_tiers.append(match_tier)
    
    parcelas_gdf['matched_curve_id'] = matched_curve_ids
    parcelas_gdf['match_tier'] = match_tiers
    
    # ------------------------------------------------------------------
    # STEP 4: Print matching summary
    # ------------------------------------------------------------------
    tier_counts = parcelas_gdf['match_tier'].value_counts()
    print(f"   [MATCHING] Results:")
    for tier, count in tier_counts.items():
        pct = count / len(parcelas_gdf) * 100
        print(f"      {tier:<25} → {count:>5} parcels ({pct:.1f}%)")
    
    # ------------------------------------------------------------------
    # STEP 5: Build potencial lookup dict per parcel
    # { id_parcela: {'NDRE_p80': float, 'NDWI_p80': float} }
    # Uses the parcel's edad and matched curve_id to get the right P80
    # ------------------------------------------------------------------
    print(f"   [MATCHING] Building potencial lookup per parcel...")
    
    # Build a fast index on df_curvas for speed
    # Key: (curve_id, edad_dias) → (NDRE_p80, NDWI_p80)
    curvas_index = df_curvas.set_index(['curve_id', 'edad_dias'])[['NDRE_p80', 'NDWI_p80']]
    
    potencial_lookup = {}
    
    for idx, row in parcelas_gdf.iterrows():
        pid = row['id_parcela']
        curve_id = row['matched_curve_id']
        edad = row.get('edad', None)
        
        ndre_val = None
        ndwi_val = None
        
        if edad is not None and pd.notna(edad):
            edad_int = int(min(max(edad, 0), 450))  # Clamp to 0-450
            try:
                vals = curvas_index.loc[(curve_id, edad_int)]
                ndre_val = float(vals['NDRE_p80'])
                ndwi_val = float(vals['NDWI_p80'])
            except KeyError:
                # Age not found in this curve — fallback to GLOBAL
                try:
                    vals = curvas_index.loc[('MASTER_GLOBAL', edad_int)]
                    ndre_val = float(vals['NDRE_p80'])
                    ndwi_val = float(vals['NDWI_p80'])
                except KeyError:
                    pass
        
        potencial_lookup[pid] = {
            'NDRE_p80': ndre_val,
            'NDWI_p80': ndwi_val
        }
    
    print(f"   ✅ Potencial lookup built for {len(potencial_lookup)} parcels.")
    
    return parcelas_gdf, potencial_lookup
        


def obtener_zafras_activas(ingenio):
    """
    Obtiene lista de zafras activas para un ingenio (solo informativo)
    Filtra por temporada_activa = True
    """
    supabase = get_supabase_client()
    
    print(f"        → Consultando zafras activas para {ingenio}...")
    
    try:
        response = supabase.table('parcelas_ingenios_reprocess') \
            .select('zafra') \
            .eq('ingenio', ingenio) \
            .eq('temporada_activa', True) \
            .execute()
        
        if not response.data:
            raise ValueError(f"No se encontraron parcelas activas para ingenio='{ingenio}'")
        
        # Obtener zafras únicas
        zafras = sorted(list(set([row['zafra'] for row in response.data])))
        
        print(f"        ✓ Zafras activas encontradas: {zafras}")
        
        return zafras
        
    except Exception as e:
        print(f"        ❌ Error consultando zafras activas: {str(e)}")
        raise

def cargar_parcelas_desde_supabase(ingenio):
    """
    Carga parcelas desde Supabase OPTIMIZADO
    Solo carga columnas necesarias para reducir transferencia de datos
    """
    supabase = get_supabase_client()
    
    print(f"        → Consultando parcelas desde Supabase...")
    print(f"           Filtros: ingenio='{ingenio}', temporada_activa=True")
    
    # ============================================
    # CRÍTICO: Solo seleccionar columnas necesarias
    # Esto reduce ENORMEMENTE el tiempo de query
    # ============================================
    columns_needed = 'id_parcela,zafra,fecha_inicio,area_calculada,ingenio,company,company_id,ingenio_id,geometry_polygon,ciclo,tipo_riego,division_03,fecha_fin_estimada'
    # columns_needed = 'id_parcela,zafra,fecha_inicio,area_calculada,ingenio,company,geometry_polygon'
    
    try:
        import time
        start = time.time()
        
        response = supabase.table('parcelas_ingenios_reprocess') \
            .select(columns_needed) \
            .eq('ingenio', ingenio) \
            .eq('temporada_activa', True) \
            .execute()
        
        elapsed = time.time() - start
        print(f"        ✓ Query completada en {elapsed:.2f} segundos")
        
        if not response.data:
            raise ValueError(f"No se encontraron parcelas activas para ingenio='{ingenio}'")
        
        df = pd.DataFrame(response.data)
        print(f"        ✓ {len(df)} parcelas encontradas")

        # --- SANITY CHECK ---
        if len(df) != df['id_parcela'].nunique():
            print(f"        ⚠ WARNING: Detectados {len(df) - df['id_parcela'].nunique()} IDs duplicados. Revise la BD.")
        # --------------------
        
        # =========================================================
        # Parse geometry (tu código existente)
        # =========================================================
        def parse_geometry(val):
            """
            Optimistic parsing: Assumes Supabase ALWAYS returns a Dict (GeoJSON).
            """
            if val is None or pd.isna(val):
                return None
            try:
                if isinstance(val, dict):
                    return shape(val)
                # -------------------------------------------------
                # If it's not a dict, return None (Strict Mode)
                return None
            
            except Exception as e:
                print(f"        ⚠ Error parseando geometría: {e}")
                return None

        df['geometry'] = df['geometry_polygon'].apply(parse_geometry)
        df_valid = df[df['geometry'].notna()].copy()
        
        if len(df_valid) < len(df):
            print(f"        ⚠ {len(df) - len(df_valid)} parcelas sin geometría válida (excluidas)")
        
        if len(df_valid) == 0:
            raise ValueError("Error Crítico: Ninguna geometría se pudo parsear. Verifique formato en BD.")

        # Detectar CRS
        sample_geom = df_valid.iloc[0].geometry
        if sample_geom.geom_type == 'Polygon':
            sample_coords = sample_geom.exterior.coords[0]
        else:
            sample_coords = list(sample_geom.geoms[0].exterior.coords)[0]
        
        x, y = sample_coords[0], sample_coords[1]
        
        if abs(x) < 180 and abs(y) < 90:
            print(f"        → Coordenadas detectadas: WGS84 (lon={x:.4f}, lat={y:.4f})")
            gdf = gpd.GeoDataFrame(df_valid, geometry='geometry', crs='EPSG:4326')
        else:
            if 100000 <= abs(x) <= 900000:
                ingenio_utm_zones = {
                    'CAC': 'EPSG:32619',
                    'Pantaleon': 'EPSG:32615',
                    'SANTA_CLARA': 'EPSG:32615',
                    'Monte Rosa' : 'EPSG:32616',
                    'Amajac' : 'EPSG:32614',
                    'EMSA' : 'EPSG:32614',
                    'IPSA' : 'EPSG:32614'
                }
                utm_crs = ingenio_utm_zones.get(ingenio, 'EPSG:32615')
                print(f"        → Coordenadas detectadas: UTM (x={x:.0f}, y={y:.0f})")
                print(f"        → CRS detectado: {utm_crs}")
                gdf = gpd.GeoDataFrame(df_valid, geometry='geometry', crs=utm_crs)
                
                print(f"        → Reproyectando {utm_crs} → EPSG:4326...")
                gdf = gdf.to_crs('EPSG:4326')
            else:
                print(f"        ⚠ Coordenadas no reconocidas (x={x:.2f}, y={y:.2f}), asumiendo WGS84")
                gdf = gpd.GeoDataFrame(df_valid, geometry='geometry', crs='EPSG:4326')
        
        sample_geom_final = gdf.iloc[0].geometry
        if sample_geom_final.geom_type == 'Polygon':
            sample_coords_final = sample_geom_final.exterior.coords[0]
        else:
            sample_coords_final = list(sample_geom_final.geoms[0].exterior.coords)[0]
        print(f"        ✓ Coordenadas finales WGS84: (lon={sample_coords_final[0]:.6f}, lat={sample_coords_final[1]:.6f})")
        
        return gdf
        
    except Exception as e:
        print(f"        ❌ Error cargando parcelas desde Supabase: {str(e)}")
        raise



def insertar_a_supabase(df, producto, ingenio, fecha):
    """
    Inserta datos del DataFrame a Supabase con UPSERT
    
    Tablas y campos por producto:
    - NDVI: public.data_ndvi
    - NDWI: public.data_ndwi  
    - SMART_GROWTH: public.data_smart_growth
    - WEED: public.data_maleza
    """
    supabase = get_supabase_client()
    
    # Mapeo de productos a tablas
    tabla_map = {
        "NDVI": "data_ndvi",
        "NDWI": "data_ndwi",
        "SMART_GROWTH": "data_sg",
        "WEED": "data_maleza"
    }
    
    if producto not in tabla_map:
        print(f"        ⚠ Producto {producto} no tiene tabla configurada")
        return
    
    tabla = tabla_map[producto]
    
    # Convertir DataFrame a lista de dicts
    df_copy = df.copy()
    
    # Convertir fecha_img a string ISO
    if 'fecha_img' in df_copy.columns:
        df_copy['fecha_img'] = df_copy['fecha_img'].dt.strftime('%Y-%m-%d')
    
    # Convertir NaN a None para JSON
    df_copy = df_copy.where(pd.notna(df_copy), None)
    
    records = df_copy.to_dict('records')
    
    print(f"        → Insertando {len(records)} registros a {tabla}...")
    
    try:
        # Inserción simple sin ON CONFLICT
        # Las tablas no tienen constraints únicos definidos
        response = supabase.table(tabla).insert(
            records
        ).execute()
        
        print(f"        ✓ {len(records)} registros insertados en {tabla}")
        
    except Exception as e:
        error_msg = str(e)
        print(f"        ❌ Error insertando a {tabla}: {error_msg}")
        
        # Si el error es por registros duplicados, intentar con delete + insert
        if 'duplicate' in error_msg.lower() or 'unique' in error_msg.lower():
            print(f"        → Intentando eliminar registros existentes y reinsertar...")
            try:
                # Eliminar registros existentes para esta fecha e ingenio
                supabase.table(tabla).delete().eq('fecha_img', fecha).execute()
                # Reintentar inserción
                response = supabase.table(tabla).insert(records).execute()
                print(f"        ✓ {len(records)} registros reinsertados en {tabla}")
            except Exception as e2:
                print(f"        ❌ Error en reintento: {str(e2)}")
                raise
        else:
            raise

def get_ingenio_meta(ingenio_name):
    """Obtiene metadata del ingenio desde el diccionario"""
    if ingenio_name not in INGENIOS_META:
        raise ValueError(f"Ingenio '{ingenio_name}' no encontrado. Opciones: {list(INGENIOS_META.keys())}")
    return INGENIOS_META[ingenio_name]

def generar_nombres_archivos(pais, empresa, ingenio, fecha_str, producto="NDVI"):
    """Genera nombres de archivos según convención: PAIS_EMPRESA_PRODUCTO_INGENIO_FECHA"""
    config = PRODUCTOS_CONFIG[producto]
    
    if config['tipo'] == 'combinado':
        # Smart Growth no tiene archivo de entrada directo
        return {
            'input_raster': None,
            'extracted': f"{pais}_{empresa}_{producto}_{ingenio}_{fecha_str}.tif",
            'potencial': None,
            'data_output': f"{pais}_{empresa}_DATA_{producto}_{ingenio}_{fecha_str}.parquet"
        }
    
    input_suffix = config['input_suffix']
    base_input = f"{pais}_{empresa}_{input_suffix}_{ingenio}_{fecha_str}"
    base_output = f"{pais}_{empresa}_{producto}_{ingenio}_{fecha_str}"
    
    return {
        'input_raster': f"{base_input}_cloudfill.tif",
        'extracted': f"{base_output}_extracted.tif",
        'potencial': f"{pais}_{empresa}_POTENCIAL_{producto}_{ingenio}_{fecha_str}.tif" if config['potencial_formula'] else None,
        'data_output': f"{pais}_{empresa}_DATA_{producto}_{ingenio}_{fecha_str}.parquet"
    }

# Calculate age based on the new logic by Sergio (where the max age should be capped to 450 days)
def calculate_days(fecha_img, fecha_inicio):
    """
    Calcula días desde fecha de inicio de zafra.
    Directive: CLIPPED to a maximum of 450 days.
    """
    raw_days = (fecha_img - fecha_inicio).days
    
    # Logic: If days > 450, force it to 450. Otherwise keep it as is.
    if raw_days > 450:
        return 450
    else:
        return raw_days

# def calculate_days(fecha_img, fecha_inicio):
#     """Calcula días desde fecha de inicio de zafra"""
#     return (fecha_img - fecha_inicio).days

def calcular_etapa_fenologica(edad):
    """Clasifica la edad en etapa fenológica"""
    if edad is None or pd.isna(edad):
        return "SF"
    elif edad <= 60:
        return "Iniciacion"
    elif edad <= 120:
        return "Macollamiento"
    elif edad <= 210:
        return "Elongacion I"
    elif edad <= 300:
        return "Elongacion II"
    else:
        return "Maduracion"

def zonal_statistics(raster_path, zones_gdf, zone_field, stats=['mean', 'std', 'min', 'max', 'median']):
    """Calcula estadísticas zonales"""
    results = []
    
    with rasterio.open(raster_path) as src:
        if zones_gdf.crs != src.crs:
            zones_gdf = zones_gdf.to_crs(src.crs)
        
        for idx, row in zones_gdf.iterrows():
            zone_id = row[zone_field]
            geom = [mapping(row.geometry)]
            
            try:
                out_image, out_transform = mask(src, geom, crop=True, all_touched=False)
                data = out_image[0]
                valid_data = data[data != src.nodata] if src.nodata is not None else data.flatten()
                valid_data = valid_data[~np.isnan(valid_data)]
                
                if len(valid_data) > 0:
                    stat_dict = {zone_field: zone_id}
                    if 'mean' in stats:
                        stat_dict['MEAN'] = float(np.mean(valid_data))
                    if 'std' in stats:
                        stat_dict['STD'] = float(np.std(valid_data))
                    if 'min' in stats:
                        stat_dict['MIN'] = float(np.min(valid_data))
                    if 'max' in stats:
                        stat_dict['MAX'] = float(np.max(valid_data))
                    if 'median' in stats:
                        stat_dict['MEDIAN'] = float(np.median(valid_data))
                    results.append(stat_dict)
            except Exception as e:
                print(f"      ⚠ Error procesando zona {zone_id}: {e}")
                continue
    
    return pd.DataFrame(results)

def buffer_geometry(gdf, distance):
    """Crea buffer de geometrías en metros"""
    original_crs = gdf.crs
    
    if gdf.crs.is_geographic:
        gdf_projected = gdf.to_crs(epsg=32615)
        gdf_buffered = gdf_projected.copy()
        gdf_buffered['geometry'] = gdf_projected.geometry.buffer(distance)
        gdf_buffered = gdf_buffered.to_crs(original_crs)
    else:
        gdf_buffered = gdf.copy()
        gdf_buffered['geometry'] = gdf.geometry.buffer(distance)
    
    return gdf_buffered

def extract_by_mask(raster_path, mask_gdf, output_path):
    """Extrae un raster usando una máscara de polígonos"""
    with rasterio.open(raster_path) as src:
        if mask_gdf.crs != src.crs:
            mask_gdf = mask_gdf.to_crs(src.crs)
        
        geoms = [mapping(geom) for geom in mask_gdf.geometry]
        out_image, out_transform = mask(src, geoms, crop=True)
        
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })
        
        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(out_image)

def reclassify_raster(raster_path, reclass_dict, output_path, nodata_value=-9999):
    """Reclasifica un raster basado en rangos"""
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        out_data = np.full_like(data, nodata_value, dtype=np.float32)
        
        for min_val, max_val, new_val in reclass_dict:
            mask_range = (data >= min_val) & (data < max_val)
            out_data[mask_range] = new_val
        
        out_meta = src.meta.copy()
        out_meta.update({'dtype': 'float32', 'nodata': nodata_value})
        
        with rasterio.open(output_path, 'w', **out_meta) as dest:
            dest.write(out_data, 1)

def raster_to_polygons(raster_path, simplify_tolerance=0):
    """Convierte un raster a polígonos"""
    with rasterio.open(raster_path) as src:
        image = src.read(1)
        results = []
        for geom, value in shapes(image, transform=src.transform):
            if value != src.nodata:
                results.append({'geometry': shape(geom), 'gridcode': int(value)})
        
        gdf = gpd.GeoDataFrame(results, crs=src.crs)
        if simplify_tolerance > 0:
            gdf['geometry'] = gdf.geometry.simplify(simplify_tolerance)
        return gdf

def polygon_to_raster(gdf, value_field, output_path, cell_size=None, bounds=None, reference_raster=None, nodata_value=-9999, all_touched=True):
    """Convierte polígonos a raster - usa reference_raster para coincidir dimensiones exactas
       UPDATED: Ensures correct NoData value (-9999) is written to file metadata and background.
    """
    if reference_raster is not None:
        # Usar las dimensiones exactas del raster de referencia
        with rasterio.open(reference_raster) as ref:
            height, width = ref.shape
            transform = ref.transform
            crs = ref.crs
            bounds = ref.bounds
    else:
        # Modo legacy - calcular desde cell_size
        if bounds is None:
            bounds = gdf.total_bounds
        
        minx, miny, maxx, maxy = bounds
        width = int((maxx - minx) / cell_size)
        height = int((maxy - miny) / cell_size)
        transform = rasterio.transform.from_bounds(minx, miny, maxx, maxy, width, height)
        crs = gdf.crs
    
    # Asegurar que gdf está en el mismo CRS
    if gdf.crs != crs:
        gdf = gdf.to_crs(crs)
    
    shapes_with_values = ((mapping(geom), value) for geom, value in 
                          zip(gdf.geometry, gdf[value_field]))
    
    raster = rasterize(shapes_with_values, out_shape=(height, width),
                      transform=transform, fill=nodata_value, dtype=np.float32, all_touched=all_touched) # earlier it was fill=0
    
    with rasterio.open(output_path, 'w', driver='GTiff', height=height,
                      width=width, count=1, dtype=np.float32, crs=crs,
                      transform=transform, nodata=nodata_value, BIGTIFF='YES') as dst:
        dst.write(raster, 1)

def raster_calculator(raster1_path, raster2_path, output_path, operation='divide'):
    """
    Operaciones entre rasters - asegura que ambos tengan las mismas dimensiones.
    FIX: Maneja correctamente los valores NoData del header Y los hardcoded (-9999, -32768).
    """
    from rasterio.warp import reproject, Resampling
    
    with rasterio.open(raster1_path) as src1:
        data1 = src1.read(1).astype(np.float32)
        nodata1 = src1.nodata if src1.nodata is not None else -9999
        meta = src1.meta.copy()
        
        with rasterio.open(raster2_path) as src2:
            nodata2 = src2.nodata if src2.nodata is not None else -9999
            
            # Si las dimensiones no coinciden, reproyectar src2 a src1
            if src1.shape != src2.shape or src1.transform != src2.transform:
                data2 = np.empty(src1.shape, dtype=np.float32)
                reproject(
                    source=rasterio.band(src2, 1),
                    destination=data2,
                    src_transform=src2.transform,
                    src_crs=src2.crs,
                    dst_transform=src1.transform,
                    dst_crs=src1.crs,
                    resampling=Resampling.bilinear
                )
            else:
                data2 = src2.read(1).astype(np.float32)
            
            # --- ROBUST VALIDITY CHECK ---
            # 1. Check against the file's defined NoData
            mask1 = (data1 != nodata1)
            mask2 = (data2 != nodata2)
            
            # 2. ALSO Check against common hardcoded garbage values (Float & Int standards)
            # This protects us if the header is missing but the pixels are -32768
            garbage_values = [-9999, -32768]
            for bad_val in garbage_values:
                mask1 &= (data1 != bad_val)
                mask2 &= (data2 != bad_val)
            
            # 3. Combine: Pixel valid ONLY if both inputs are valid
            valid_mask = mask1 & mask2
            # -----------------------------
            
            # Inicializar resultado con NoData
            result = np.full_like(data1, nodata1)
            
            # Ejecutar operación solo en pixeles válidos
            if operation == 'divide':
                with np.errstate(divide='ignore', invalid='ignore'):
                    denom = data2[valid_mask]
                    safe_mask = denom != 0
                    
                    vals = np.zeros_like(denom)
                    vals[safe_mask] = data1[valid_mask][safe_mask] / denom[safe_mask]
                    result[valid_mask] = vals
                    
            elif operation == 'multiply':
                result[valid_mask] = data1[valid_mask] * data2[valid_mask]
            elif operation == 'add':
                result[valid_mask] = data1[valid_mask] + data2[valid_mask]
            elif operation == 'subtract':
                result[valid_mask] = data1[valid_mask] - data2[valid_mask]
            
            # Asegurar que el output tenga el NoData correcto definido
            meta.update(nodata=nodata1)
            
            with rasterio.open(output_path, 'w', **meta) as dest:
                dest.write(result.astype(np.float32), 1)



def clip_raster_with_polygons(raster_path, polygons_gdf, output_path, all_touched=True):
    """Recorta un raster usando polígonos (mask) - solo mantiene áreas dentro de las parcelas"""
    from rasterio.mask import mask as rasterio_mask
    
    # Eliminar archivo de salida si existe
    if os.path.exists(output_path):
        try:
            os.remove(output_path)
        except PermissionError:
            pass  # Intentar de todos modos
    
    # Asegurar que los polígonos están en el mismo CRS que el raster
    with rasterio.open(raster_path) as src:
        if polygons_gdf.crs != src.crs:
            polygons_gdf = polygons_gdf.to_crs(src.crs)
        
        # Convertir geometrías a formato GeoJSON
        geoms = [mapping(geom) for geom in polygons_gdf.geometry]
        
        # Hacer el clip (crop=True recorta al extent de los polígonos)
        out_image, out_transform = rasterio_mask(src, geoms, crop=True, all_touched=all_touched)
        
        # Actualizar metadata
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })
        
        # Guardar raster recortado
        with rasterio.open(output_path, 'w', **out_meta) as dest:
            dest.write(out_image)

# =======================================================
# FUNCIONES ESPECÍFICAS POR TIPO DE PRODUCTO
# =======================================================

def procesar_smart_growth(ingenio, fecha, parcelas_gdf, output_dir, id_field, meta, archivos):
    """
    Procesa Smart Growth combinando potenciales de NDVI y NDWI
    Formula: (potencial_ndvi * 12) + (potencial_ndwi * 8)
    FIX 1: Maneja correctamente los valores NoData (-9999)
    FIX 2: Busca archivos usando guiones bajos (Monte_Rosa)
    """
    from rasterio.warp import reproject, Resampling
    
    print(f"\n📐 Procesando Smart Growth (combina NDVI y NDWI)...")
    
    # Buscar archivos de potencial NDVI y NDWI
    pais = meta['PAIS']
    empresa = meta['EMPRESA']
    
    # --- FILENAME LOGIC ---
    # Los archivos de salida (Potencial) se generaron con guiones bajos en el paso anterior.
    # Por eso forzamos '_' aquí para poder encontrarlos.
    ingenio_safe = ingenio.replace(' ', '_') 
    
    fecha_dt = datetime.strptime(fecha, "%Y-%m-%d")
    fecha_str = fecha_dt.strftime("%Y_%m_%d")
    
    # Calcular fecha_img y edad
    parcelas_gdf['fecha_inicio_converted'] = pd.to_datetime(parcelas_gdf['fecha_inicio'], format='%Y-%m-%d', errors='coerce')
    parcelas_gdf['fecha_img'] = fecha_dt
    parcelas_gdf['edad'] = parcelas_gdf.apply(
        lambda row: calculate_days(row['fecha_img'], row['fecha_inicio_converted'])
        if pd.notna(row['fecha_inicio_converted']) else None, axis=1
    )
    
    # Usamos ingenio_safe (Monte_Rosa) para encontrar los archivos
    potencial_ndvi_path = os.path.join(output_dir, f"{pais}_{empresa}_POTENCIAL_NDVI_{ingenio_safe}_{fecha_str}.tif")
    potencial_ndwi_path = os.path.join(output_dir, f"{pais}_{empresa}_POTENCIAL_NDWI_{ingenio_safe}_{fecha_str}.tif")
    
    if not os.path.exists(potencial_ndvi_path):
        raise FileNotFoundError(f"Potencial NDVI no encontrado: {potencial_ndvi_path}")
    if not os.path.exists(potencial_ndwi_path):
        raise FileNotFoundError(f"Potencial NDWI no encontrado: {potencial_ndwi_path}")
    
    sg_raster_path = os.path.join(output_dir, archivos['extracted'])
    
    with rasterio.open(potencial_ndvi_path) as src_ndvi:
        ndvi_data = src_ndvi.read(1).astype(np.float32)
        ndvi_nodata = src_ndvi.nodata if src_ndvi.nodata is not None else -9999
        meta_out = src_ndvi.meta.copy()
        
        with rasterio.open(potencial_ndwi_path) as src_ndwi:
            # Si las dimensiones no coinciden, reproyectar NDWI a NDVI
            if src_ndvi.shape != src_ndwi.shape or src_ndvi.transform != src_ndwi.transform:
                ndwi_data = np.empty(src_ndvi.shape, dtype=np.float32)
                reproject(
                    source=rasterio.band(src_ndwi, 1),
                    destination=ndwi_data,
                    src_transform=src_ndwi.transform,
                    src_crs=src_ndwi.crs,
                    dst_transform=src_ndvi.transform,
                    dst_crs=src_ndvi.crs,
                    resampling=Resampling.bilinear
                )
            else:
                ndwi_data = src_ndwi.read(1).astype(np.float32)
            
            ndwi_nodata = src_ndwi.nodata if src_ndwi.nodata is not None else -9999

            # --- MATH FIX START (-9999 Issue) ---
            # 1. Crear mascara de validez: Un pixel es válido SOLO si ambos inputs son válidos
            valid_mask = (ndvi_data != ndvi_nodata) & (ndwi_data != ndwi_nodata)
            
            # 2. Inicializar el canvas completo con el valor NoData
            sg_data = np.full_like(ndvi_data, ndvi_nodata)
            
            # 3. Calcular la fórmula SOLO en los pixeles válidos
            # Esto evita que -9999 + -9999 se convierta en -199980
            sg_data[valid_mask] = (ndvi_data[valid_mask] * 12) + (ndwi_data[valid_mask] * 8)
            
            # 4. Asegurar que la metadata de salida defina el NoData correcto
            meta_out.update(nodata=ndvi_nodata)
            # --- MATH FIX END ---
            
            # Guardar raster temporal completo
            sg_temp_path = sg_raster_path.replace('.tif', '_temp.tif')
            with rasterio.open(sg_temp_path, 'w', **meta_out) as dest:
                dest.write(sg_data, 1)
    
    print(f"        ✓ Smart Growth calculado correctamente (NoData preservado)")
    
    # Recortar a parcelas
    print(f"        ✓ Recortando Smart Growth a parcelas...")
    clip_raster_with_polygons(sg_temp_path, parcelas_gdf, sg_raster_path)
    
    # Eliminar temporal
    if os.path.exists(sg_temp_path):
        os.remove(sg_temp_path)
    
    # Estadísticas zonales
    stats = zonal_statistics(sg_raster_path, parcelas_gdf, id_field)
    
    if not stats.empty:
        stats.columns = [id_field, 'sg_mean', 'sg_stdv', 'sg_min', 'sg_max', 'sg_median']
        parcelas_gdf = parcelas_gdf.merge(stats, on=id_field, how='left')
    else:
        # Fallback por si acaso
        for col in ['sg_mean', 'sg_stdv', 'sg_min', 'sg_max', 'sg_median']:
            parcelas_gdf[col] = 0
            
    return parcelas_gdf



def procesar_weed(ingenio, fecha, parcelas_gdf, ndvi_raster_path, output_dir, id_field, meta, archivos, config):
    """
    Procesa detección de malezas (Weed)
    Calcula umbral, filtra, buffers, resta rasters, calcula % de maleza
    Genera raster de temporalidad (overlapping de últimas 5 detecciones)
    """
    print(f"\n🌱 Procesando detección de malezas (Weed)...")
    
    pais = meta['PAIS']
    empresa = meta['EMPRESA']
    fecha_dt = datetime.strptime(fecha, "%Y-%m-%d")
    
    # Archivo histórico acumulativo (GeoJSON)
    historico_path = os.path.join(output_dir, f"{pais}_{empresa}_WEED_HISTORICO_{ingenio}.geojson")
    
    # Calcular fecha_img y edad
    parcelas_gdf['fecha_inicio_converted'] = pd.to_datetime(parcelas_gdf['fecha_inicio'], format='%Y-%m-%d', errors='coerce')
    parcelas_gdf['fecha_img'] = fecha_dt
    parcelas_gdf['edad'] = parcelas_gdf.apply(
        lambda row: calculate_days(row['fecha_img'], row['fecha_inicio_converted'])
        if pd.notna(row['fecha_inicio_converted']) else None, axis=1
    )

    # ==============================================================================
    # 🛑 LOGIC CHANGE: RESTRICT TO AGE <= 90 DAYS (ADDED ON 12-02-2026)
    # ==============================================================================
    print(f"   [FILTER] Applying agronomic filter: Age <= 90 days...")
    initial_count = len(parcelas_gdf)
    
    # Keep only young cane
    parcelas_gdf = parcelas_gdf[parcelas_gdf['edad'] <= 90].copy()
    
    filtered_count = len(parcelas_gdf)
    dropped_count = initial_count - filtered_count
    print(f"        ✓ Retained: {filtered_count} parcels (Dropped {dropped_count} parcels > 90 days)")

    # HANDLE EDGE CASE: If no parcels are <= 90 days
    if filtered_count == 0:
        print(f"        ⚠ STOPPING WEED: No parcels meet the age criteria (<= 90).")
        print(f"        → Generating BLANK (NoData) rasters for consistency...")

        # ---------------------------------------------------------
        # GENERATE GHOST RASTERS (So file system doesn't break)
        # ---------------------------------------------------------
        # Define paths exactly as they are defined later in the code
        # 1. Main Weed Raster
        weed_out = os.path.join(output_dir, archivos['extracted'])
        # 2. Diff Raster
        diff_out = os.path.join(output_dir, f"{pais}_{empresa}_WEED_DIFF_{ingenio}_{fecha.replace('-', '_')}.tif")
        # 3. Threshold Raster
        thresh_out = os.path.join(output_dir, f"{pais}_{empresa}_WEED_THRESHOLD_{ingenio}_{fecha.replace('-', '_')}.tif")
        # 4. Temporalidad Raster
        temp_out = os.path.join(output_dir, archivos['extracted'].replace('_extracted.tif', '_temporalidad.tif'))

        # Create blank TIFs using input raster as template
        with rasterio.open(ndvi_raster_path) as src:
            meta = src.meta.copy()
            # Ensure nodata is set
            nodata_val = src.nodata if src.nodata is not None else -9999
            meta.update(nodata=nodata_val)
            
            # Create array full of NoData
            empty_img = np.full((src.count, src.height, src.width), nodata_val, dtype=meta['dtype'])
            
            # Write the 4 files
            for fpath in [weed_out, diff_out, thresh_out, temp_out]:
                with rasterio.open(fpath, 'w', **meta) as dst:
                    dst.write(empty_img)
                print(f"        ✓ Created blank file: {os.path.basename(fpath)}")

        # ---------------------------------------------------------
        # RETURN EMPTY DATAFRAME
        # ---------------------------------------------------------
        empty_cols = [id_field, 'zafra', 'fecha_img', 'edad', 'area_maleza', 'area_parcela', 
                      'percent_maleza', 'etapa_f', 'status_maleza', 'ingenio', 'company', 
                      'ingenio_id', 'company_id']
        
        # Ensure cols exist in empty df
        empty_df = pd.DataFrame(columns=empty_cols)
        if id_field != 'id_parcela':
            empty_df.rename(columns={id_field: 'id_parcela'}, inplace=True)
            
        return empty_df
    # ==============================================================================
    
    # 1. Calcular umbral de maleza
    print(f"   [1/9] Calculando umbral de maleza (MEAN + 1.5*STD)...")
    stats = zonal_statistics(ndvi_raster_path, parcelas_gdf, id_field)
    stats.columns = [id_field, 'MEAN', 'STD', 'MIN', 'MAX', 'MEDIAN']
    stats['MALEZA'] = stats.apply(lambda row: config['maleza_threshold_formula'](row['MEAN'], row['STD']), axis=1)
    
    parcelas_gdf = parcelas_gdf.merge(stats[[id_field, 'MALEZA']], on=id_field, how='left')
    
    # 2. Filtrar parcelas con maleza válida
    print(f"   [2/9] Filtrando parcelas con maleza (0 < MALEZA < 1)...")
    parcelas_maleza = parcelas_gdf[
        (parcelas_gdf['MALEZA'].notna()) &
        (parcelas_gdf['MALEZA'] > config['maleza_min']) &
        (parcelas_gdf['MALEZA'] < config['maleza_max'])
    ].copy()
    
    if len(parcelas_maleza) == 0:
        print(f"        ⚠ No se encontraron parcelas con maleza")
        parcelas_gdf['area_maleza'] = 0
        parcelas_gdf['area_parcela'] = parcelas_gdf.get('area_calculada', 0)
        parcelas_gdf['percent_maleza'] = 0
        parcelas_gdf['status_maleza'] = "Sin Maleza"
        parcelas_gdf['etapa_f'] = parcelas_gdf['edad'].apply(calcular_etapa_fenologica)
        return parcelas_gdf[['id_parcela' if 'id_parcela' in parcelas_gdf.columns else id_field, 'zafra', 'fecha_img', 'edad', 'area_maleza', 'area_parcela', 'percent_maleza', 'etapa_f', 'status_maleza']]
    
    print(f"        ✓ {len(parcelas_maleza)} parcelas con maleza detectada")
    
    # 3. Rasterizar umbral de maleza (SIN buffer)
    print(f"   [3/9] Rasterizando umbral de maleza por parcela...")
    maleza_raster_path = os.path.join(output_dir, 'maleza_raster_temp.tif')
    # Usar parcelas SIN buffer - cada parcela con su propio umbral (MEAN + 1.4*STD)
    polygon_to_raster(parcelas_maleza, 'MALEZA', maleza_raster_path, reference_raster=ndvi_raster_path, all_touched=False)
    
    # 4. Restar NDVI - Umbral: negativo = maleza (bajo vigor)
    print(f"   [4/9] Calculando NDVI - Umbral (negativo = maleza)...")
    ndvi_minus_maleza_path = os.path.join(output_dir, 'ndvi_minus_maleza_temp.tif')
    raster_calculator(ndvi_raster_path, maleza_raster_path, ndvi_minus_maleza_path, 'subtract')
    
    # 5. Reclasificar: >0 = maleza (1), ≤0 = no maleza (2)
    print(f"   [5/9] Reclasificando (1=Maleza >0, 2=No maleza ≤0)...")
    reclass_path = os.path.join(output_dir, 'weed_reclass_temp.tif')
    reclassify_raster(ndvi_minus_maleza_path, config['reclass_ranges'], reclass_path)
    
    # 6. Convertir a polígonos: SOLO gridcode=1 (maleza)
    print(f"   [6/9] Calculando áreas y % de maleza...")
    polygons = raster_to_polygons(reclass_path)

    # === CRITICAL CRS FIX ===
    # We align the raster polygons (UTM) to the parcel CRS (WGS84).
    if polygons.crs != parcelas_gdf.crs:
        polygons = polygons.to_crs(parcelas_gdf.crs)
    # ========================

    polygons_maleza = polygons[polygons['gridcode'] == 1].copy()  # Solo maleza (gridcode=1)
    
    if len(polygons_maleza) == 0:
        print(f"        ⚠ No se detectaron áreas con maleza después de reclasificación")
        parcelas_gdf['area_maleza'] = 0
        parcelas_gdf['percent_maleza'] = 0
        parcelas_gdf['status_maleza'] = "Sin Maleza"
        parcelas_gdf['etapa_f'] = parcelas_gdf['edad'].apply(calcular_etapa_fenologica)
    else:
        clipped = gpd.overlay(polygons_maleza, parcelas_gdf[[id_field, 'geometry', 'area_calculada']], how='intersection')
        single = clipped.explode(index_parts=False).reset_index(drop=True)
        
        crs_utm = meta['CRS']
        single_proj = single.to_crs(crs_utm)
        single_proj['area_ha'] = single_proj.geometry.area / 10000
        
        maleza_por_parcela = single_proj.groupby(id_field).agg({'area_ha': 'sum'}).reset_index()
        maleza_por_parcela.columns = [id_field, 'area_maleza']
        
        parcelas_gdf = parcelas_gdf.merge(maleza_por_parcela, on=id_field, how='left')
        parcelas_gdf['area_maleza'] = parcelas_gdf['area_maleza'].fillna(0)
        parcelas_gdf['area_parcela'] = parcelas_gdf.get('area_calculada', 0)
        parcelas_gdf['percent_maleza'] = np.where(
            parcelas_gdf['area_parcela'] > 0,
            (parcelas_gdf['area_maleza'] / parcelas_gdf['area_parcela'] * 100).round(2),
            0
        )
        parcelas_gdf['status_maleza'] = np.where(
            parcelas_gdf['percent_maleza'] >= config['maleza_critica_percent'],
            "Critica", "Evaluar"
        )
        parcelas_gdf['etapa_f'] = parcelas_gdf['edad'].apply(calcular_etapa_fenologica)
    
    # 7. Recortar y guardar raster final de malezas
    print(f"   [7/9] Recortando y guardando raster TIFF de malezas...")
    weed_temp_path = os.path.join(output_dir, 'weed_temp_full.tif')
    weed_output_path = os.path.join(output_dir, archivos['extracted'])
    
    # Primero copiar a temporal
    import shutil
    shutil.copy2(reclass_path, weed_temp_path)
    
    # Recortar con los polígonos de las parcelas
    clip_raster_with_polygons(weed_temp_path, parcelas_gdf, weed_output_path, all_touched=False)
    
    # Eliminar temporal
    if os.path.exists(weed_temp_path):
        os.remove(weed_temp_path)
    
    print(f"        ✓ Raster WEED guardado: {archivos['extracted']}")
    print(f"        ✓ Valores: 1=Maleza (NDVI>umbral, exceso vigor), 2=No maleza (NDVI≤umbral)")
    print(f"        ✓ Recortado al extent de las parcelas")
    
    # 7b. Guardar raster de diferencia (NDVI - Umbral) sin reclasificar
    print(f"   [7b/9] Guardando raster de diferencia (NDVI - Umbral)...")
    diff_output_name = f"{pais}_{empresa}_WEED_DIFF_{ingenio}_{fecha.replace('-', '_')}.tif"
    diff_output_path = os.path.join(output_dir, diff_output_name)
    diff_temp_path = os.path.join(output_dir, 'weed_diff_temp.tif')
    
    # Copiar y recortar raster de diferencia
    shutil.copy2(ndvi_minus_maleza_path, diff_temp_path)
    clip_raster_with_polygons(diff_temp_path, parcelas_gdf, diff_output_path, all_touched=False)
    
    if os.path.exists(diff_temp_path):
        os.remove(diff_temp_path)
    
    print(f"        ✓ Raster DIFF guardado: {diff_output_name}")
    print(f"        ✓ Valores continuos: >0 = NDVI excede umbral, ≤0 = NDVI bajo/normal")
    
    # 7c. Guardar raster de umbrales por parcela
    print(f"   [7c/9] Guardando raster de umbrales por parcela...")
    threshold_output_name = f"{pais}_{empresa}_WEED_THRESHOLD_{ingenio}_{fecha.replace('-', '_')}.tif"
    threshold_output_path = os.path.join(output_dir, threshold_output_name)
    threshold_temp_path = os.path.join(output_dir, 'weed_threshold_temp.tif')
    
    # Copiar y recortar raster de umbrales
    shutil.copy2(maleza_raster_path, threshold_temp_path)
    clip_raster_with_polygons(threshold_temp_path, parcelas_gdf, threshold_output_path, all_touched=False)
    
    if os.path.exists(threshold_temp_path):
        os.remove(threshold_temp_path)
    
    print(f"        ✓ Raster THRESHOLD guardado: {threshold_output_name}")
    print(f"        ✓ Cada parcela tiene su umbral (MEAN + 1.5*STD)")
    
    # 8. Generar raster de temporalidad (overlapping)
    print(f"   [8/9] Generando raster de temporalidad (overlapping)...")
    
    # Guardar detección actual en histórico
    deteccion_actual = clipped.copy() if len(clipped) > 0 else gpd.GeoDataFrame()
    if len(deteccion_actual) > 0:
        deteccion_actual['fecha_deteccion'] = fecha_dt
        # zafra ya viene en los datos, no asignar
        
        # Cargar o crear histórico
        if os.path.exists(historico_path):
            historico = gpd.read_file(historico_path)
            
            # ELIMINAR fecha actual si ya existe (evitar duplicados)
            historico = historico[historico['fecha_deteccion'] != fecha_dt]
            
            # Mantener solo últimas 5 fechas únicas
            fechas_unicas = sorted(historico['fecha_deteccion'].unique(), reverse=True)
            if len(fechas_unicas) >= 5:
                fechas_mantener = fechas_unicas[:4]  # Últimas 4 + la actual = 5
                historico = historico[historico['fecha_deteccion'].isin(fechas_mantener)]
            
            # Append nueva detección
            historico = pd.concat([historico, deteccion_actual], ignore_index=True)
        else:
            historico = deteccion_actual
        
        # Guardar histórico actualizado
        historico.to_file(historico_path, driver='GeoJSON')
        
        # Filtrar solo última fecha para clip
        ultima_fecha = historico['fecha_deteccion'].max()
        deteccion_ultima = historico[historico['fecha_deteccion'] == ultima_fecha].copy()
        
        # Clip histórico con última detección
        if len(deteccion_ultima) > 0:
            historico_clipped = gpd.overlay(historico, deteccion_ultima[['geometry']], how='intersection')
            
            # Count overlapping (groupby geometría)
            from shapely.ops import unary_union
            overlapping = historico_clipped.copy()
            overlapping['geometry_wkt'] = overlapping.geometry.apply(lambda g: g.wkt)
            overlapping_count = overlapping.groupby('geometry_wkt').size().reset_index(name='COUNT_')
            overlapping_count['geometry'] = overlapping_count['geometry_wkt'].apply(lambda wkt: gpd.GeoSeries.from_wkt([wkt])[0])
            overlapping_gdf = gpd.GeoDataFrame(overlapping_count, geometry='geometry', crs=historico.crs)
            
            # Limitar a máximo 5
            overlapping_gdf['COUNT_'] = overlapping_gdf['COUNT_'].clip(upper=5)
            
            # Generar raster de temporalidad (temporal completo)
            temporalidad_temp_path = os.path.join(output_dir, 'temporalidad_temp_full.tif')
            temporalidad_raster_path = os.path.join(output_dir, archivos['extracted'].replace('_extracted.tif', '_temporalidad.tif'))
            polygon_to_raster(overlapping_gdf, 'COUNT_', temporalidad_temp_path, reference_raster=ndvi_raster_path, all_touched=False)
            
            # Recortar raster de temporalidad con parcelas
            clip_raster_with_polygons(temporalidad_temp_path, parcelas_gdf, temporalidad_raster_path, all_touched=False)
            
            # Eliminar temporal
            if os.path.exists(temporalidad_temp_path):
                os.remove(temporalidad_temp_path)
            
            print(f"        ✓ Raster temporalidad guardado: {os.path.basename(temporalidad_raster_path)}")
            print(f"        ✓ Rango temporalidad: 1-{int(overlapping_gdf['COUNT_'].max())} detecciones")
            print(f"        ✓ Lógica: 1ª imagen=1, 2ª imagen=1-2, 3ª=1-2-3, hasta máx 5")
            print(f"        ✓ Recortado al extent de las parcelas")
        else:
            print(f"        ⚠ No hay detecciones en última fecha para overlapping")
    else:
        print(f"        ⚠ No hay detecciones para guardar en histórico")
    
    # 9. Limpiar archivos temporales
    print(f"   [9/9] Limpiando archivos temporales...")
    for temp_file in [maleza_raster_path, ndvi_minus_maleza_path, reclass_path, 
                      os.path.join(output_dir, 'weed_temp_full.tif'),
                      os.path.join(output_dir, 'temporalidad_temp_full.tif')]:
        if os.path.exists(temp_file):
            os.remove(temp_file)
    
    print(f"\n        ✓ Parcelas con maleza detectada: {len(parcelas_gdf[parcelas_gdf['area_maleza'] > 0])}")
    print(f"        ✓ Parcelas críticas (≥15%): {len(parcelas_gdf[parcelas_gdf.get('status_maleza', '') == 'Critica'])}")
    
    # Renombrar id_field a 'id_parcela' para salida estandarizada
    # ADDED: ingenio_id, company_id
    cols_weed = [id_field, 'zafra', 'fecha_img', 'edad', 'area_maleza', 'area_parcela', 'percent_maleza', 'etapa_f', 'status_maleza', 'ingenio_id', 'company_id', 'ingenio', 'company']
    # Ensure columns exist before selecting (safety check)
    cols_weed = [c for c in cols_weed if c in parcelas_gdf.columns]
    
    result_gdf = parcelas_gdf[cols_weed].copy()
    result_gdf.rename(columns={id_field: 'id_parcela'}, inplace=True)
    return result_gdf
    
    # # Renombrar id_field a 'id_parcela' para salida estandarizada
    # result_gdf = parcelas_gdf[[id_field, 'zafra', 'fecha_img', 'edad', 'area_maleza', 'area_parcela', 'percent_maleza', 'etapa_f', 'status_maleza']].copy()
    # result_gdf.rename(columns={id_field: 'id_parcela'}, inplace=True)
    # return result_gdf

# =======================================================
# FUNCIÓN PRINCIPAL DE PROCESAMIENTO
# =======================================================

def procesar_producto(ingenio, fecha, producto, input_dir, output_dir, zafras=None, bd_insert=False, id_field='Clave_area', parcelas_geojson_path=None, curve_assets=None):
    """
    Función principal que procesa un producto (NDVI, NDWI, Smart Growth o Weed)
    """
    os.makedirs(output_dir, exist_ok=True)
    
    if producto not in PRODUCTOS_CONFIG:
        raise ValueError(f"Producto '{producto}' no válido. Opciones: {list(PRODUCTOS_CONFIG.keys())}")
    
    config = PRODUCTOS_CONFIG[producto]
    
    # Normalizar nombre de ingenio
    ingenio_key = ingenio.replace(' ', '_')
    meta = get_ingenio_meta(ingenio_key)
    pais, empresa, crs_utm = meta['PAIS'], meta['EMPRESA'], meta['CRS']
    
    fecha_dt = datetime.strptime(fecha, "%Y-%m-%d")
    fecha_str = fecha_dt.strftime("%Y_%m_%d")
    archivos = generar_nombres_archivos(pais, empresa, ingenio_key, fecha_str, producto)
    
    # Auto-detectar zafras
    if zafras is None and not parcelas_geojson_path:
        zafras = obtener_zafras_activas(ingenio)
    elif zafras is None:
        zafras = [2025]
    
    if not isinstance(zafras, list):
        zafras = [zafras]
    
    print(f"\n{'='*70}")
    print(f"  Procesando {producto} - {ingenio} ({pais}/{empresa})")
    print(f"  Fecha: {fecha} | Zafras: {zafras}")
    print(f"{'='*70}\n")
    
    # [1/9] Cargar parcelas
    print("\n[1/9] Cargando parcelas...")
    if parcelas_geojson_path:
        print(f"        → Fuente: GeoJSON local ({parcelas_geojson_path})")
        parcelas_gdf = gpd.read_file(parcelas_geojson_path)
        if 'zafra' not in parcelas_gdf.columns:
            parcelas_gdf['zafra'] = zafras[0]
        print(f"        ✓ {len(parcelas_gdf)} parcelas cargadas")
    else:
        print(f"        → Fuente: Supabase (public.parcelas_ingenios_reprocess)")
        parcelas_gdf = cargar_parcelas_desde_supabase(ingenio)

        # ==============================================================================
        # LOGIC: SAVE IDS WITH MISSING DATES TO JSON
        # ==============================================================================
        # Detect missing dates BEFORE filtering
        missing_dates_mask = parcelas_gdf['fecha_inicio'].isna()
        missing_count = missing_dates_mask.sum()
        
        if missing_count > 0:
            print(f"        ⚠ Found {missing_count} parcels with missing 'fecha_inicio'. saving log...")
            
            # Extract IDs
            missing_ids = parcelas_gdf.loc[missing_dates_mask, id_field].tolist()
            
            # Define JSON filename
            json_filename = f"{pais}_{empresa}_MISSING_DATES_{ingenio}_{fecha_str}.json"
            json_path = os.path.join(output_dir, json_filename)
            
            # Save to JSON
            log_data = {
                "ingenio": ingenio,
                "date": fecha,
                "total_missing": int(missing_count),
                "parcel_ids": missing_ids
            }
            
            with open(json_path, 'w') as f:
                json.dump(log_data, f, indent=4)
                
            print(f"        📝 Log saved: {json_filename}")
        # ==============================================================================
        
        # ==============================================================================
        # <--- CORRECCIÓN 1: FILTRO FECHA_INICIO (Eliminar Nulos)
        # ==============================================================================
        total_encontradas = len(parcelas_gdf)
        print(f"        ✓ {total_encontradas} parcelas cargadas desde BD")
        
        # Filtrar donde fecha_inicio NO es NaT/None/NaN
        parcelas_gdf = parcelas_gdf[parcelas_gdf['fecha_inicio'].notna()].copy()
        total_validas = len(parcelas_gdf)
        eliminadas = total_encontradas - total_validas
        
        print(f"        ✓ Filtrado por fecha_inicio: {total_validas} válidas")
        if eliminadas > 0:
            print(f"        ⚠ ATENCIÓN: Se eliminaron {eliminadas} parcelas por tener 'fecha_inicio' NULA")
        
        if total_validas == 0:
            print("        ❌ Error: No quedan parcelas válidas después del filtro de fechas.")
            return None
        # ==============================================================================

        zafras_cargadas = sorted(parcelas_gdf['zafra'].unique())
        print(f"        ✓ Distribución por zafra: {dict(parcelas_gdf['zafra'].value_counts())}")
    
    # PROCESAMIENTO SEGÚN TIPO
    if config['tipo'] == 'combinado':
        # SMART GROWTH
        print("\n[2/9] Smart Growth requiere NDVI y NDWI procesados previamente...")
        resultado = procesar_smart_growth(ingenio, fecha, parcelas_gdf, output_dir, id_field, meta, archivos)
        columns_to_save = [id_field, 'zafra', 'fecha_img', 'edad', 'sg_mean', 'sg_stdv', 'sg_min', 'sg_max', 'sg_median', 'ingenio_id', 'company_id','ingenio', 'company']
        
    elif config['tipo'] == 'complejo':
        # WEED
        input_raster_path = os.path.join(input_dir, archivos['input_raster'])
        if not os.path.exists(input_raster_path):
            raise FileNotFoundError(f"Archivo no encontrado: {input_raster_path}")
        
        print(f"📂 Input: {archivos['input_raster']}")
        
        # Procesar fechas y edad
        print("\n[2/9] Procesando fechas y edad...")
        parcelas_gdf['fecha_inicio_converted'] = pd.to_datetime(parcelas_gdf['fecha_inicio'], format='%Y-%m-%d', errors='coerce')
        parcelas_gdf['fecha_img'] = fecha_dt
        parcelas_gdf['edad'] = parcelas_gdf.apply(
            lambda row: calculate_days(row['fecha_img'], row['fecha_inicio_converted'])
            if pd.notna(row['fecha_inicio_converted']) else None, axis=1
        )
        
        resultado = procesar_weed(ingenio, fecha, parcelas_gdf, input_raster_path, output_dir, id_field, meta, archivos, config)
        columns_to_save = [id_field, 'zafra', 'fecha_img', 'edad', 'area_maleza', 'area_parcela', 'percent_maleza', 'etapa_f', 'status_maleza', 'ingenio_id', 'company_id','ingenio', 'company']
        
    else:
        # SIMPLE (NDVI, NDWI)
        input_raster_path = os.path.join(input_dir, archivos['input_raster'])
        if not os.path.exists(input_raster_path):
            raise FileNotFoundError(f"Archivo no encontrado: {input_raster_path}")
        
        print(f"📂 Input: {archivos['input_raster']}")
        
        # Filtrar parcelas que intersectan con el raster
        print("\n[2/9] Filtrando parcelas dentro del área del raster...")
        with rasterio.open(input_raster_path) as src:
            raster_crs = src.crs
            raster_bounds = src.bounds
            raster_bbox_wgs84 = gpd.GeoSeries([box(*raster_bounds)], crs=raster_crs).to_crs("EPSG:4326").iloc[0]
            
            parcelas_in_raster = parcelas_gdf[parcelas_gdf.intersects(raster_bbox_wgs84)].copy()
            
            if len(parcelas_in_raster) == 0:
                print("        ⚠ Still zero intersection after reprojection – check raster/path/parcels")
                return None
            
            print(f"        ✓ {len(parcelas_in_raster)} parcelas intersectan el raster (de {len(parcelas_gdf)} totales)")
            parcelas_gdf = parcelas_in_raster
        
        # Procesar fechas, edad y potencial
        print("\n[3/9] Procesando fechas y edad...")
        parcelas_gdf['fecha_inicio_converted'] = pd.to_datetime(parcelas_gdf['fecha_inicio'], format='%Y-%m-%d', errors='coerce')
        parcelas_gdf['fecha_img'] = fecha_dt
        parcelas_gdf['edad'] = parcelas_gdf.apply(
            lambda row: calculate_days(row['fecha_img'], row['fecha_inicio_converted'])
            if pd.notna(row['fecha_inicio_converted']) else None, axis=1
        )
        
        #print("\n[3/9] Calculando potencial...")
        #print(f"        → Usando curva potencial para {ingenio}")
        #parcelas_gdf['potencial'] = parcelas_gdf['edad'].apply(
        #    lambda x: config['potencial_formula'](x, ingenio) if pd.notna(x) else None
        #)

        print("\n[3/9] Calculando potencial...")

        # Determine which P80 column to use based on product
        p80_col = 'NDRE_p80' if producto == 'NDVI' else 'NDWI_p80'
        
        # ------------------------------------------------------------------
        # NEW: Use consolidated curve parquet if assets are available
        # ------------------------------------------------------------------
        if curve_assets is not None:
            df_curvas, available_curve_ids, lookup_json, is_ni_gt = curve_assets
            
            print(f"        → Usando curva consolidada (grupo-específica) para {ingenio}")
            
            # Run matching engine — needs edad to already be computed (Step 3 above)
            parcelas_gdf, potencial_lookup = aplicar_matching_curvas(
                parcelas_gdf=parcelas_gdf,
                df_curvas=df_curvas,
                available_curve_ids=available_curve_ids,
                lookup_json=lookup_json,
                is_ni_gt=is_ni_gt,
                fecha_dt=fecha_dt
            )
            
            # Assign potencial from lookup using the correct P80 column
            # def get_potencial_from_lookup(row):
            #     pid = row['id_parcela'] if 'id_parcela' in row.index else row[id_field]
            #     entry = potencial_lookup.get(pid, {})
            #     val = entry.get(p80_col, None)
                
            #     # If None for any reason → loud warning + polynomial fallback
            #     if val is None or (isinstance(val, float) and np.isnan(val)):
            #         edad = row.get('edad', None)
            #         if pd.notna(edad):
            #             fallback_val = config['potencial_formula'](edad, ingenio)
            #             return fallback_val
            #         return None
            #     return val
            
            # parcelas_gdf['potencial'] = parcelas_gdf.apply(get_potencial_from_lookup, axis=1)
            # print(f"        ✓ Potencial asignado desde curva consolidada ({p80_col})")

            def get_potencial_from_lookup(row):                                         # <--------- 02/03/2026
                pid = row['id_parcela'] if 'id_parcela' in row.index else row[id_field]
                entry = potencial_lookup.get(pid, {})
                val = entry.get(p80_col, None)
                
                if val is None or (isinstance(val, float) and np.isnan(val)):
                    edad = row.get('edad', None)
                    if pd.notna(edad):
                        fallback_val = config['potencial_formula'](edad, ingenio)
                        # Return tuple: (potencial, matched_curve_id, match_tier)
                        return fallback_val, 'FORMULA_FALLBACK', 'FORMULA_FALLBACK'
                    return None, 'FORMULA_FALLBACK', 'FORMULA_FALLBACK'
                return val, row['matched_curve_id'], row['match_tier']

            # Apply and unpack all three columns at once
            results = parcelas_gdf.apply(get_potencial_from_lookup, axis=1)
            parcelas_gdf['potencial']        = results.apply(lambda x: x[0])
            parcelas_gdf['matched_curve_id'] = results.apply(lambda x: x[1])
            parcelas_gdf['match_tier']       = results.apply(lambda x: x[2])
            print(f"        ✓ Potencial asignado desde curva consolidada ({p80_col})")
            
        
        # ------------------------------------------------------------------
        # FALLBACK: No curve assets → polynomial formula (loud warning)
        # ------------------------------------------------------------------
        else:
            print(f"\n{'!'*70}")
            print(f"  ⚠ WARNING: Consolidated curve assets NOT available for {ingenio}.")
            print(f"  ⚠ REASON: JSON or Parquet file was not found in input directory.")
            print(f"  ⚠ FALLBACK: Using polynomial formula for potencial calculation.")
            print(f"  ⚠ ACTION NEEDED: Check that these files exist in input folder:")
            print(f"      - curvas_potenciales_{ingenio.replace(' ', '_')}_consolidado.parquet")
            print(f"      - mapping_production_{ingenio.replace(' ', '_')}.json")
            print(f"{'!'*70}\n")
            
            parcelas_gdf['potencial'] = parcelas_gdf['edad'].apply(
                lambda x: config['potencial_formula'](x, ingenio) if pd.notna(x) else None
            )
            # Add empty columns so downstream code does not break
            parcelas_gdf['matched_curve_id'] = 'FORMULA_FALLBACK'
            parcelas_gdf['match_tier'] = 'FORMULA_FALLBACK'
        
        # Estadisticas zonales
        print(f"\n[4/9] Calculando estadísticas zonales de {producto}...")
        stats = zonal_statistics(input_raster_path, parcelas_gdf, id_field)
        
        if len(stats) == 0:
            print(f"        ⚠ No se pudieron calcular estadísticas.")
            return None
        
        prefix = producto.lower()
        stats.columns = [id_field, f'{prefix}_mean', f'{prefix}_stdv', f'{prefix}_min', f'{prefix}_max', f'{prefix}_median']
        parcelas_gdf = parcelas_gdf.merge(stats, on=id_field, how='left')
        print(f"        ✓ Estadísticas calculadas para {len(stats)} parcelas")
        
        # Buffer y extraer
        print(f"\n[5/9] Creando buffer y extrayendo {producto}...")
        parcelas_buffer = buffer_geometry(parcelas_gdf, 20)
        extracted_temp_path = os.path.join(output_dir, f'{producto.lower()}_extracted_temp.tif')
        extract_by_mask(input_raster_path, parcelas_buffer, extracted_temp_path)
        
        # Recortar a parcelas (sin buffer)
        extracted_path = os.path.join(output_dir, archivos['extracted'])
        clip_raster_with_polygons(extracted_temp_path, parcelas_gdf, extracted_path)
        if os.path.exists(extracted_temp_path):
            os.remove(extracted_temp_path)
        print(f"        ✓ Extraído y recortado: {archivos['extracted']}")
        
        # Reclasificar
        print(f"\n[6/9] Reclasificando {producto}...")
        reclass_path = os.path.join(output_dir, f'{producto.lower()}_reclass_temp.tif')

        # FIX: Use 'extracted_path' (The raster already clipped to the parcels from Step 5)
        extracted_path = os.path.join(output_dir, archivos['extracted']) 
        reclassify_raster(extracted_path, config['reclass_ranges'], reclass_path)
        # reclassify_raster(input_raster_path, config['reclass_ranges'], reclass_path)
        
        # Convertir a polígonos y calcular áreas
        print(f"\n[7/9] Procesando distribución de {producto} por clase...")
        polygons = raster_to_polygons(reclass_path)
        
        # ==============================================================================
        # <--- CORRECCIÓN 2: CRS MATCH (Evitar Fallo Silencioso de Columnas)
        # ==============================================================================
        if polygons.crs != parcelas_gdf.crs:
            # Reproyectar polígonos del raster (UTM) al CRS de parcelas (WGS84)
            polygons = polygons.to_crs(parcelas_gdf.crs)
        # ==============================================================================

        clipped = gpd.overlay(polygons, parcelas_gdf[[id_field, 'geometry']], how='intersection')
        
        if len(clipped) == 0:
            print(f"        ⚠ WARNING: No se generaron intersecciones para la distribución.")
        else:
            single = clipped.explode(index_parts=False).reset_index(drop=True)
            single_proj = single.to_crs(crs_utm)
            single_proj['area_ha'] = single_proj.geometry.area / 10000
            
            dissolved = single_proj.groupby([id_field, 'gridcode']).agg({'area_ha': 'sum'}).reset_index()
            pivot = dissolved.pivot(index=id_field, columns='gridcode', values='area_ha').reset_index()
            pivot.columns = [id_field] + [f'{prefix}_{int(col)}' for col in pivot.columns[1:]]
            pivot = pivot.fillna(0)
            parcelas_gdf = parcelas_gdf.merge(pivot, on=id_field, how='left')
            print(f"        ✓ Distribución por clase calculada")
        
        # Procesar potencial
        print(f"\n[8/9] Procesando potencial {producto}...")
        potencial_raster_path = os.path.join(output_dir, f'potencial_{producto.lower()}_temp.tif')
        
        polygon_to_raster(parcelas_gdf[parcelas_gdf['potencial'].notna()], 'potencial',
                         potencial_raster_path, reference_raster=extracted_path) # reference_raster=input_raster_path
        
        ratio_path = os.path.join(output_dir, f'ratio_{producto.lower()}_temp.tif')
        raster_calculator(extracted_path, potencial_raster_path, ratio_path, 'divide')

        # raster_calculator(input_raster_path, potencial_raster_path, ratio_path, 'divide')
        
        potencial_reclass_temp = os.path.join(output_dir, f'potencial_{producto.lower()}_reclass_temp.tif')
        reclassify_raster(ratio_path, config['reclass_potencial'], potencial_reclass_temp)
        
        # Recortar potencial a parcelas
        potencial_final_path = os.path.join(output_dir, archivos['potencial'])
        clip_raster_with_polygons(potencial_reclass_temp, parcelas_gdf, potencial_final_path)
        if os.path.exists(potencial_reclass_temp):
            os.remove(potencial_reclass_temp)
        
        pot_polygons = raster_to_polygons(potencial_final_path)
        
        # ==============================================================================
        # <--- CORRECCIÓN 3: CRS MATCH (Evitar Fallo Silencioso de Potencial)
        # ==============================================================================
        if pot_polygons.crs != parcelas_gdf.crs:
            pot_polygons = pot_polygons.to_crs(parcelas_gdf.crs)
        # ==============================================================================

        pot_clipped = gpd.overlay(pot_polygons, parcelas_gdf[[id_field, 'geometry']], how='intersection')
        
        if len(pot_clipped) > 0:
            pot_single = pot_clipped.explode(index_parts=False).reset_index(drop=True)
            pot_single_proj = pot_single.to_crs(crs_utm)
            pot_single_proj['area_ha'] = pot_single_proj.geometry.area / 10000
            
            pot_dissolved = pot_single_proj.groupby([id_field, 'gridcode']).agg({'area_ha': 'sum'}).reset_index()
            pot_pivot = pot_dissolved.pivot(index=id_field, columns='gridcode', values='area_ha').reset_index()
            pot_pivot.columns = [id_field] + [f'{prefix}_pot_{int(col)}' for col in pot_pivot.columns[1:]]
            pot_pivot = pot_pivot.fillna(0)
            parcelas_gdf = parcelas_gdf.merge(pot_pivot, on=id_field, how='left')
        
        print(f"        ✓ Potencial {producto}: {archivos['potencial']}")
        
        # Limpiar temporales
        for temp_file in [potencial_raster_path, ratio_path, reclass_path]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        
        # Calcular cosechado
        if config['cosechado_classes']:
            cosechado_cols = [f'{prefix}_{c}' for c in config['cosechado_classes'] if f'{prefix}_{c}' in parcelas_gdf.columns]
            if cosechado_cols:
                parcelas_gdf['cosechado'] = parcelas_gdf[cosechado_cols].sum(axis=1, min_count=1)
            else:
                parcelas_gdf['cosechado'] = 0
            
            if 'area_calculada' in parcelas_gdf.columns:
                parcelas_gdf['cultivo_en_pie'] = parcelas_gdf['area_calculada'] - parcelas_gdf['cosechado']
            else:
                parcelas_gdf['cultivo_en_pie'] = 0
        
        resultado = parcelas_gdf
        
        # Seleccionar columnas
        #columns_to_save = [id_field, 'zafra', 'fecha_img', 'edad','ingenio', 'company', 'ingenio_id', 'company_id']
        columns_to_save = [id_field, 'zafra', 'fecha_img', 'edad', 'ingenio', 'company', 'ingenio_id', 'company_id', 'matched_curve_id', 'match_tier']
        image_cols = [col for col in parcelas_gdf.columns if any([
            col.startswith(f'{prefix}_'),
            col in [f'{prefix}_mean', f'{prefix}_stdv', f'{prefix}_min', f'{prefix}_max', f'{prefix}_median', 'potencial']
        ])]
        columns_to_save.extend(image_cols)
    
    # Guardar resultado final
    print(f"\n[9/9] Guardando resultado final...")
    output_parquet = os.path.join(output_dir, archivos['data_output'])
    
    columns_to_save = [col for col in columns_to_save if col in resultado.columns]
    df_output = resultado[columns_to_save].copy()
    
    if id_field != 'id_parcela':
        df_output = df_output.rename(columns={id_field: 'id_parcela'})
    
    df_output.to_parquet(output_parquet, index=False)
    
    print(f"        ✓ Datos guardados: {archivos['data_output']}")
    
    if bd_insert:
        print(f"\n[SUPABASE] Insertando datos a base de datos...")
        try:
            insertar_a_supabase(df_output, producto, ingenio, fecha)
        except Exception as e:
            print(f"        ⚠ Error en inserción a Supabase: {str(e)}")
            print(f"        → Los datos están guardados en {archivos['data_output']}")
    
    print(f"\n{'='*70}")
    print(f"  ✅ {producto} COMPLETADO")
    print(f"{'='*70}")
    print(f"\n📊 Resumen:")
    print(f"   - Parcelas procesadas: {len(df_output)}")
    if 'zafra' in df_output.columns:
        print(f"   - Distribución por zafra: {dict(df_output['zafra'].value_counts())}")
    print(f"   - Columnas generadas: {len(df_output.columns)}")
    
    return df_output


# =======================================================
# FUNCIÓN PARA PROCESAR MÚLTIPLES PRODUCTOS
# =======================================================

def procesar_todos_productos(ingenio, fecha, productos, input_dir, output_dir, zafras=None, bd_insert=False, id_field='id_parcela', parcelas_geojson_path=None):
    """
    Procesa múltiples productos para un ingenio y fecha
    Consolida todas las zafras activas en un solo archivo de salida por producto
    
    Args:
        zafras: Lista de zafras a procesar o None para auto-detectar desde BD (temporada_activa=True)
    """
    # -----------------------------------------------------------
    # SETUP: DEFINIR VARIABLES DE METADATA PARA NOMBRES DE ARCHIVO
    # -----------------------------------------------------------
    # Necesario para reconstruir el nombre del archivo en la auditoría
    ingenio_key = ingenio.replace(' ', '_')
    try:
        meta = get_ingenio_meta(ingenio_key)
        pais = meta['PAIS']
        empresa = meta['EMPRESA']
        fecha_dt = datetime.strptime(fecha, "%Y-%m-%d")
        fecha_str = fecha_dt.strftime("%Y_%m_%d")
    except Exception as e:
        print(f"⚠ Warning: No se pudo cargar metadata para generar nombres de archivo: {e}")
        pais, empresa, fecha_str = "UNK", "UNK", fecha.replace('-', '_')
    # -----------------------------------------------------------

    resultados = {}
    
    # Auto-detectar zafras activas si no se especifican
    if zafras is None and not parcelas_geojson_path:
        print(f"\n{'~'*80}")
        print(f"  DETECCIÓN AUTOMÁTICA DE ZAFRAS ACTIVAS")
        print(f"{'~'*80}")
        zafras = obtener_zafras_activas(ingenio)
        print(f"\n  → Se procesarán {len(zafras)} zafra(s): {zafras}")
        print(f"  → Salida: 1 archivo por producto (consolidado)\n")
    elif zafras is None:
        # Modo GeoJSON local: usar zafra por defecto
        zafras = [2025]
        print(f"\n  ⚠ Modo GeoJSON local: usando zafra por defecto {zafras[0]}\n")
    
    # Asegurar que zafras sea lista
    if not isinstance(zafras, list):
        zafras = [zafras]

    # === INSERT THIS BLOCK HERE ===
    # Cargar curva específica para este ingenio antes de procesar
    #cargar_curva_dinamica(ingenio)
    # ==============================
    # === CURVE LOADING BLOCK === <-----------02-03-2026
    # Cargar curva dinámica (polynomial fallback — legacy)
    cargar_curva_dinamica(ingenio)

    # Cargar assets de curva consolidada (nuevo sistema grupo-específico)
    print(f"\n📐 Cargando assets de curva consolidada para {ingenio}...")
    curve_assets_result = construir_lookup_potencial(ingenio, input_dir)

    if curve_assets_result[0] is not None:
        curve_assets = curve_assets_result  # (df_curvas, available_curve_ids, lookup_json, is_ni_gt)
        print(f"   ✅ Curve assets listos para matching grupo-específico.")
    else:
        curve_assets = None
        print(f"\n{'!'*70}")
        print(f"  ⚠ WARNING: Consolidated curve assets NOT found for {ingenio}.")
        print(f"  ⚠ ALL PRODUCTS will use polynomial formula as fallback.")
        print(f"  ⚠ Check input folder for:")
        print(f"      - curvas_potenciales_{ingenio.replace(' ', '_')}_consolidado.parquet")
        print(f"      - mapping_production_{ingenio.replace(' ', '_')}.json")
        print(f"{'!'*70}\n")
    # ===========================
    
    
    print(f"\n{'#'*80}")
    print(f"# PROCESAMIENTO DE PRODUCTOS FINALES")
    print(f"# Ingenio: {ingenio}")
    print(f"# Fecha: {fecha}")
    print(f"# Zafras activas: {', '.join(map(str, zafras))}")
    print(f"# Productos: {', '.join(productos)}")
    print(f"# Fuente parcelas: {'Supabase (auto)' if not parcelas_geojson_path else 'GeoJSON local'}")
    print(f"# Insertar a BD: {'SÍ' if bd_insert else 'NO'}")
    print(f"{'#'*80}\n")
    
    # Procesar cada producto (sin iterar por zafra - se procesan todas juntas)
    for i, producto in enumerate(productos, 1):
        print(f"\n{'~'*80}")
        print(f"  [{i}/{len(productos)}] Iniciando {producto}...")
        print(f"{'~'*80}")
        
        try:
            resultado = procesar_producto(
            ingenio=ingenio, fecha=fecha, producto=producto,
            input_dir=input_dir, output_dir=output_dir, 
            zafras=zafras, bd_insert=bd_insert, id_field=id_field,
            parcelas_geojson_path=parcelas_geojson_path,
            curve_assets=curve_assets
            )

            # --- AUDITORÍA DE ARCHIVOS GENERADOS ---
            if resultado is not None:
                # Reconstruir el nombre exacto del archivo parquet
                # Nota: Usamos ingenio_key (con guiones bajos) para coincidir con la convención
                filename = f"{pais}_{empresa}_DATA_{producto}_{ingenio_key}_{fecha_str}.parquet"
                output_parquet = os.path.join(output_dir, filename)
                
                # Ejecutar auditoría
                audit_parquet_file(output_parquet, producto)
            # ---------------------------------------
            
            resultados[producto] = resultado
            
            if resultado is not None:
                total_parcelas = len(resultado)
                print(f"\n✅ {producto} completado - TOTAL: {total_parcelas} parcelas\n")
            else:
                print(f"\n⚠ {producto} - Sin resultados\n")

            # =======================================================
            #  GARBAGE COLLECTION ADDITION (Newly added for efficiency)
            # =======================================================
            # Delete the large result variable from memory (we stored it in the 'resultados' dict anyway)
            del resultado 
            
            # Force Python to clean up RAM immediately
            gc.collect()
            print(f"      🧹 Memory cleaned after {producto}")
            # =======================================================
                
        except Exception as e:
            print(f"\n❌ Error procesando {producto}: {str(e)}\n")
            import traceback
            traceback.print_exc()
            resultados[producto] = None
            
            # Also clean up if it crashes, so the next product has a fresh start
            gc.collect()
    
    print(f"\n{'#'*80}")
    print(f"# RESUMEN FINAL - CONSOLIDADO")
    print(f"{'#'*80}")
    print(f"# Zafras procesadas: {', '.join(map(str, zafras))}")
    print(f"# Salida: 1 archivo por producto (todas las zafras combinadas)")
    print(f"{'#'*80}")
    for producto, resultado in resultados.items():
        if resultado is not None:
            zafras_en_resultado = resultado['zafra'].unique() if 'zafra' in resultado.columns else []
            print(f"  ✅ {producto:20} - {len(resultado)} parcelas (zafras: {list(zafras_en_resultado)})")
        else:
            print(f"  ❌ {producto:20} - Error en procesamiento")
    print(f"{'#'*80}\n")
    
    return resultados

def audit_parquet_file(file_path, expected_product):
    """
    Audits the generated Parquet file.
    UPDATED: Smart detection of product type to avoid false alarms on SG/Weed.
    """
    if not os.path.exists(file_path):
        print(f"❌ CRITICAL: Output file not found: {file_path}")
        return

    df = pd.read_parquet(file_path)
    total_rows = len(df)
    
    print(f"\n🕵️ STARTING AUDIT: {os.path.basename(file_path)}")
    print(f"   Total Rows: {total_rows}")

    # ----------------------------------------------------
    # CHECK 1: NULL VALUES (Data Quality - Applies to ALL)
    # ----------------------------------------------------
    if 'edad' in df.columns:
        missing_age = df['edad'].isna().sum()
        if missing_age > 0:
            print(f"   ⚠️ WARNING: {missing_age} parcels ({missing_age/total_rows:.1%}) have NaN 'edad'.")
        else:
            print(f"   ✅ PASS: All parcels have valid Age.")

    # ----------------------------------------------------
    # CHECK 2: PRODUCT-SPECIFIC COLUMNS
    # ----------------------------------------------------
    product_key = expected_product.upper()
    
    # CASE A: SIMPLE PRODUCTS (NDVI, NDWI) -> Check for Distribution Classes (_1, _2...)
    if product_key in ["NDVI", "NDWI"]:
        prefix = product_key.lower()
        # We expect at least the first few classes to exist (e.g., ndvi_1, ndvi_2)
        expected_cols = [f'{prefix}_{i}' for i in range(1, 3)] 
        missing_cols = [c for c in expected_cols if c not in df.columns]
        
        if missing_cols:
            print(f"   ❌ FAIL: Distribution columns (Histogram) are MISSING!")
            print(f"      Expected {expected_cols}, but not found.")
        else:
            print(f"   ✅ PASS: Distribution columns present (Histogram data ok).")
            
        # Check Mean
        mean_col = f'{prefix}_mean'
        if mean_col in df.columns and df[mean_col].isna().sum() > 0:
             print(f"   ❌ FAIL: Some parcels have NaN '{mean_col}'.")
        else:
             print(f"   ✅ PASS: Mean values are valid.")

    # CASE B: SMART GROWTH -> Check for Statistics (sg_mean)
    elif product_key == "SMART_GROWTH":
        if 'sg_mean' not in df.columns:
            print(f"   ❌ FAIL: 'sg_mean' column is MISSING!")
        else:
            # Check if values are within expected range (20-100)
            # Ignoring 0s which might be potential issues, looking for real data
            valid_sg = df[df['sg_mean'] > 0]['sg_mean']
            if len(valid_sg) > 0:
                print(f"   ✅ PASS: Smart Growth stats present. Range: {valid_sg.min():.1f} - {valid_sg.max():.1f}")
            else:
                print(f"   ⚠️ WARNING: Smart Growth columns exist but values seem empty/zero.")

    # CASE C: WEED -> Check for Weed Specifics (percent_maleza)
    elif product_key == "WEED":
        required = ['area_maleza', 'percent_maleza', 'status_maleza']
        missing = [c for c in required if c not in df.columns]
        
        if missing:
            print(f"   ❌ FAIL: Missing Weed columns: {missing}")
        else:
            critica = df[df['status_maleza'] == 'Critica'].shape[0]
            print(f"   ✅ PASS: Weed data present. Found {critica} critical parcels.")

    print(f"👮 AUDIT COMPLETE\n")

# =======================================================
# USO
# =======================================================
# ============================================
# SUPABASE CONFIGURATION
# ============================================
if not os.getenv("SUPABASE_URL"):
    os.environ["SUPABASE_URL"] = "https://szolgllhvurovqdeprfn.supabase.co"
    os.environ["SUPABASE_KEY"] = "sb_secret_OtQcrVEJD0MWqkm1meLljA_wpIJfO0v"

# ============================================
# MAIN PARAMETERS
# ============================================
if __name__ == '__main__':
    
    # Mill to process
    INGENIO = INGENIO_ENV
    
    # Dates to process (can be single date or list)
    FECHAS = FECHAS_ENV

    # Or single date: FECHAS = "2026-01-10"
    
    # Excel file path
    EXCEL_PATH = "Plantilla-march24.xlsx"
    
    # Processing parameters
    PRODUCTOS = ["NDVI", "NDWI", "SMART_GROWTH", "WEED"]
    #PRODUCTOS = ["NDWI"]
    INPUT_DIR = INPUT_DIR_ENV
    OUTPUT_DIR = OUTPUT_DIR_ENV
    ID_FIELD = "id_parcela"
    PARCELAS_GEOJSON = None
    ZAFRAS = None
    BD_INSERT = False
    
    # Sync parameters
    DRY_RUN_SYNC = False  # Set to True to test without writing to DB
    
    # ============================================
    # EXECUTION OPTIONS
    # ============================================
    
    # Option 1: Full pipeline with automatic sync before each date
    # This is the recommended approach for production
    print("\n" + "="*80)
    print("  STARTING FULL PIPELINE WITH INCREMENTAL SYNC")
    print("="*80)
    
    all_results = procesar_con_sync(
        excel_path=EXCEL_PATH,
        ingenio=INGENIO,
        fechas=FECHAS,
        productos=PRODUCTOS,
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        bd_insert=BD_INSERT,
        dry_run_sync=DRY_RUN_SYNC
    )
    
    # ============================================
    # Option 3: Processing only (no sync)
    # Use this if you've already synced the data
    # ============================================
    """
    print("\n" + "="*80)
    print("  PROCESSING ONLY (NO SYNC)")
    print("="*80)
    
    # Normalize to list
    if isinstance(FECHAS, str):
        fechas_list = [FECHAS]
    else:
        fechas_list = FECHAS
    
    all_results = {}
    
    for idx, fecha in enumerate(fechas_list, 1):
        print(f"\n{'#'*80}")
        print(f"# DATE [{idx}/{len(fechas_list)}]: {fecha}")
        print(f"{'#'*80}")
        
        try:
            resultados = procesar_todos_productos(
                ingenio=INGENIO,
                fecha=fecha,
                productos=PRODUCTOS,
                input_dir=INPUT_DIR,
                output_dir=OUTPUT_DIR,
                zafras=ZAFRAS,
                bd_insert=BD_INSERT,
                id_field=ID_FIELD,
                parcelas_geojson_path=PARCELAS_GEOJSON
            )
            
            all_results[fecha] = resultados
            print(f"\n✅ Date {fecha} completed successfully")
            
        except Exception as e:
            print(f"\n❌ Error processing {fecha}: {str(e)}")
            import traceback
            traceback.print_exc()
            all_results[fecha] = None
    
    # Final summary
    print(f"\n{'#'*80}")
    print(f"# 📊 FINAL SUMMARY")
    print(f"{'#'*80}")
    for fecha, resultados in all_results.items():
        if resultados:
            print(f"\n  📅 {fecha}:")
            for producto, df in resultados.items():
                if df is not None:
                    print(f"      ✅ {producto:20} - {len(df)} parcels")
                else:
                    print(f"      ❌ {producto:20} - Error")
        else:
            print(f"\n  ❌ {fecha}: ERROR")
    print(f"{'#'*80}\n")
    """
    
    print("\n🎉 Execution completed!\n")

✅ Diccionario de ingenios cargado
   Total ingenios configurados: 5
   - Pantaleon       (GT/Grupo Pantaleon): ~5771 km²
   - Monte_Rosa      (NI/Grupo Pantaleon): ~5845 km²
   - Amajac          (MX06/Grupo Pantaleon): ~3975 km²
   - EMSA            (MX07/Grupo Pantaleon): ~9344 km²
   - IPSA            (MX02/Grupo Pantaleon): ~6726 km²

  STARTING FULL PIPELINE WITH INCREMENTAL SYNC

######################################################################
  MULTI-DATE: EMSA | Dates: ['2026-3-26'] | Mode: INCREMENTAL
######################################################################

──────────────────────────────────────────────────────────────────────
  [1/1] 2026-3-26
──────────────────────────────────────────────────────────────────────

  SYNC: EMSA | Date: 2026-3-26 | Mode: INCREMENTAL (25d window)
  Window: 2026-03-01 → 2026-03-26
  [PRODUCTION]
  📊 Loaded: 2026=8057 | 2027=8057 records (after ingenio filter)

  📌 Step 1: Closing 2026 parcels...
     Window: 2026-03-01 ≤ fecha

In [ ]:
#!/usr/bin/env python3
"""
Push parquet files to Supabase for the inference dates in FECHAS_ENV only.
- Filters parquet files by date in filename (YYYY_MM_DD suffix)
- Only pushes files whose date matches a date in FECHAS_ENV
- Drops rows with any null values before inserting
- No human confirmation required — runs headlessly
"""

import os
import re
import pandas as pd
from pathlib import Path
from datetime import datetime
from supabase import create_client, Client
from typing import Dict, List, Tuple

# Supabase credentials
SUPABASE_URL = "https://szolgllhvurovqdeprfn.supabase.co"
SUPABASE_KEY = "sb_secret_OtQcrVEJD0MWqkm1meLljA_wpIJfO0v"

# Parquet filename → Supabase table
TABLE_MAPPING = {
    "DATA_NDVI":         "data_ndvi",
    "DATA_NDWI":         "data_ndwi",
    "DATA_SMART_GROWTH": "data_sg",
    "DATA_WEED":         "data_maleza",
}

# Normalise a date string to a datetime.date regardless of format
# Handles: "2026-3-26", "2026-03-26", "2026_03_26"
def _parse_date(s: str):
    s = s.replace("_", "-")
    for fmt in ("%Y-%m-%d", "%Y-%-m-%-d"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    # fallback: split and zero-pad
    parts = s.split("-")
    if len(parts) == 3:
        return datetime(int(parts[0]), int(parts[1]), int(parts[2])).date()
    raise ValueError(f"Cannot parse date: {s}")

# Extract the YYYY_MM_DD date from a parquet filename stem
# e.g. "GT_Grupo Pantaleon_DATA_SMART_GROWTH_Pantaleon_2026_03_29" -> date(2026,3,29)
_DATE_RE = re.compile(r"(\d{4})_(\d{2})_(\d{2})$")

def _date_from_filename(stem: str):
    m = _DATE_RE.search(stem)
    if not m:
        return None
    return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3))).date()


class ParquetPusher:
    def __init__(self, output_folder: str, inference_dates: List[str]):
        self.output_folder = Path(output_folder)
        self.supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
        self.file_stats: Dict = {}
        # Normalise inference dates for comparison
        self.inference_dates = set()
        for d in inference_dates:
            try:
                self.inference_dates.add(_parse_date(d.strip()))
            except ValueError as e:
                print(f"⚠️  Could not parse FECHAS date '{d}': {e}")

    def _matches_inference_date(self, filename: str) -> bool:
        """Return True if the parquet filename's date is in inference_dates."""
        d = _date_from_filename(Path(filename).stem)
        if d is None:
            print(f"   ⚠️  Cannot extract date from '{filename}' — skipping")
            return False
        return d in self.inference_dates

    def get_table_name(self, filename: str):
        for pattern, table in TABLE_MAPPING.items():
            if pattern in filename:
                return table
        return None

    def clean_dataframe(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
        original = len(df)
        df_clean = df.dropna(how="any")
        return df_clean, original - len(df_clean)

    def convert_types_for_db(self, df: pd.DataFrame, table_name: str) -> pd.DataFrame:
        df = df.copy()
        integer_cols = {
            "data_ndvi":   ["company_id", "ingenio_id"],
            "data_ndwi":   ["zafra", "edad", "company_id", "ingenio_id"],
            "data_sg":     ["edad", "company_id", "ingenio_id"],
            "data_maleza": ["edad", "company_id", "ingenio_id"],
        }
        text_cols = {
            "data_sg":     ["zafra"],
            "data_maleza": ["zafra"],
        }
        for col in integer_cols.get(table_name, []):
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        for col in text_cols.get(table_name, []):
            if col in df.columns:
                df[col] = df[col].astype(str).replace("nan", None)
        return df

    def analyze_files(self) -> Dict:
        all_files = list(self.output_folder.glob("*DATA_*.parquet"))

        # Filter to inference dates only
        matched = [f for f in all_files if self._matches_inference_date(f.name)]
        skipped = len(all_files) - len(matched)

        if not matched:
            print(f"❌ No parquet files found for dates: {sorted(self.inference_dates)}")
            print(f"   (scanned {len(all_files)} files in {self.output_folder})")
            return {}

        print(f"\n📊 Parquet files found:   {len(all_files)}")
        print(f"   Matching inference dates: {len(matched)}")
        if skipped:
            print(f"   Skipped (other dates):    {skipped}")
        print()

        stats: Dict = {}
        for file_path in sorted(matched):
            table_name = self.get_table_name(file_path.name)
            if not table_name:
                print(f"⚠️  No table mapping for '{file_path.name}' — skipping")
                continue

            df = pd.read_parquet(file_path)
            df_clean, dropped = self.clean_dataframe(df)

            if table_name not in stats:
                stats[table_name] = {
                    "files": [],
                    "total_rows_original": 0,
                    "total_rows_clean": 0,
                    "total_rows_dropped": 0,
                }

            stats[table_name]["files"].append({
                "filename":      file_path.name,
                "rows_original": len(df),
                "rows_clean":    len(df_clean),
                "rows_dropped":  dropped,
                "df_clean":      df_clean,
            })
            stats[table_name]["total_rows_original"] += len(df)
            stats[table_name]["total_rows_clean"]    += len(df_clean)
            stats[table_name]["total_rows_dropped"]  += dropped

        self.file_stats = stats
        return stats

    def print_summary(self, stats: Dict):
        print("=" * 70)
        print("📋  DATA TO BE PUSHED")
        print(f"    Inference dates: {sorted(self.inference_dates)}")
        print("=" * 70)

        total_files = total_orig = total_clean = total_dropped = 0
        for table_name, ts in stats.items():
            print(f"\n🗂️   {table_name}")
            print(f"     Files:    {len(ts['files'])}")
            print(f"     Rows in:  {ts['total_rows_original']:,}")
            print(f"     Rows out: {ts['total_rows_clean']:,}  (dropped {ts['total_rows_dropped']:,} nulls)")
            for fi in ts["files"]:
                tag = f"  ⚠️  {fi['rows_dropped']} null rows dropped" if fi["rows_dropped"] else ""
                print(f"     • {fi['filename']}{tag}")
            total_files  += len(ts["files"])
            total_orig   += ts["total_rows_original"]
            total_clean  += ts["total_rows_clean"]
            total_dropped += ts["total_rows_dropped"]

        print("\n" + "=" * 70)
        print(f"   {total_files} files  |  {total_clean:,} rows to insert  |  {total_dropped:,} null rows dropped")
        print("=" * 70)

    def push_to_supabase(self):
        print("\n🚀  Pushing to Supabase...\n")
        for table_name, ts in self.file_stats.items():
            print(f"📤  {table_name}")
            for fi in ts["files"]:
                df_clean = fi["df_clean"]
                if df_clean.empty:
                    print(f"   ⏭️  {fi['filename']} — no rows to insert")
                    continue

                print(f"   → {fi['filename']}  ({len(df_clean):,} rows)")
                try:
                    df_clean = self.convert_types_for_db(df_clean, table_name)
                    records = df_clean.to_dict("records")

                    # Serialise to JSON-safe types
                    clean_records = []
                    for rec in records:
                        clean_rec = {}
                        for k, v in rec.items():
                            if isinstance(v, pd.Timestamp):
                                clean_rec[k] = v.isoformat()
                            elif pd.isna(v) if not isinstance(v, (list, dict)) else False:
                                clean_rec[k] = None
                            elif hasattr(v, "item"):
                                try:
                                    clean_rec[k] = v.item()
                                except (ValueError, OverflowError):
                                    clean_rec[k] = None
                            else:
                                clean_rec[k] = v
                        clean_records.append(clean_rec)

                    # Batch insert
                    BATCH = 1000
                    for i in range(0, len(clean_records), BATCH):
                        self.supabase.table(table_name).insert(clean_records[i:i+BATCH]).execute()
                    print(f"   ✅  {len(clean_records):,} rows inserted")

                except Exception as e:
                    print(f"   ❌  Error: {e}")

        print("\n✅  Supabase push complete")

    def run(self):
        stats = self.analyze_files()
        if not stats:
            return
        self.print_summary(stats)
        self.push_to_supabase()


# ── Main ──────────────────────────────────────────────────────────────────────
output_folder = Path(OUTPUT_DIR_ENV)
if not output_folder.exists():
    print(f"❌ Output folder not found: {output_folder}")
else:
    print(f"🔧  Parquet → Supabase")
    print(f"    Folder:          {output_folder.absolute()}")
    print(f"    Inference dates: {FECHAS_ENV}")
    pusher = ParquetPusher(output_folder=OUTPUT_DIR_ENV, inference_dates=FECHAS_ENV)
    pusher.run()
